# Modulos

In [ ]:
# Módulo para manipulación y análisis de datos con DataFrames
import pandas as pd

# Módulo para trabajar con archivos Excel (lectura y escritura)
import openpyxl

# Módulo para manipular fechas y horas
from datetime import datetime, timedelta

# Módulo para operaciones relacionadas con el tiempo (pausas, mediciones)
import time

# Módulo para trabajar con rutas del sistema operativo
import os

# Módulo para expresiones regulares (búsqueda y manipulación de patrones de texto)
import re

import gc
from pathlib import Path
from typing import Dict

# Rutas
### 1. Explicación técnica del bloque
Se ha migrado la gestión de rutas de un formato estático (hardcoded con usuario específico `osmarrincon`) a un esquema dinámico utilizando la librería `pathlib`. La lógica implementada busca el directorio raíz del proyecto `capresoca-data-automation` subiendo niveles desde el directorio de trabajo actual (`Path.cwd()`). Una vez encontrada la raíz, se reconstruyen las subrutas manteniendo la estructura estándar del repositorio.

### 2. Justificación de decisiones de optimización
- **Uso de Pathlib:** A diferencia de `os.path`, `pathlib` gestiona automáticamente los separadores de ruta (`/` vs `\`) dependiendo de si el notebook se ejecuta en Windows o Linux/macOS.
- **get_project_root():** Esta función garantiza idempotencia y portabilidad, permitiendo que el notebook funcione en cualquier máquina sin modificar una sola línea de código, siempre que se mantenga la estructura interna de carpetas.
- **Evaluación Lazy:** Se definen las rutas como strings (`str()`) al final para mantener compatibilidad con métodos de carga de pandas que a veces tienen conflictos con objetos `PosixPath` o `WindowsPath` en versiones antiguas.

### 3. Validaciones de integridad aplicadas
- **Detección de Directorio Base:** Si no se encuentra la carpeta `Red Asignada` dentro del proyecto, el código se detiene inmediatamente con un `FileNotFoundError` para evitar errores en cascada.
- **Verificación de Existencia:** Se implementó un diccionario de validación (`paths_to_verify`) que comprueba físicamente si los archivos más críticos existen antes de proceder con el flujo de automatización.

### 4. Listado de variables temporales eliminadas
- `paths_to_verify`: Diccionario utilizado únicamente para el check de integridad.
- `missing_files`: Lista de strings de validación.

### 5. Posibles advertencias
- **Nombres de archivos con fecha:** Rutas como `Reporte_Red Asignada General_2026_02_18.csv` siguen siendo sensibles al nombre exacto del archivo. Si el reporte cambia de fecha diariamente, se recomienda en el futuro implementar un buscador de patrones (`glob`) para tomar el archivo más reciente.
- **Sensibilidad a Mayúsculas:** En sistemas Linux, las rutas son *case-sensitive*. Se han respetado las mayúsculas/minúsculas proporcionadas en el string original.

### 6. Validaciones sugeridas
- Ejecutar `os.path.exists(R_Red_SIE)` en una celda separada para confirmar que el acceso al archivo específico de red es exitoso.
- Verificar que el `ROOT_DIR` impreso coincida con la ubicación local del repositorio.

In [ ]:
def get_project_root() -> Path:
    """
    Determina la raíz del proyecto basándose en la ubicación del notebook.
    Busca la carpeta 'capresoca-data-automation' para garantizar independencia del usuario.
    """
    # En VS Code/Jupyter, __file__ no siempre está disponible; usamos Path.cwd()
    current_path = Path.cwd()
    
    # Buscamos el nodo 'capresoca-data-automation' en la jerarquía actual
    for parent in [current_path] + list(current_path.parents):
        if parent.name == "capresoca-data-automation":
            return parent
            
    # Si no se encuentra (por ejemplo, ejecución fuera de la estructura), 
    # se asume que el notebook está en la raíz o un nivel abajo.
    return current_path

# --- Configuración de Rutas Dinámicas ---
ROOT_DIR = get_project_root()
DATA_DIR = ROOT_DIR / "data" / "Red Asignada"

# Validación de existencia de la estructura base
if not DATA_DIR.exists():
    raise FileNotFoundError(f"No se encontró el directorio base de datos en: {DATA_DIR}")

# 1. Reporte SIE
R_Red_SIE = str(DATA_DIR / "Reporte SIE" / "Reporte_Red Asignada General_2026_02_18.csv")

# 2. Portabilidad
PORT_DIR = DATA_DIR / "Portabilidad"
R_portabilidad_NAc_EPSC25 = str(PORT_DIR / "PORTABILIDAD_NACIONAL_REGIMEN_CONTRIBUTIVO_ENERO26.xlsx")
H_portabilidad_NAc_EPSC25 = "PORT NACIONAL CONTRB"

R_portabilidad_NAc_EPS025 = str(PORT_DIR / "PORTABILIDAD_NACIONAL_REGIMEN_SUBSIDIADO_ENERO26.xlsx")
H_portabilidad_NAc_EPS025 = "PORT NACIONAL SUB"

R_portabilidad_Reg_EPSC25 = str(PORT_DIR / "PORTABILIDAD_REGIONAL_REGIMEN_CONTRIBUTIVO_ENERO26.xlsx")
H_portabilidad_Reg_EPSC25 = "PORT REGIONAL CONTRB"

R_portabilidad_Reg_EPS025 = str(PORT_DIR / "PORTABILIDAD_REGIONAL_REGIMEN_SUBSIDIADO_ENERO26.xlsx")
H_portabilidad_Reg_EPS025 = "PORT REGIONAL SUB"

# 3. LMA y Compensados
LMA_DIR = DATA_DIR / "LMA-Compensados"
R_lma = str(LMA_DIR / "LMA ENERO 2026.xlsx")
H_lma = "BD"

R_compensados = str(LMA_DIR / "COMPENSADOS ENERO.xlsx")
H_compensados = "COMPENSADOS"

R_ms_indigenas = str(LMA_DIR / "RESGUARDO INDIGENA.xlsx")
H_ms_indigenas = "Hoja1"

# 4. Alto Costo
R_ms_altocosto = str(DATA_DIR / "Alto costo" / "Maestro-retiros-Altocostos.xlsx")
H_ms_altocosto = "Sheet1"

# 5. Maestros y Consolidados
MAESTROS_DIR = DATA_DIR / "Maestros"
R_Consolidad_EPS025 = str(MAESTROS_DIR / "EPS025MS0016022026.TXT")
R_Consolidad_EPSC25 = str(MAESTROS_DIR / "EPSC25MC0016022026.TXT")
R_Consolidado_SIE = str(MAESTROS_DIR / "Reporte_Validación Archivos Maestro_2026_02_18.csv")

# 6. Normalización y Contratos
R_Municipios_SIE = str(DATA_DIR / "Nomalización SIE" / "Reporte_MUNICIPIOS_2025_05_14.csv")
R_red_Contratada = str(DATA_DIR / "Tabla de contratos" / "cto.txt")

# 7. Históricos
HIST_DIR = DATA_DIR / "Historicos"
r_historico_eps025 = str(HIST_DIR / "HISTORICO_IDENTIFICACION_S_E EPS025.TXT")
r_historico_epsC25 = str(HIST_DIR / "HISTORIA_IDENTIFICACION EPSC25.TXT")

# 8. Salida
R_Salida = str(ROOT_DIR / "data")

# --- Validación de Integridad ---
paths_to_verify: Dict[str, str] = {
    "SIE": R_Red_SIE,
    "LMA": R_lma,
    "Maestro_EPS025": R_Consolidad_EPS025,
    "Contratos": R_red_Contratada
}

missing_files = [name for name, path in paths_to_verify.items() if not Path(path).exists()]

if missing_files:
    print(f"ALERTA: Los siguientes archivos críticos no existen en la ruta dinámica: {missing_files}")
else:
    print("ÉXITO: Todas las rutas críticas han sido validadas dinámicamente.")

# Limpieza selectiva
del paths_to_verify
gc.collect()

# Cargue datafrmes

## df Red de servicios SIE

In [ ]:
# Cargar el archivo CSV con las especificaciones indicadas
df_red_sie = pd.read_csv(R_Red_SIE, sep=';', encoding='latin-1', dtype=str)

## df PN

In [ ]:
# Cargar el archivo Excel de portabilidad nacional
df_pt_nac_EPSC25 = pd.read_excel(R_portabilidad_NAc_EPSC25, sheet_name=H_portabilidad_NAc_EPSC25)
df_pt_nac_EPS025 = pd.read_excel(R_portabilidad_NAc_EPS025, sheet_name=H_portabilidad_NAc_EPS025)

# UNIFICAR DATAFRAMES DE PORTABILIDAD
df_pt_nac = pd.concat([df_pt_nac_EPSC25, df_pt_nac_EPS025], ignore_index=True)

# MANTENER SOLO LAS 2 COLUMNAS REQUERIDAS
df_pt_nac = df_pt_nac[['tipo_identificacion', 'numero_identificacion']]

print("="*80)

# ELIMINAR DATAFRAMES ORIGINALES PARA LIBERAR MEMORIA
del df_pt_nac_EPSC25, df_pt_nac_EPS025

print(f"✓ DataFrame unificado creado: df_pt_nac")
print(f"✓ Registros totales: {len(df_pt_nac)}")
print(f"✓ Columnas: {df_pt_nac.columns.tolist()}")

# Contar registros antes de eliminar duplicados
registros_antes = len(df_pt_nac)

# Verificar duplicados usando ambas columnas como clave
duplicados_pt = df_pt_nac.duplicated(subset=['tipo_identificacion', 'numero_identificacion'])

print(f"Duplicados encontrados: {duplicados_pt.sum()}")

# Mostrar registros duplicados (si existen)
print("\nRegistros duplicados (muestra):")
print(df_pt_nac[duplicados_pt].head())

print("="*80)

# ELIMINAR DUPLICADOS: Mantener solo el primer registro de cada combinación
df_pt_nac = df_pt_nac.drop_duplicates(subset=['tipo_identificacion', 'numero_identificacion'], keep='first')

# Contar registros después de eliminar duplicados
registros_despues = len(df_pt_nac)
registros_eliminados = registros_antes - registros_despues

print("\n" + "="*80)
print("📊 RESUMEN DE ELIMINACIÓN DE DUPLICADOS")
print("="*80)
print(f"✓ Registros ANTES: {registros_antes}")
print(f"✓ Registros DESPUÉS: {registros_despues}")
print(f"✓ Registros ELIMINADOS: {registros_eliminados}")
print(f"✓ Porcentaje eliminado: {(registros_eliminados/registros_antes)*100:.2f}%")

## df PR

In [ ]:
# Cargar el archivo Excel de portabilidad nacional contributivo
df_pt_reg_EPSC25 = pd.read_excel(R_portabilidad_Reg_EPSC25, sheet_name=H_portabilidad_Reg_EPSC25)
df_pt_reg_EPS025 = pd.read_excel(R_portabilidad_Reg_EPS025, sheet_name=H_portabilidad_Reg_EPS025)

# UNIFICAR DATAFRAMES DE PORTABILIDAD
df_pt_reg = pd.concat([df_pt_reg_EPSC25, df_pt_reg_EPS025], ignore_index=True)

print(f"Columnas PR: {df_pt_reg_EPSC25.columns}")

# MANTENER SOLO LAS 2 COLUMNAS REQUERIDAS
df_pt_reg = df_pt_reg[['tipo_identificacion', 'numero_identificacion', 'municipio_receptor']]

print("="*80)

# ELIMINAR DATAFRAMES ORIGINALES PARA LIBERAR MEMORIA
del df_pt_reg_EPSC25, df_pt_reg_EPS025

print(f"✓ DataFrame unificado creado: df_pt_reg")
print(f"✓ Registros totales: {len(df_pt_reg)}")
print(f"✓ Columnas: {df_pt_reg.columns.tolist()}")

# Contar registros antes de eliminar duplicados
registros_antes = len(df_pt_reg)

# Verificar duplicados usando ambas columnas como clave
duplicados_pt = df_pt_reg.duplicated(subset=['tipo_identificacion', 'numero_identificacion'])

print(f"Duplicados encontrados: {duplicados_pt.sum()}")

# Mostrar registros duplicados (si existen)
print("\nRegistros duplicados (muestra):")
print(df_pt_reg[duplicados_pt].head())

print("="*80)

# ELIMINAR DUPLICADOS: Mantener solo el primer registro de cada combinación
df_pt_reg = df_pt_reg.drop_duplicates(subset=['tipo_identificacion', 'numero_identificacion', 'municipio_receptor'], keep='first')

# Contar registros después de eliminar duplicados
registros_despues = len(df_pt_reg)
registros_eliminados = registros_antes - registros_despues

print("\n" + "="*80)
print("📊 RESUMEN DE ELIMINACIÓN DE DUPLICADOS")
print("="*80)
print(f"✓ Registros ANTES: {registros_antes}")
print(f"✓ Registros DESPUÉS: {registros_despues}")
print(f"✓ Registros ELIMINADOS: {registros_eliminados}")
print(f"✓ Porcentaje eliminado: {(registros_eliminados/registros_antes)*100:.2f}%")

## df LMA

In [ ]:
print("\n" + "="*80)
print("📥 CARGA Y LIMPIEZA DE LMA (OPTIMIZADO)")
print("="*80)

import time
import pandas as pd

tiempo_inicio = time.time()

# PASO 1: CARGAR SOLO LAS COLUMNAS NECESARIAS (OPTIMIZACIÓN DE MEMORIA)
df_lma = pd.read_excel(
    R_lma, 
    sheet_name=H_lma,
    usecols=['TPS_IDN_ID', 'HST_IDN_NUMERO_IDENTIFICACION']  # ← Cargar solo lo necesario
)

print(f"✓ Columnas de df_lma: {df_lma.columns.tolist()}")
print(f"✓ Registros totales: {len(df_lma):,}")

# PASO 2: DETECTAR Y ELIMINAR DUPLICADOS EN UNA SOLA PASADA
registros_antes = len(df_lma)

# Crear máscara de duplicados (incluyendo el primero)
duplicados_todas = df_lma.duplicated(
    subset=['TPS_IDN_ID', 'HST_IDN_NUMERO_IDENTIFICACION'], 
    keep=False
)

# PASO 3: ANÁLISIS DE DUPLICADOS (solo si existen)
if duplicados_todas.any():
    print("\n" + "="*80)
    print("📋 REGISTROS DUPLICADOS (muestra)")
    print("="*80)
    # Mostrar solo duplicados (los que se van a eliminar)
    duplicados_a_eliminar = df_lma.duplicated(
        subset=['TPS_IDN_ID', 'HST_IDN_NUMERO_IDENTIFICACION'], 
        keep='first'  # ← Solo los que se van a eliminar
    )
    print(df_lma[duplicados_a_eliminar].head(10))
    
    # ANÁLISIS AGRUPADO (vectorizado, eficiente)
    print("\n" + "="*80)
    print("📊 ANÁLISIS DE DUPLICADOS POR CATEGORÍA")
    print("="*80)
    
    duplicados_agrupados = (
        df_lma[duplicados_todas]
        .groupby(['TPS_IDN_ID', 'HST_IDN_NUMERO_IDENTIFICACION'])
        .size()
        .reset_index(name='cantidad_duplicados')
        .sort_values('cantidad_duplicados', ascending=False)
    )
    
    print(f"Total de registros únicos duplicados: {len(duplicados_agrupados):,}")
    print("\nDetalle de duplicados:")
    print(duplicados_agrupados.to_string(index=False))
    
else:
    print("\n" + "="*80)
    print("✓ No se encontraron duplicados")
    print("="*80)

# PASO 4: ELIMINAR DUPLICADOS (una sola pasada)
df_lma = df_lma.drop_duplicates(
    subset=['TPS_IDN_ID', 'HST_IDN_NUMERO_IDENTIFICACION'], 
    keep='first'
)

registros_despues = len(df_lma)
registros_eliminados = registros_antes - registros_despues
porcentaje_eliminado = (registros_eliminados / registros_antes * 100) if registros_antes > 0 else 0

print("\n" + "="*80)
print("📊 RESUMEN DE ELIMINACIÓN DE DUPLICADOS")
print("="*80)
print(f"✓ Registros ANTES: {registros_antes:,}")
print(f"✓ Registros DESPUÉS: {registros_despues:,}")
print(f"✓ Registros ELIMINADOS: {registros_eliminados:,}")
print(f"✓ Porcentaje eliminado: {porcentaje_eliminado:.2f}%")

# VALIDACIÓN DE INTEGRIDAD
print("\n" + "="*80)
print("🔍 VALIDACIÓN DE INTEGRIDAD")
print("="*80)
print(f"✓ Duplicados residuales: {df_lma.duplicated(subset=['TPS_IDN_ID', 'HST_IDN_NUMERO_IDENTIFICACION']).sum():,}")
print(f"✓ Valores nulos TPS_IDN_ID: {df_lma['TPS_IDN_ID'].isna().sum():,}")
print(f"✓ Valores nulos HST_IDN_NUMERO_IDENTIFICACION: {df_lma['HST_IDN_NUMERO_IDENTIFICACION'].isna().sum():,}")

tiempo_final = time.time()
tiempo_transcurrido = tiempo_final - tiempo_inicio

print("\n" + "="*80)
print(f"⏱️ TIEMPO TRANSCURRIDO: {tiempo_transcurrido:.4f} segundos")
print("="*80)

## df Compensados

In [ ]:
# Cargar el archivo Excel de compensados
df_compensados = pd.read_excel(R_compensados, sheet_name=H_compensados)
print(f"Columnas: {df_compensados.columns}")

print("="*80)

# Mantener solo las 2 columnas requeridas
df_compensados = df_compensados[['Tp_Dc', 'Num_coti']]

print(f"✓ Columnas de df_compensados: {df_compensados.columns.tolist()}")
print(f"✓ Registros totales: {len(df_compensados)}")

print("="*80)

# Contar registros antes de eliminar duplicados
registros_antes = len(df_compensados)

# Verificar duplicados usando ambas columnas como clave
duplicados_compensados = df_compensados.duplicated(subset=['Tp_Dc', 'Num_coti'])

print(f"Duplicados encontrados: {duplicados_compensados.sum()}")

# Mostrar registros duplicados (si existen)
print("\n" + "="*80)
print("📋 REGISTROS DUPLICADOS (muestra)")
print("="*80)
print(df_compensados[duplicados_compensados].head(10))

print("\n" + "="*80)
print("📊 ANÁLISIS DE DUPLICADOS POR CATEGORÍA")
print("="*80)

# Encontrar todos los registros duplicados (incluyendo el primero)
duplicados_todas_ocurrencias = df_compensados.duplicated(
    subset=['Tp_Dc', 'Num_coti'], 
    keep=False
)

# Agrupar por la combinación de columnas y contar ocurrencias
duplicados_agrupados = df_compensados[duplicados_todas_ocurrencias].groupby(
    ['Tp_Dc', 'Num_coti']
).size().reset_index(name='cantidad_duplicados')

# Ordenar por cantidad de duplicados (mayor a menor)
duplicados_agrupados = duplicados_agrupados.sort_values('cantidad_duplicados', ascending=False)

print(f"\nTotal de registros únicos duplicados: {len(duplicados_agrupados)}")
print("\nDetalle de duplicados:")
print(duplicados_agrupados.to_string(index=False))

print("\n" + "="*80)

# ELIMINAR DUPLICADOS: Mantener solo el primer registro de cada combinación
df_compensados = df_compensados.drop_duplicates(
    subset=['Tp_Dc', 'Num_coti'], 
    keep='first'
)

# Contar registros después de eliminar duplicados
registros_despues = len(df_compensados)
registros_eliminados = registros_antes - registros_despues

print("\n" + "="*80)
print("📊 RESUMEN DE ELIMINACIÓN DE DUPLICADOS")
print("="*80)
print(f"✓ Registros ANTES: {registros_antes:,}")
print(f"✓ Registros DESPUÉS: {registros_despues:,}")
print(f"✓ Registros ELIMINADOS: {registros_eliminados:,}")
print(f"✓ Porcentaje eliminado: {(registros_eliminados/registros_antes)*100:.2f}%")
print("="*80)

## df_Maestro Indigenas

In [ ]:
print("\n" + "="*80)
print("📥 CARGA Y LIMPIEZA DE MAESTRO INDIGENAS (OPTIMIZADO)")
print("="*80)

import time
import pandas as pd

tiempo_inicio = time.time()

# PASO 1: CARGAR SOLO LAS COLUMNAS NECESARIAS (OPTIMIZACIÓN DE MEMORIA)
df_ms_indigenas = pd.read_excel(
    R_ms_indigenas, 
    sheet_name=H_ms_indigenas,
    usecols=['T_DOC', 'N_DOC', 'MUNICIPIO_IPS', 'POBLACIÓN']  # ← Cargar solo lo necesario
)

print(f"✓ Columnas de df_ms_indigenas: {df_ms_indigenas.columns.tolist()}")
print(f"✓ Registros totales: {len(df_ms_indigenas):,}")

# PASO 2: DETECTAR Y ELIMINAR DUPLICADOS EN UNA SOLA PASADA
registros_antes = len(df_ms_indigenas)

# Crear máscara de duplicados (incluyendo el primero)
duplicados_todas = df_ms_indigenas.duplicated(
    subset=['T_DOC', 'N_DOC'], 
    keep=False
)

# PASO 3: ANÁLISIS DE DUPLICADOS (solo si existen)
if duplicados_todas.any():
    print("\n" + "="*80)
    print("📋 REGISTROS DUPLICADOS (muestra)")
    print("="*80)
    # Mostrar solo duplicados (los que se van a eliminar)
    duplicados_a_eliminar = df_ms_indigenas.duplicated(
        subset=['T_DOC', 'N_DOC'], 
        keep='first'  # ← Solo los que se van a eliminar
    )
    print(df_ms_indigenas[duplicados_a_eliminar].head(10))
    
    # ANÁLISIS AGRUPADO (vectorizado, eficiente)
    print("\n" + "="*80)
    print("📊 ANÁLISIS DE DUPLICADOS POR CATEGORÍA")
    print("="*80)
    
    duplicados_agrupados = (
        df_ms_indigenas[duplicados_todas]
        .groupby(['T_DOC', 'N_DOC'])
        .size()
        .reset_index(name='cantidad_duplicados')
        .sort_values('cantidad_duplicados', ascending=False)
    )
    
    print(f"Total de registros únicos duplicados: {len(duplicados_agrupados):,}")
    print("\nDetalle de duplicados:")
    print(duplicados_agrupados.to_string(index=False))
    
else:
    print("\n" + "="*80)
    print("✓ No se encontraron duplicados")
    print("="*80)

# PASO 4: ELIMINAR DUPLICADOS (una sola pasada)
df_ms_indigenas = df_ms_indigenas.drop_duplicates(
    subset=['T_DOC', 'N_DOC'], 
    keep='first'
)

registros_despues = len(df_ms_indigenas)
registros_eliminados = registros_antes - registros_despues
porcentaje_eliminado = (registros_eliminados / registros_antes * 100) if registros_antes > 0 else 0

print("\n" + "="*80)
print("📊 RESUMEN DE ELIMINACIÓN DE DUPLICADOS")
print("="*80)
print(f"✓ Registros ANTES: {registros_antes:,}")
print(f"✓ Registros DESPUÉS: {registros_despues:,}")
print(f"✓ Registros ELIMINADOS: {registros_eliminados:,}")
print(f"✓ Porcentaje eliminado: {porcentaje_eliminado:.2f}%")

# VALIDACIÓN DE INTEGRIDAD
print("\n" + "="*80)
print("🔍 VALIDACIÓN DE INTEGRIDAD")
print("="*80)
print(f"✓ Duplicados residuales: {df_ms_indigenas.duplicated(subset=['T_DOC', 'N_DOC']).sum():,}")
print(f"✓ Valores nulos T_DOC: {df_ms_indigenas['T_DOC'].isna().sum():,}")
print(f"✓ Valores nulos N_DOC: {df_ms_indigenas['N_DOC'].isna().sum():,}")

tiempo_final = time.time()
tiempo_transcurrido = tiempo_final - tiempo_inicio

print("\n" + "="*80)
print(f"⏱️ TIEMPO TRANSCURRIDO: {tiempo_transcurrido:.4f} segundos")
print("="*80)

## df_Maestro Alto Costo

In [ ]:
print("\n" + "="*80)
print("📥 CARGA Y LIMPIEZA DE MAESTRO ALTO COSTO (OPTIMIZADO)")
print("="*80)

import time
import pandas as pd

tiempo_inicio = time.time()

# PASO 1: CARGAR SOLO LAS COLUMNAS NECESARIAS (OPTIMIZACIÓN DE MEMORIA)
df_ms_altocosto = pd.read_excel(
    R_ms_altocosto, 
    sheet_name=H_ms_altocosto,
    usecols=['tipo_documento', 'numero_documento', 'descripcion']  # ← Cargar solo lo necesario
)

print(f"✓ Columnas de df_ms_altocosto: {df_ms_altocosto.columns.tolist()}")
print(f"✓ Registros totales: {len(df_ms_altocosto):,}")

# PASO 2: DETECTAR Y ELIMINAR DUPLICADOS EN UNA SOLA PASADA
registros_antes = len(df_ms_altocosto)

# Crear máscara de duplicados (incluyendo el primero)
duplicados_todas = df_ms_altocosto.duplicated(
    subset=['tipo_documento', 'numero_documento'], 
    keep=False
)

# PASO 3: ANÁLISIS DE DUPLICADOS (solo si existen)
if duplicados_todas.any():
    print("\n" + "="*80)
    print("📋 REGISTROS DUPLICADOS (muestra)")
    print("="*80)
    # Mostrar solo duplicados (los que se van a eliminar)
    duplicados_a_eliminar = df_ms_altocosto.duplicated(
        subset=['tipo_documento', 'numero_documento'], 
        keep='first'  # ← Solo los que se van a eliminar
    )
    print(df_ms_altocosto[duplicados_a_eliminar].head(10))
    
    # ANÁLISIS AGRUPADO (vectorizado, eficiente)
    print("\n" + "="*80)
    print("📊 ANÁLISIS DE DUPLICADOS POR CATEGORÍA")
    print("="*80)
    
    duplicados_agrupados = (
        df_ms_altocosto[duplicados_todas]
        .groupby(['tipo_documento', 'numero_documento'])
        .size()
        .reset_index(name='cantidad_duplicados')
        .sort_values('cantidad_duplicados', ascending=False)
    )
    
    print(f"Total de registros únicos duplicados: {len(duplicados_agrupados):,}")
    print("\nDetalle de duplicados:")
    print(duplicados_agrupados.to_string(index=False))
    
else:
    print("\n" + "="*80)
    print("✓ No se encontraron duplicados")
    print("="*80)

# PASO 4: ELIMINAR DUPLICADOS (una sola pasada)
df_ms_altocosto = df_ms_altocosto.drop_duplicates(
    subset=['tipo_documento', 'numero_documento'], 
    keep='first'
)

registros_despues = len(df_ms_altocosto)
registros_eliminados = registros_antes - registros_despues
porcentaje_eliminado = (registros_eliminados / registros_antes * 100) if registros_antes > 0 else 0

print("\n" + "="*80)
print("📊 RESUMEN DE ELIMINACIÓN DE DUPLICADOS")
print("="*80)
print(f"✓ Registros ANTES: {registros_antes:,}")
print(f"✓ Registros DESPUÉS: {registros_despues:,}")
print(f"✓ Registros ELIMINADOS: {registros_eliminados:,}")
print(f"✓ Porcentaje eliminado: {porcentaje_eliminado:.2f}%")

# VALIDACIÓN DE INTEGRIDAD
print("\n" + "="*80)
print("🔍 VALIDACIÓN DE INTEGRIDAD")
print("="*80)
print(f"✓ Duplicados residuales: {df_ms_altocosto.duplicated(subset=['tipo_documento', 'numero_documento']).sum():,}")
print(f"✓ Valores nulos tipo_documento: {df_ms_altocosto['tipo_documento'].isna().sum():,}")
print(f"✓ Valores nulos numero_documento: {df_ms_altocosto['numero_documento'].isna().sum():,}")

tiempo_final = time.time()
tiempo_transcurrido = tiempo_final - tiempo_inicio

print("\n" + "="*80)
print(f"⏱️ TIEMPO TRANSCURRIDO: {tiempo_transcurrido:.4f} segundos")
print("="*80)

## df Consolidados ADRES

In [ ]:
# Cargar los archivos TXT de consolidados ADRES
df_consolidad_EPSC25 = pd.read_csv(
    R_Consolidad_EPSC25, 
    sep=',', 
    encoding='latin-1', 
    dtype=str,
    header=None,
    names=[
        'AFL_ID', 'ENT_ID', 'TPS_IDN_ID_CF', 'HST_IDN_NUMERO_IDENTIFICACION_CF',
        'TPS_IDN_ID', 'HST_IDN_NUMERO_IDENTIFICACION', 'AFL_PRIMER_APELLIDO',
        'AFL_SEGUNDO_APELLIDO', 'AFL_PRIMER_NOMBRE', 'AFL_SEGUNDO_NOMBRE',
        'AFL_FECHA_NACIMIENTO', 'TPS_GNR_ID', 'AFL_PAIS_NACIMIENTO',
        'AFL_MUNICIPIO_NACIMIENTO', 'AFL_NACIONALIDAD', 'AFL_SEXO_IDENTIFICACION',
        'AFL_DISCAPACIDAD', 'TPS_AFL_ID', 'TPS_PRN_ID', 'TPS_GRP_PBL_ID',
        'TPS_NVL_SSB_ID', 'NUMEROFICHASISBEN', 'TPS_CND_BNF_ID', 'DPR_ID',
        'MNC_ID', 'ZNS_ID', 'AFL_FECHA_AFILIACION_SGSSS', 'AFC_FECHA_INICIO',
        'NUMERO_CONTRATO', 'FECHA_INICIO_CONTRATO', 'CNT_AFL_TPS_GRP_PBL_ID',
        'CNT_AFL_TPS_PRT_ETN_ID', 'TPS_MDL_SBS_ID', 'TPS_EST_AFL_ID',
        'CND_AFL_FECHA_INICIO', 'CND_AFL_FECHA_INICIO_2', 'GRP_FML_COTIZANTE_ID',
        'PORTABILIDAD', 'COD_IPS_P', 'MTDLG_G_P', 'SUB_SISBEN_IV',
        'MARCASISBENIV_MARCASISBENIII', 'CRUCE_BDEX_RNEC'
    ]
)

df_consolidad_EPS025 = pd.read_csv(
    R_Consolidad_EPS025,
    sep=',',
    encoding='latin-1',
    dtype=str,
    header=None,
    names=[
        'AFL_ID', 'ENT_ID', 'TPS_IDN_ID_CF', 'HST_IDN_NUMERO_IDENTIFICACION_CF',
        'TPS_IDN_ID', 'HST_IDN_NUMERO_IDENTIFICACION', 'AFL_PRIMER_APELLIDO',
        'AFL_SEGUNDO_APELLIDO', 'AFL_PRIMER_NOMBRE', 'AFL_SEGUNDO_NOMBRE',
        'AFL_FECHA_NACIMIENTO', 'TPS_GNR_ID', 'AFL_PAIS_NACIMIENTO',
        'AFL_MUNICIPIO_NACIMIENTO', 'AFL_NACIONALIDAD', 'AFL_SEXO_IDENTIFICACION',
        'AFL_DISCAPACIDAD', 'TPS_AFL_ID', 'TPS_PRN_ID', 'TPS_GRP_PBL_ID',
        'TPS_NVL_SSB_ID', 'NUMEROFICHASISBEN', 'TPS_CND_BNF_ID', 'DPR_ID',
        'MNC_ID', 'ZNS_ID', 'AFL_FECHA_AFILIACION_SGSSS', 'AFC_FECHA_INICIO',
        'NUMERO_CONTRATO', 'FECHA_INICIO_CONTRATO', 'CNT_AFL_TPS_GRP_PBL_ID',
        'CNT_AFL_TPS_PRT_ETN_ID', 'TPS_MDL_SBS_ID', 'TPS_EST_AFL_ID',
        'CND_AFL_FECHA_INICIO', 'CND_AFL_FECHA_INICIO_2', 'GRP_FML_COTIZANTE_ID',
        'PORTABILIDAD', 'COD_IPS_P', 'MTDLG_G_P', 'SUB_SISBEN_IV',
        'MARCASISBENIV_MARCASISBENIII', 'CRUCE_BDEX_RNEC'
    ]
)

# UNIFICAR AMBOS DATAFRAMES
df_consolidad_adres = pd.concat([df_consolidad_EPSC25, df_consolidad_EPS025], ignore_index=True)

print("="*80)
print("📊 CARGA DE CONSOLIDADOS ADRES")
print("="*80)
print(f"✓ Registros EPSC25 (Contributivo): {len(df_consolidad_EPSC25):,}")
print(f"✓ Registros EPS025 (Subsidiado): {len(df_consolidad_EPS025):,}")
print(f"✓ Total registros unificados: {len(df_consolidad_adres):,}")
print(f"✓ Columnas: {len(df_consolidad_adres.columns)}")
print(f"\n✓ Primeras columnas: {df_consolidad_adres.columns[:10].tolist()}")

# ELIMINAR DATAFRAMES ORIGINALES PARA LIBERAR MEMORIA
del df_consolidad_EPSC25, df_consolidad_EPS025

print("\n✓ DataFrames originales eliminados")
print("✓ DataFrame consolidado creado: df_consolidad_adres")
print("="*80)

# MOSTRAR MUESTRA DE DATOS
print(f"\n📋 MUESTRA DE DATOS")
print("="*80)
print(df_consolidad_adres.head())
print("\n" + "="*80)
print(f"✓ Información del DataFrame:")
df_consolidad_adres.info()

# 🧹 Mantener solo las columnas requeridas en df_consolidad_adres
df_consolidad_adres = df_consolidad_adres[['AFL_ID', 'ENT_ID', 'TPS_IDN_ID', 'HST_IDN_NUMERO_IDENTIFICACION', 'DPR_ID', 'MNC_ID', 'TPS_EST_AFL_ID']]

print("\n" + "="*80)
print("📊 REDUCCIÓN DE COLUMNAS EN df_consolidad_adres")
print("="*80)
print(f"✓ Columnas seleccionadas: {df_consolidad_adres.columns.tolist()}")
print(f"✓ Total de registros: {len(df_consolidad_adres):,}")
print(f"✓ Total de columnas: {len(df_consolidad_adres.columns)}")

print("\n📋 MUESTRA DE DATOS")
print("="*80)
print(df_consolidad_adres.head(10))

print("\n" + "="*80)
print("ℹ️ INFORMACIÓN DEL DATAFRAME")
print("="*80)
df_consolidad_adres.info()

## df_consolidado_sie

In [ ]:
# Cargar el archivo CSV del Consolidado SIE
df_consolidado_sie = pd.read_csv(
    R_Consolidado_SIE, 
    sep=';', 
    encoding='latin-1', 
    dtype=str
)

print("\n" + "="*80)
print("📊 CARGA DE CONSOLIDADO SIE")
print("="*80)
print(f"✓ Total de registros: {len(df_consolidado_sie):,}")
print(f"✓ Total de columnas: {len(df_consolidado_sie.columns)}")
print(f"\n✓ Columnas del archivo:")
print(df_consolidado_sie.columns.tolist())


print("\n" + "="*80)
print("ℹ️ INFORMACIÓN DEL DATAFRAME")
print("="*80)
df_consolidado_sie.info()

print("\n" + "="*80)
print("📋 PROCESAMIENTO DE CONSOLIDADO SIE")
print("="*80)

import time
tiempo_inicio = time.time()

# ============================================================================
# PASO 1: SELECCIONAR COLUMNAS NECESARIAS
# ============================================================================
columnas_seleccionar = [
    'tipo_documento',
    'numero_identificacion',
    'municipio',
    'estado',
    'regimen',
    'estado_traslado',
    'resguardo_indigena'
]

# Validar que todas las columnas existan
columnas_faltantes = [col for col in columnas_seleccionar if col not in df_consolidado_sie.columns]

if columnas_faltantes:
    print(f"⚠️  ADVERTENCIA: Columnas no encontradas: {columnas_faltantes}")
    print(f"✓ Columnas disponibles: {df_consolidado_sie.columns.tolist()}")
else:
    print(f"✓ Todas las columnas están presentes")

# Seleccionar solo las columnas necesarias
df_consolidado_sie = df_consolidado_sie[columnas_seleccionar]

print(f"\n✓ Columnas ANTES de renombramiento: {df_consolidado_sie.columns.tolist()}")
print(f"✓ Total de registros: {len(df_consolidado_sie):,}")
print(f"✓ Total de columnas: {len(df_consolidado_sie.columns)}")

# ============================================================================
# PASO 2: RENOMBRAR COLUMNAS (MAPPING EXPLÍCITO)
# ============================================================================
diccionario_rename = {
    'municipio': 'municipio_SIE',
    'estado': 'estado_SIE',
    'regimen': 'regimen_SIE',
    'resguardo_indigena': 'PE'
}

df_consolidado_sie = df_consolidado_sie.rename(columns=diccionario_rename)

print(f"\n✓ Columnas DESPUÉS de renombramiento: {df_consolidado_sie.columns.tolist()}")

# ============================================================================
# PASO 3: VALIDACIÓN DE INTEGRIDAD
# ============================================================================
print("\n" + "="*80)
print("🔍 VALIDACIÓN DE INTEGRIDAD")
print("="*80)

# Validar que no hay ninguna columna vieja
columnas_esperadas = ['tipo_documento', 'numero_identificacion', 'municipio_SIE', 
                      'estado_SIE', 'regimen_SIE', 'estado_traslado', 'PE']
columnas_actuales = df_consolidado_sie.columns.tolist()

print(f"\n✓ Columnas esperadas: {columnas_esperadas}")
print(f"✓ Columnas actuales: {columnas_actuales}")
print(f"✓ ¿Coinciden?: {columnas_esperadas == columnas_actuales}")

# Validar nulidad en columnas clave
print(f"\n✓ ANÁLISIS DE NULIDAD:")
print("-" * 80)
for col in df_consolidado_sie.columns:
    nulos = df_consolidado_sie[col].isna().sum()
    porcentaje = (nulos / len(df_consolidado_sie)) * 100
    print(f"  {col:30s}: {nulos:10,d} nulos ({porcentaje:5.1f}%)")

# Validar duplicados
duplicados_clave = df_consolidado_sie.duplicated(
    subset=['tipo_documento', 'numero_identificacion']
).sum()

print(f"\n✓ Duplicados en clave (tipo_documento + numero_identificacion): {duplicados_clave:,}")

# ============================================================================
# PASO 4: MUESTRA DE DATOS
# ============================================================================
print("\n" + "="*80)
print("📋 MUESTRA DE DATOS")
print("="*80)
print(df_consolidado_sie.head(10).to_string())

# ============================================================================
# PASO 5: INFORMACIÓN DEL DATAFRAME
# ============================================================================
print("\n" + "="*80)
print("ℹ️  INFORMACIÓN DEL DATAFRAME")
print("="*80)
df_consolidado_sie.info()

# ============================================================================
# RESUMEN FINAL
# ============================================================================
print("\n" + "="*80)
print("📊 RESUMEN DEL PROCESAMIENTO")
print("="*80)
print(f"✓ Registros totales: {len(df_consolidado_sie):,}")
print(f"✓ Columnas finales: {len(df_consolidado_sie.columns)}")
print(f"✓ Orden de columnas: {df_consolidado_sie.columns.tolist()}")

print(f"\n✓ Renombramiento realizados:")
for columna_vieja, columna_nueva in diccionario_rename.items():
    print(f"  {columna_vieja} → {columna_nueva}")

tiempo_final = time.time()
tiempo_transcurrido = tiempo_final - tiempo_inicio

print(f"\n⏱️  TIEMPO TRANSCURRIDO: {tiempo_transcurrido:.4f} segundos")
print("="*80)

## df_Municipios_SIE

In [ ]:
# Cargar el archivo CSV de Municipios SIE
df_municipios_sie = pd.read_csv(
    R_Municipios_SIE,
    sep=';',
    encoding='latin-1',
    dtype=str
)

print("\n" + "="*80)
print("📊 CARGA DE MUNICIPIOS SIE")
print("="*80)
print(f"✓ Total de registros: {len(df_municipios_sie):,}")
print(f"✓ Total de columnas: {len(df_municipios_sie.columns)}")
print(f"\n✓ Columnas del archivo:")
print(df_municipios_sie.columns.tolist())

print("\n" + "="*80)
print("ℹ️ INFORMACIÓN DEL DATAFRAME")
print("="*80)
df_municipios_sie.info()

## df_red_Contratada

In [ ]:
df_red_contratada = pd.read_csv(
    R_red_Contratada,
    sep=',',
    encoding='ansi',
    dtype=str,
    header=0
)

print("\n" + "="*80)
print("📊 CARGA DE RED CONTRATADA")
print("="*80)
print(f"✓ Total de registros: {len(df_red_contratada):,}")
print(f"✓ Total de columnas: {len(df_red_contratada.columns)}")
print(f"\n✓ Columnas del archivo:")
print(df_red_contratada.columns.tolist())

## df_Historicos_BDUA

In [ ]:
# =============================================================================
# 📂 CARGA DE HISTÓRICOS BDUA (CONTRIBUTIVO + SUBSIDIADO)
# =============================================================================
# Fuentes:
#   - r_historico_epsC25 → Histórico Contributivo (EPSC25)
#   - r_historico_eps025 → Histórico Subsidiado (EPS025)
# Archivos .txt separados por coma (,), sin encabezados, codificación UTF-8.
# Se cargan únicamente las columnas 1, 2 y 3 (índice 0, 1, 2):
#   - AFL_ID: Identificador único ADRES del afiliado (persiste ante cambios de documento)
#   - TPS_IDN_ID: Tipo de documento del afiliado
#   - HST_IDN_NUMERO_IDENTIFICACION: Número de documento del afiliado
# Se eliminan duplicados por las 3 columnas, ya que un afiliado puede aparecer
# en ambos históricos por movilidad entre regímenes.
# =============================================================================

import time
import pandas as pd

start = time.time()

# Definición de columnas a cargar y sus nombres
columnas_a_cargar = [0, 1, 2]
nombres_columnas = ['AFL_ID', 'TPS_IDN_ID', 'HST_IDN_NUMERO_IDENTIFICACION']

# Cargar ambos históricos con solo las columnas necesarias
df_historico_C25 = pd.read_csv(
    r_historico_epsC25,
    sep=',',
    encoding='ansi',
    header=None,
    usecols=columnas_a_cargar,
    names=nombres_columnas,
    dtype=str  # Todo como string para evitar conversiones numéricas indeseadas
)

df_historico_025 = pd.read_csv(
    r_historico_eps025,
    sep=',',
    encoding='ansi',
    header=None,
    usecols=columnas_a_cargar,
    names=nombres_columnas,
    dtype=str
)

registros_C25 = len(df_historico_C25)
registros_025 = len(df_historico_025)

# Unificar ambos DataFrames en uno solo
df_historico_bdua = pd.concat([df_historico_C25, df_historico_025], ignore_index=True)

registros_antes_dedup = len(df_historico_bdua)

# Eliminar duplicados por las 3 columnas (movilidad entre regímenes)
df_historico_bdua.drop_duplicates(
    subset=['AFL_ID', 'TPS_IDN_ID', 'HST_IDN_NUMERO_IDENTIFICACION'],
    keep='first',
    inplace=True
)
df_historico_bdua.reset_index(drop=True, inplace=True)

registros_despues_dedup = len(df_historico_bdua)
duplicados_eliminados = registros_antes_dedup - registros_despues_dedup

# Liberar memoria de los DataFrames parciales
del df_historico_C25, df_historico_025

end = time.time()

# Resumen de carga
print(f"{'='*80}")
print(f"📊 CARGA DE HISTÓRICOS BDUA")
print(f"{'='*80}")
print(f"  ✅ Contributivo (EPSC25):  {registros_C25:>12,} registros")
print(f"  ✅ Subsidiado  (EPS025):   {registros_025:>12,} registros")
print(f"  📋 Total combinado:        {registros_antes_dedup:>12,} registros")
print(f"  🔄 Duplicados eliminados:  {duplicados_eliminados:>12,} registros (movilidad entre regímenes)")
print(f"  ✅ Total final único:      {registros_despues_dedup:>12,} registros")
print(f"  ⏱️  Tiempo de carga:        {end - start:.2f} segundos")
print(f"{'='*80}")
print(f"\n📌 Columnas: {list(df_historico_bdua.columns)}")

# Depuración bases de datos

## Reporte red Asignada General SIE

### Normalización y Pivoteo de Red de Servicios por Afiliado

1. **Objetivo**
Transformar un listado de servicios duplicados por afiliado en un formato normalizado donde cada afiliado ocupa una sola fila y sus servicios asignados se distribuyen en columnas.

2. **Descripción del Proceso**

**Paso 1: Crear Columna de Información del Prestador (SIMPLIFICADA)**
Se crea una nueva columna `prestador_info` que concatena dos datos clave separados por guión:
- **NIT del prestador**: Identificador único de la entidad de salud
- **Código de servicio**: Identificador del tipo de servicio

**Ejemplo:**
844004197-328

Esta información será el valor que se almacenará cuando se pivoteen los servicios.

**Mejora:** Se eliminó la razón social para simplificar el proceso, manteniendo solo la información esencial (NIT + Código de servicio).

---

**Paso 2: Convertir Servicios en Columnas (Pivoting)**
Se utiliza `pivot_table` para transformar la estructura de datos:

| Antes (formato largo) | Después (formato ancho) |
|---|---|
| Múltiples filas por afiliado (una por servicio) | Una sola fila por afiliado |
| Columna `servicio` con valores como "MEDICINA GENERAL", "ODONTOLOGIA", etc. | Cada servicio se convierte en una columna |

**Índices (filas):** Se agrupan por `abreviatura + numero_identificacion + municipio` (identifican únicamente a cada afiliado)

**Columnas:** Los valores únicos de la columna `servicio` se convierten en nuevas columnas

**Valores:** La información concatenada del prestador (`prestador_info` = NIT-Código)

**Manejo de duplicados:** Si un afiliado tiene múltiples prestadores para el mismo servicio, se toma el primer registro (`aggfunc='first'`)

---

**Paso 3: Limpiar Nombres de Columnas**
Se normalizan los nombres de las columnas de servicios:
- Se eliminan espacios en blanco al inicio y final
- Se convierten a mayúsculas para consistencia
- Se asegura que no haya espacios innecesarios

**Ejemplo:** 
- `medicina general` → `MEDICINA GENERAL`
- ` odontologia ` → `ODONTOLOGIA`

---

**Paso 4: Reorganizar Columnas por Cobertura**
Se reordena el DataFrame para que los servicios con mayor cobertura (más datos disponibles) aparezcan primero, facilitando la visualización y análisis de información más completa.

---

3. **Validaciones Aplicadas**

El bloque final de validación genera un reporte detallado que incluye:

| Validación | Descripción |
|---|---|
| **Conteo Antes/Después** | Muestra cuántos registros había originalmente vs. después del pivoting, y calcula el factor de compresión |
| **Servicios Identificados** | Lista todos los tipos de servicios encontrados en la base de datos |
| **Cobertura por Servicio** | Identifica para cada tipo de servicio: cuántos afiliados tienen datos y su porcentaje de cobertura |
| **Análisis de Duplicados en la Fuente** | Cuenta cuántos afiliados tenían múltiples servicios y su distribución |

---

**4. Resultado Final**

Un DataFrame limpio y consolidado donde:
- ✅ Cada fila representa un **afiliado único**
- ✅ Cada columna representa un **tipo de servicio**
- ✅ Las celdas contienen información del **prestador asignado** (NIT-Código)
- ✅ Las celdas vacías indican que ese afiliado **no tiene asignado ese servicio**

**Ejemplo de resultado:**

| abreviatura | numero_identificacion | municipio | MEDICINA GENERAL | ODONTOLOGIA | MEDICAMENTOS |
|---|---|---|---|---|---|
| CC | 1117324769 | Belen - Boyaca | 844004197-328 | 844004197-334 | 1032360739-989 |

**Ventajas del formato simplificado:**
- ✨ Celdas más limpias y legibles
- ⚡ Menor consumo de memoria
- 🔍 Más fácil de buscar e indexar
- 📊 Información esencial preservada (NIT + Servicio)

In [ ]:
import time
import pandas as pd

print("\n" + "="*80)
print("🔄 NORMALIZACIÓN Y PIVOTEO DE RED DE SERVICIOS (OPTIMIZADO)")
print("="*80)

tiempo_inicio = time.time()

# ============================================================================
# PASO 1: CREAR COLUMNA CON INFORMACIÓN AGREGADA (SIMPLIFICADA)
# ============================================================================
# Concatenar SOLO NIT + código (más simple y limpio)
df_red_sie['prestador_info'] = (
    df_red_sie['nit'].astype(str) + '-' + 
    df_red_sie['codigo'].astype(str)
)

# ============================================================================
# PASO 2: PIVOT
# ============================================================================
registros_antes = len(df_red_sie)

df_pivot = df_red_sie.pivot_table(
    index=['abreviatura', 'numero_identificacion', 'municipio'],
    columns='servicio',
    values='prestador_info',
    aggfunc='first'
).reset_index()

registros_despues = len(df_pivot)

# ============================================================================
# PASO 3: LIMPIAR NOMBRES DE COLUMNAS
# ============================================================================
df_pivot.columns.name = None
df_pivot.columns = df_pivot.columns.str.strip().str.upper()

# ============================================================================
# PASO 4: VALIDACIONES (TODO VECTORIZADO)
# ============================================================================
print("\n" + "="*80)
print("📊 VALIDACIÓN DEL PROCESO DE NORMALIZACIÓN")
print("="*80)

# CONTEOS
factor_compresion = registros_antes / registros_despues
print(f"\n✓ Registros ANTES: {registros_antes:,}")
print(f"✓ Registros DESPUÉS: {registros_despues:,}")
print(f"✓ Factor de compresión: {factor_compresion:.1f}x")
print(f"✓ Registros consolidados: {registros_antes - registros_despues:,}")

# ============================================================================
# IDENTIFICAR COLUMNAS DE SERVICIOS
# ============================================================================
columnas_id = {'ABREVIATURA', 'NUMERO_IDENTIFICACION', 'MUNICIPIO'}
servicios = [col for col in df_pivot.columns if col not in columnas_id]

print(f"\n✓ Servicios identificados ({len(servicios)}): {servicios}")

# ============================================================================
# VALIDACIÓN DE NULIDAD - VECTORIZADO
# ============================================================================
print(f"\n✓ ANÁLISIS DE COBERTURA POR SERVICIO:")
print("-" * 80)

nulidad_por_servicio = df_pivot[servicios].isnull().sum()
cobertura_por_servicio = registros_despues - nulidad_por_servicio
porcentaje_cobertura = (cobertura_por_servicio / registros_despues * 100)

# CREAR DATAFRAME DE ANÁLISIS
df_analisis_servicios = pd.DataFrame({
    'Servicio': servicios,
    'Datos': cobertura_por_servicio.values,
    'Vacíos': nulidad_por_servicio.values,
    'Cobertura %': porcentaje_cobertura.values
}).sort_values('Datos', ascending=False)

# Mostrar con formato mejorado
for _, row in df_analisis_servicios.iterrows():
    print(f"  {row['Servicio']:70s} | Datos: {row['Datos']:6.0f} ({row['Cobertura %']:5.1f}%) | Vacíos: {row['Vacíos']:6.0f}")

# ============================================================================
# VALIDACIÓN DE DUPLICADOS EN LA FUENTE
# ============================================================================
print("\n" + "="*80)
print("📋 ANÁLISIS DE DUPLICADOS EN LA FUENTE (df_red_sie)")
print("="*80)

conteos_afiliados = df_red_sie.groupby(['abreviatura', 'numero_identificacion']).size()

duplicados_stats = {
    'Total afiliados únicos (después pivot)': registros_despues,
    'Afiliados con múltiples servicios': (conteos_afiliados > 1).sum(),
    'Servicios promedio por afiliado': registros_antes / registros_despues,
    'Máximo servicios por afiliado': conteos_afiliados.max(),
    'Mínimo servicios por afiliado': conteos_afiliados.min()
}

for clave, valor in duplicados_stats.items():
    if isinstance(valor, float):
        print(f"  ✓ {clave:50s}: {valor:8.2f}")
    else:
        print(f"  ✓ {clave:50s}: {valor:8,d}")

# Mostrar distribución detallada de servicios
print(f"\n  Distribución de servicios por afiliado:")
print("  " + "-" * 76)
distribucion = conteos_afiliados.value_counts().sort_index()
for num_servicios, cantidad_afiliados in distribucion.items():
    porcentaje = (cantidad_afiliados / registros_despues * 100)
    print(f"    {num_servicios:2.0f} servicios: {cantidad_afiliados:7,d} afiliados ({porcentaje:5.1f}%)")

# ============================================================================
# RESUMEN FINAL CONSOLIDADO
# ============================================================================
print("\n" + "="*80)
print("📈 RESUMEN FINAL DE NORMALIZACIÓN")
print("="*80)
print(f"✓ Registros originales (con duplicados): {registros_antes:,}")
print(f"✓ Registros únicos (después pivot): {registros_despues:,}")
print(f"✓ Registros consolidados: {registros_antes - registros_despues:,}")
print(f"✓ Tipo de servicios identificados: {len(servicios)}")
print(f"✓ Cobertura promedio de servicios: {porcentaje_cobertura.mean():.1f}%")
print(f"✓ Servicio con mayor cobertura: {df_analisis_servicios.iloc[0]['Servicio']} ({df_analisis_servicios.iloc[0]['Cobertura %']:.1f}%)")
print(f"✓ Servicio con menor cobertura: {df_analisis_servicios.iloc[-1]['Servicio']} ({df_analisis_servicios.iloc[-1]['Cobertura %']:.1f}%)")

# ============================================================================
# VALIDACIÓN DE INTEGRIDAD
# ============================================================================
print("\n" + "="*80)
print("🔍 VALIDACIÓN DE INTEGRIDAD")
print("="*80)
print(f"✓ Filas en df_pivot: {len(df_pivot):,}")
print(f"✓ Columnas en df_pivot: {len(df_pivot.columns)}")
print(f"✓ Nulos en ABREVIATURA: {df_pivot['ABREVIATURA'].isna().sum():,}")
print(f"✓ Nulos en NUMERO_IDENTIFICACION: {df_pivot['NUMERO_IDENTIFICACION'].isna().sum():,}")
print(f"✓ Nulos en MUNICIPIO: {df_pivot['MUNICIPIO'].isna().sum():,}")
print(f"✓ Primera columna es: {df_pivot.columns[0]}")

# ============================================================================
# MUESTRA DE DATOS (NUEVO - para verificar formato simplificado)
# ============================================================================
print("\n" + "="*80)
print("📋 MUESTRA DE DATOS - FORMATO DE prestador_info SIMPLIFICADO")
print("="*80)

# Mostrar ejemplos de los servicios con el nuevo formato
print(f"\nEjemplos de prestador_info (NIT-Código):")
print("-" * 80)

for col in servicios[:5]:  # Mostrar primeros 5 servicios
    ejemplos = df_pivot[col].dropna().unique()[:3]
    print(f"\n  {col}:")
    for ej in ejemplos:
        print(f"    • {ej}")

tiempo_final = time.time()
tiempo_transcurrido = tiempo_final - tiempo_inicio

print("\n" + "="*80)
print(f"⏱️  TIEMPO TRANSCURRIDO: {tiempo_transcurrido:.4f} segundos")
print("="*80)

## Reorganización de Columnas por Cobertura de Datos

**🎯 Objetivo**
Reorganizar las columnas del DataFrame para que los servicios con **mayor cobertura** (más datos disponibles) aparezcan primero, facilitando la visualización y análisis de la información más completa.

---


#### **Paso 1: Calcular Cobertura por Servicio**

In [ ]:
# CALCULAR CANTIDAD DE VALORES NO VACÍOS POR SERVICIO
cobertura_servicio = df_pivot.isnull().sum().sort_values()

# SEPARAR COLUMNAS EN GRUPOS
columnas_identificacion = ['ABREVIATURA', 'NUMERO_IDENTIFICACION', 'MUNICIPIO']
columnas_servicios = [col for col in df_pivot.columns if col not in columnas_identificacion]

# ORDENAR SERVICIOS POR MENOS VACÍOS (más datos primero)
columnas_servicios_ordenadas = cobertura_servicio[cobertura_servicio.index.isin(columnas_servicios)].index.tolist()

# REORGANIZAR DATAFRAME: Primero identificadores, luego servicios ordenados
df_pivot = df_pivot[columnas_identificacion + columnas_servicios_ordenadas]

print("\n" + "="*80)
print("📋 COLUMNAS REORDENADAS POR COBERTURA (Menos vacíos → Más vacíos)")
print("="*80)

for col in columnas_servicios_ordenadas:
    nulos = df_pivot[col].isna().sum()
    datos = len(df_pivot) - nulos
    porcentaje_datos = ((len(df_pivot) - nulos) / len(df_pivot)) * 100
    print(f"  {col:70s} | Datos: {datos:6d} ({porcentaje_datos:5.1f}%) | Vacíos: {nulos:6d}")

print("\n✓ Nuevo orden de columnas:")
print(df_pivot.columns.tolist())

## Identificación de portabilidad

### PN

In [ ]:
# CREAR NUEVA COLUMNA "Estado_Portabilidad" VACÍA
df_pivot['Estado_Portabilidad'] = None

print("\n" + "="*80)
print("✓ Nueva columna creada: 'Estado_Portabilidad'")
print("="*80)
print(f"Total de registros con la nueva columna: {len(df_pivot)}")
print(f"Columnas actuales: {df_pivot.columns.tolist()}")

print("="*80)

# CREAR COLUMNA TEMPORAL PARA MERGE
df_pt_nac['clave'] = df_pt_nac['tipo_identificacion'] + '_' + df_pt_nac['numero_identificacion'].astype(str)
df_pivot['clave'] = df_pivot['ABREVIATURA'] + '_' + df_pivot['NUMERO_IDENTIFICACION'].astype(str)

# IDENTIFICAR REGISTROS DE PORTABILIDAD EN df_pivot
df_pivot['Estado_Portabilidad'] = df_pivot['clave'].isin(df_pt_nac['clave']).apply(lambda x: 'PN' if x else None)

# CALCULAR ESTADÍSTICAS
total_df_pivot = len(df_pivot)
total_df_pt_nac = len(df_pt_nac)
identificados = df_pivot['Estado_Portabilidad'].notna().sum()
no_identificados = total_df_pivot - identificados

print("\n" + "="*80)
print("📊 IDENTIFICACIÓN DE PORTABILIDAD NACIONAL")
print("="*80)
print(f"✓ Total de registros en df_pivot: {total_df_pivot:,}")
print(f"✓ Total de registros en df_pt_nac: {total_df_pt_nac:,}")
print(f"✓ Registros identificados (PN): {identificados:,} ({(identificados/total_df_pivot)*100:.2f}%)")
print(f"✓ Registros NO identificados: {no_identificados:,} ({(no_identificados/total_df_pivot)*100:.2f}%)")

# AUDITORÍA: IDENTIFICAR REGISTROS DE PORTABILIDAD QUE NO APARECEN EN df_pivot
df_pt_sin_servicios = df_pt_nac[~df_pt_nac['clave'].isin(df_pivot['clave'])]

# CREAR DATAFRAME DE AUDITORÍA
df_auditoria = pd.DataFrame({
    'Tp_D': df_pt_sin_servicios['tipo_identificacion'],
    'No_D': df_pt_sin_servicios['numero_identificacion'],
    'Descripción': 'PN sin servicios'
})

print("\n" + "="*80)
print("📋 AUDITORÍA DE PORTABILIDAD")
print("="*80)
print(f"✓ Registros de portabilidad SIN servicios asignados: {len(df_auditoria):,}")
print(f"\nMuestra de registros en auditoría:")
print(df_auditoria.head(10))

# LIMPIAR COLUMNAS TEMPORALES
df_pivot = df_pivot.drop('clave', axis=1)
df_pt_nac = df_pt_nac.drop('clave', axis=1)

print("\n✓ Columna 'Estado_Portabilidad' actualizada con identificaciones")
print("✓ DataFrame de auditoría creado: df_auditoria")

### PR

In [ ]:
# CREAR COLUMNA TEMPORAL PARA MERGE CON df_pt_reg
df_pt_reg['clave'] = df_pt_reg['tipo_identificacion'] + '_' + df_pt_reg['numero_identificacion'].astype(str)
df_pivot['clave'] = df_pivot['ABREVIATURA'] + '_' + df_pivot['NUMERO_IDENTIFICACION'].astype(str)

# IDENTIFICAR REGISTROS DE PORTABILIDAD REGIONAL EN df_pivot
# Si ya tiene "PN", agregar "PR" para obtener "PN y PR"
df_pivot['Estado_Portabilidad'] = df_pivot.apply(
    lambda row: (row['Estado_Portabilidad'] + ' y PR' if row['Estado_Portabilidad'] else 'PR') 
    if row['clave'] in df_pt_reg['clave'].values else row['Estado_Portabilidad'],
    axis=1
)

# OBTENER EL MUNICIPIO RECEPTOR Y CREAR NUEVA COLUMNA
df_pivot['municipio_receptor'] = None

# Crear un diccionario de clave -> municipio_receptor para búsqueda rápida
dict_municipio = dict(zip(df_pt_reg['clave'], df_pt_reg['municipio_receptor']))

# Asignar los municipios receptores a los registros identificados en PR
df_pivot.loc[df_pivot['clave'].isin(df_pt_reg['clave']), 'municipio_receptor'] = (
    df_pivot.loc[df_pivot['clave'].isin(df_pt_reg['clave']), 'clave'].map(dict_municipio)
)

# CALCULAR ESTADÍSTICAS ANTES Y DESPUÉS
total_antes = len(df_pivot)
identificados_pr = (df_pivot['Estado_Portabilidad'].notna()) & (df_pivot['Estado_Portabilidad'].str.contains('PR', na=False))
total_pr = identificados_pr.sum()
total_pn = df_pivot['Estado_Portabilidad'].eq('PN').sum()
total_pn_y_pr = df_pivot['Estado_Portabilidad'].eq('PN y PR').sum()

print("\n" + "="*80)
print("📊 IDENTIFICACIÓN DE PORTABILIDAD REGIONAL")
print("="*80)
print(f"✓ Total de registros en df_pivot: {total_antes:,}")
print(f"✓ Total de registros en df_pt_reg: {len(df_pt_reg):,}")
print(f"✓ Registros identificados SOLO PR: {(total_pr - total_pn_y_pr):,}")
print(f"✓ Registros identificados PN y PR: {total_pn_y_pr:,}")
print(f"✓ Total registros con PR: {total_pr:,} ({(total_pr/total_antes)*100:.2f}%)")

# AUDITORÍA: IDENTIFICAR REGISTROS DE PORTABILIDAD REGIONAL QUE NO APARECEN EN df_pivot
df_pt_reg_sin_servicios = df_pt_reg[~df_pt_reg['clave'].isin(df_pivot['clave'])]

# AGREGAR A df_auditoria LOS REGISTROS DE PR SIN SERVICIOS
df_auditoria_pr = pd.DataFrame({
    'Tp_D': df_pt_reg_sin_servicios['tipo_identificacion'],
    'No_D': df_pt_reg_sin_servicios['numero_identificacion'],
    'Descripción': 'PR sin servicios'
})

# CONCATENAR CON df_auditoria EXISTENTE
df_auditoria = pd.concat([df_auditoria, df_auditoria_pr], ignore_index=True)

print("\n" + "="*80)
print("📋 AUDITORÍA DE PORTABILIDAD REGIONAL")
print("="*80)
print(f"✓ Registros de portabilidad regional SIN servicios asignados: {len(df_auditoria_pr):,}")
print(f"✓ Total de registros en auditoría: {len(df_auditoria):,}")
print(f"\nMuestra de registros en auditoría:")
print(df_auditoria.head(15))

# LIMPIAR COLUMNAS TEMPORALES
df_pivot = df_pivot.drop('clave', axis=1)
df_pt_reg = df_pt_reg.drop('clave', axis=1)

print("\n" + "="*80)
print("✓ Columna 'Estado_Portabilidad' actualizada con identificaciones PR")
print("✓ Columna 'municipio_receptor' creada con datos de df_pt_reg")
print("✓ DataFrame de auditoría actualizado con registros PR sin servicios")
print("="*80)

### Eliminar df Portabilidad

In [ ]:
del df_pt_nac, df_pt_reg

### Normalizar municipios portabilidad

In [ ]:
import pandas as pd
import gc
from typing import List, Dict

# ============================================================================
# NORMALIZACIÓN DE 'municipio_receptor' A CÓDIGO DANE
# ============================================================================

# 1. Validación de Pre-requisitos
required_pivot: List[str] = ['municipio_receptor', 'ABREVIATURA', 'NUMERO_IDENTIFICACION']
required_cat: List[str] = ['descripcion', 'municipio']

if 'df_municipios_sie' not in locals():
    raise NameError("El DataFrame 'df_municipios_sie' no está definido.")

# Validación de columnas en df_pivot
missing_pivot = [c for c in required_pivot if c not in df_pivot.columns]
if missing_pivot:
    raise ValueError(f"Faltan columnas en df_pivot: {missing_pivot}")

# Validación de columnas en df_municipios_sie
missing_cat = [c for c in required_cat if c not in df_municipios_sie.columns]
if missing_cat:
    raise ValueError(f"Faltan columnas en df_municipios_sie: {missing_cat}")

# 2. Captura de estado inicial para validación
rows_initial: int = len(df_pivot)

# 3. Preparación del Catálogo para Mapeo Vectorizado
# Se normaliza la descripción (llave) a Mayúsculas y sin espacios
mapeo_dane: Dict[str, str] = df_municipios_sie.assign(
    desc_clean=df_municipios_sie['descripcion'].astype(str).str.strip().str.upper()
).drop_duplicates('desc_clean').set_index('desc_clean')['municipio'].astype(str).to_dict()

# 4. Normalización Temporal para Búsqueda
# Creamos una serie con los nombres limpios para el mapeo
serie_nombres_limpios: pd.Series = df_pivot['municipio_receptor'].astype(str).str.strip().str.upper()

# Realizamos el mapeo en una columna temporal
df_pivot['_cod_temp'] = serie_nombres_limpios.map(mapeo_dane)

# 5. Lógica de Identificación de Errores (Criterio Ajustado)
# Solo es error si: TENÍA valor original PERO NO encontró código en el catálogo.
# Los valores nulos o vacíos originales NO se consideran errores y permanecen en df_pivot.

mask_tenia_valor: pd.Series = (
    df_pivot['municipio_receptor'].notna() & 
    (df_pivot['municipio_receptor'].astype(str).str.strip() != '') &
    (df_pivot['municipio_receptor'].astype(str).str.upper() != 'NAN')
)

mask_sin_codigo: pd.Series = df_pivot['_cod_temp'].isna()

# Registros a mover: Tenían nombre de municipio pero el catálogo no los reconoció
mask_movibles_a_auditoria: pd.Series = mask_tenia_valor & mask_sin_codigo

# 6. Gestión de Auditoría y Depuración
if mask_movibles_a_auditoria.any():
    # Extraemos los registros fallidos
    df_fallos: pd.DataFrame = df_pivot.loc[
        mask_movibles_a_auditoria, 
        ['ABREVIATURA', 'NUMERO_IDENTIFICACION', 'municipio_receptor']
    ].copy()
    
    # Estandarizamos para df_auditoria
    df_fallos = df_fallos.rename(columns={
        'ABREVIATURA': 'Tp_D',
        'NUMERO_IDENTIFICACION': 'No_D',
        'municipio_receptor': 'Valor_Original'
    })
    df_fallos['Descripción'] = 'No se identifica codigo DANE municipio receptor (Valor no encontrado en catálogo)'
    
    # Concatenación segura a df_auditoria
    if 'df_auditoria' not in locals() or df_auditoria is None:
        df_auditoria = pd.DataFrame(columns=['Tp_D', 'No_D', 'Descripción', 'Valor_Original'])
    
    df_auditoria = pd.concat([df_auditoria, df_fallos], ignore_index=True)
    
    # Eliminamos de df_pivot solo los que tenían valor pero fallaron la normalización
    # Los que estaban vacíos originalmente se mantienen (como NaNs)
    df_pivot = df_pivot.loc[~mask_movibles_a_auditoria].reset_index(drop=True)
    
    print(f"⚠️ Se movieron {len(df_fallos):,} registros con nombres no reconocidos a auditoría.")
else:
    print("✅ No se detectaron nombres de municipios inválidos.")

# 7. Finalización de la Columna
# Asignamos el código obtenido (los vacíos seguirán siendo vacíos)
df_pivot['municipio_receptor'] = df_pivot['_cod_temp']
df_pivot.drop(columns=['_cod_temp'], inplace=True)

# 8. Validación de Integridad Final
rows_final: int = len(df_pivot)
rows_moved: int = mask_movibles_a_auditoria.sum()

if rows_initial != (rows_final + rows_moved):
    raise AssertionError(
        f"Inconsistencia de datos: Inicial ({rows_initial}) != Final ({rows_final}) + Auditados ({rows_moved})"
    )

print(f"✅ Proceso finalizado. Registros en df_pivot: {rows_final:,}")

# 9. Optimización de Recursos y Eliminación de Catálogo
del required_pivot, required_cat, mapeo_dane, serie_nombres_limpios
del mask_tenia_valor, mask_sin_codigo, mask_movibles_a_auditoria
if 'df_fallos' in locals(): del df_fallos

# Eliminación definitiva del catálogo solicitada
#del df_municipios_sie
gc.collect()

print("🗑️ Catálogo 'df_municipios_sie' eliminado y memoria RAM liberada.")

## Identificar Usaurios Certificados

### LMA

In [ ]:
# CREAR NUEVA COLUMNA "Certificados" VACÍA
df_pivot['Certificados'] = None

print("\n" + "="*80)
print("✓ Nueva columna creada: 'Certificados'")
print("="*80)
print(f"Total de registros con la nueva columna: {len(df_pivot)}")
print(f"Columnas actuales: {df_pivot.columns.tolist()}")

print("="*80)

# CREAR COLUMNA TEMPORAL PARA MERGE
df_lma['clave'] = df_lma['TPS_IDN_ID'] + '_' + df_lma['HST_IDN_NUMERO_IDENTIFICACION'].astype(str)
df_pivot['clave'] = df_pivot['ABREVIATURA'] + '_' + df_pivot['NUMERO_IDENTIFICACION'].astype(str)

# IDENTIFICAR REGISTROS CERTIFICADOS EN df_pivot
df_pivot['Certificados'] = df_pivot['clave'].isin(df_lma['clave']).apply(lambda x: 'LMA' if x else None)

# CALCULAR ESTADÍSTICAS
total_df_pivot = len(df_pivot)
total_df_lma = len(df_lma)
identificados = df_pivot['Certificados'].notna().sum()
no_identificados = total_df_pivot - identificados

print("\n" + "="*80)
print("📊 IDENTIFICACIÓN DE USUARIOS CERTIFICADOS (LMA)")
print("="*80)
print(f"✓ Total de registros en df_pivot: {total_df_pivot:,}")
print(f"✓ Total de registros en df_lma: {total_df_lma:,}")
print(f"✓ Registros identificados (LMA): {identificados:,} ({(identificados/total_df_pivot)*100:.2f}%)")
print(f"✓ Registros NO identificados: {no_identificados:,} ({(no_identificados/total_df_pivot)*100:.2f}%)")

# AUDITORÍA: IDENTIFICAR REGISTROS DE LMA QUE NO APARECEN EN df_pivot
df_lma_sin_servicios = df_lma[~df_lma['clave'].isin(df_pivot['clave'])]

# CREAR DATAFRAME AUXILIAR CON ESTRUCTURA COMPLETA DE df_pivot PARA LMA SIN SERVICIOS
# Obtener nombres de columnas de df_pivot (excepto 'clave')
columnas_pivot = [col for col in df_pivot.columns if col != 'clave']

# Crear DataFrame vacío con la estructura de df_pivot
df_lma_sin_servicios_completo = pd.DataFrame(columns=columnas_pivot)

# Llenar solo las columnas necesarias
for idx, row in df_lma_sin_servicios.iterrows():
    nuevo_registro = {col: None for col in columnas_pivot}
    nuevo_registro['ABREVIATURA'] = row['TPS_IDN_ID']
    nuevo_registro['NUMERO_IDENTIFICACION'] = row['HST_IDN_NUMERO_IDENTIFICACION']
    nuevo_registro['Certificados'] = 'LMA'
    df_lma_sin_servicios_completo = pd.concat(
        [df_lma_sin_servicios_completo, pd.DataFrame([nuevo_registro])],
        ignore_index=True
    )

print("\n" + "="*80)
print("📋 AUDITORÍA DE USUARIOS CERTIFICADOS")
print("="*80)
print(f"✓ Registros de LMA SIN servicios asignados: {len(df_lma_sin_servicios_completo):,}")

# CONCATENAR CON df_auditoria EXISTENTE (si existe)
if len(df_auditoria) > 0:
    print(f"✓ Registros en auditoría antes: {len(df_auditoria):,}")
    
    # Agregar los registros de LMA en formato df_pivot a df_auditoria
    for idx, row in df_lma_sin_servicios_completo.iterrows():
        registro_auditoria = {
            'Tp_D': row['ABREVIATURA'],
            'No_D': row['NUMERO_IDENTIFICACION'],
            'Descripción': 'LMA sin servicios'
        }
        df_auditoria = pd.concat(
            [df_auditoria, pd.DataFrame([registro_auditoria])],
            ignore_index=True
        )
    
    print(f"✓ Registros en auditoría después: {len(df_auditoria):,}")
else:
    # Si df_auditoria no existe, crear con registros de LMA
    df_auditoria = pd.DataFrame({
        'Tp_D': df_lma_sin_servicios['TPS_IDN_ID'],
        'No_D': df_lma_sin_servicios['HST_IDN_NUMERO_IDENTIFICACION'],
        'Descripción': 'LMA sin servicios'
    })

print(f"✓ Total de registros en auditoría: {len(df_auditoria):,}")

# AGREGAR REGISTROS LMA SIN SERVICIOS A df_pivot CON ESTRUCTURA COMPLETA
df_pivot = pd.concat([df_pivot, df_lma_sin_servicios_completo], ignore_index=True)

print(f"\n✓ Registros en df_pivot después de agregar LMA sin servicios: {len(df_pivot):,}")

# LIMPIAR COLUMNAS TEMPORALES
df_pivot = df_pivot.drop('clave', axis=1)
df_lma = df_lma.drop('clave', axis=1)

print("\n" + "="*80)
print("✓ Columna 'Certificados' actualizada con identificaciones LMA")
print("✓ Registros de LMA sin servicios agregados a df_pivot")
print("✓ DataFrame de auditoría actualizado con registros LMA sin servicios")
print(f"✓ Total de registros en df_pivot nuevos registros: {len(df_pivot),}")
print("="*80)

### Compensados

In [ ]:
# CREAR COLUMNA TEMPORAL PARA MERGE CON df_compensados
df_compensados['clave'] = df_compensados['Tp_Dc'] + '_' + df_compensados['Num_coti'].astype(str)
df_pivot['clave'] = df_pivot['ABREVIATURA'] + '_' + df_pivot['NUMERO_IDENTIFICACION'].astype(str)

# IDENTIFICAR REGISTROS COMPENSADOS EN df_pivot
# Si ya tiene "LMA", agregar "y Compensado" para obtener "LMA y Compensado"
# IMPORTANTE: Solo actualizar si Certificados es None o contiene 'Compensado'
df_pivot['Certificados'] = df_pivot.apply(
    lambda row: (row['Certificados'] + ' y Compensado' if pd.notna(row['Certificados']) and row['Certificados'] != '' else 'Compensado') 
    if row['clave'] in df_compensados['clave'].values else row['Certificados'],
    axis=1
)

# CALCULAR ESTADÍSTICAS
total_antes = len(df_pivot)
identificados_comp = (df_pivot['Certificados'].notna()) & (df_pivot['Certificados'].str.contains('Compensado', na=False))
total_comp = identificados_comp.sum()
total_lma = df_pivot['Certificados'].eq('LMA').sum()
total_lma_y_comp = df_pivot['Certificados'].eq('LMA y Compensado').sum()
total_solo_comp = (df_pivot['Certificados'].eq('Compensado')).sum()

print("\n" + "="*80)
print("📊 IDENTIFICACIÓN DE USUARIOS COMPENSADOS")
print("="*80)
print(f"✓ Total de registros en df_pivot: {total_antes:,}")
print(f"✓ Total de registros en df_compensados: {len(df_compensados):,}")
print(f"✓ Registros identificados SOLO Compensado: {total_solo_comp:,}")
print(f"✓ Registros identificados LMA y Compensado: {total_lma_y_comp:,}")
print(f"✓ Registros SOLO LMA (sin cambios): {total_lma:,}")
print(f"✓ Total registros con Compensado: {total_comp:,} ({(total_comp/total_antes)*100:.2f}%)")

# VALIDACIÓN: Contar registros que mantienen "LMA"
registros_lma_mantenidos = df_pivot['Certificados'].eq('LMA').sum()
print(f"✓ Registros LMA preservados: {registros_lma_mantenidos:,}")

# AUDITORÍA: IDENTIFICAR REGISTROS DE COMPENSADOS QUE NO APARECEN EN df_pivot
df_compensados_sin_servicios = df_compensados[~df_compensados['clave'].isin(df_pivot['clave'])]

# CREAR DATAFRAME AUXILIAR CON ESTRUCTURA COMPLETA DE df_pivot PARA COMPENSADOS SIN SERVICIOS
# Obtener nombres de columnas de df_pivot (excepto 'clave')
columnas_pivot = [col for col in df_pivot.columns if col != 'clave']

# Crear DataFrame vacío con la estructura de df_pivot
df_compensados_sin_servicios_completo = pd.DataFrame(columns=columnas_pivot)

# Llenar solo las columnas necesarias
for idx, row in df_compensados_sin_servicios.iterrows():
    nuevo_registro = {col: None for col in columnas_pivot}
    nuevo_registro['ABREVIATURA'] = row['Tp_Dc']
    nuevo_registro['NUMERO_IDENTIFICACION'] = row['Num_coti']
    nuevo_registro['Certificados'] = 'Compensado'
    df_compensados_sin_servicios_completo = pd.concat(
        [df_compensados_sin_servicios_completo, pd.DataFrame([nuevo_registro])],
        ignore_index=True
    )

print("\n" + "="*80)
print("📋 AUDITORÍA DE USUARIOS COMPENSADOS")
print("="*80)
print(f"✓ Registros de Compensados SIN servicios asignados: {len(df_compensados_sin_servicios_completo):,}")

# CONCATENAR CON df_auditoria EXISTENTE (si existe)
if len(df_auditoria) > 0:
    print(f"✓ Registros en auditoría antes: {len(df_auditoria):,}")
    
    # Agregar los registros de Compensados en formato df_pivot a df_auditoria
    for idx, row in df_compensados_sin_servicios_completo.iterrows():
        registro_auditoria = {
            'Tp_D': row['ABREVIATURA'],
            'No_D': row['NUMERO_IDENTIFICACION'],
            'Descripción': 'Compensado sin servicios'
        }
        df_auditoria = pd.concat(
            [df_auditoria, pd.DataFrame([registro_auditoria])],
            ignore_index=True
        )
    
    print(f"✓ Registros en auditoría después: {len(df_auditoria):,}")
else:
    # Si df_auditoria no existe, crear con registros de Compensados
    df_auditoria = pd.DataFrame({
        'Tp_D': df_compensados_sin_servicios['Tp_Dc'],
        'No_D': df_compensados_sin_servicios['Num_coti'],
        'Descripción': 'Compensado sin servicios'
    })

print(f"✓ Total de registros en auditoría: {len(df_auditoria):,}")

# AGREGAR REGISTROS COMPENSADOS SIN SERVICIOS A df_pivot CON ESTRUCTURA COMPLETA
df_pivot = pd.concat([df_pivot, df_compensados_sin_servicios_completo], ignore_index=True)

print(f"\n✓ Registros en df_pivot después de agregar Compensados sin servicios: {len(df_pivot):,}")

# LIMPIAR COLUMNAS TEMPORALES
df_pivot = df_pivot.drop('clave', axis=1)
df_compensados = df_compensados.drop('clave', axis=1)

print("\n" + "="*80)
print("✓ Columna 'Certificados' actualizada con identificaciones de Compensados")
print("✓ Registros LMA preservados sin modificación")
print("✓ Registros de Compensados sin servicios agregados a df_pivot")
print("✓ DataFrame de auditoría actualizado con registros Compensados sin servicios")
print("="*80)

In [ ]:
# CONTAR REGISTROS POR CATEGORÍA EN LA COLUMNA 'Certificados'
conteo_certificados = df_pivot['Certificados'].value_counts(dropna=False)

print("\n" + "="*80)
print("📊 DISTRIBUCIÓN DE REGISTROS POR CATEGORÍA DE CERTIFICADOS")
print("="*80)
print(conteo_certificados)
print("\n" + "="*80)
print("📈 RESUMEN DETALLADO")
print("="*80)

for categoria, cantidad in conteo_certificados.items():
    porcentaje = (cantidad / len(df_pivot)) * 100
    label = categoria if pd.notna(categoria) else "Sin certificado"
    print(f"✓ {label:30s}: {cantidad:7,d} ({porcentaje:6.2f}%)")

print("\n" + "="*80)
print(f"✓ Total de registros: {len(df_pivot):,}")
print("="*80)

### Eliminar certificados

In [ ]:
del df_lma, df_compensados

## Identicar Usuarios Maestro ADRES

In [ ]:
print("\n" + "="*80)
print("⏱️ INICIO: IDENTIFICACIÓN DE USUARIOS EN MAESTRO ADRES")
print("="*80)

import time
tiempo_inicio = time.time()

# PASO 1: CREAR CLAVES VECTORIZADAS (SIN CONVERSIONES AUTOMÁTICAS)
# Convertir explícitamente a string antes de concatenar
df_consolidad_adres['clave'] = (
    df_consolidad_adres['TPS_IDN_ID'].astype(str) + '_' + 
    df_consolidad_adres['HST_IDN_NUMERO_IDENTIFICACION'].astype(str)
)

df_pivot['clave'] = (
    df_pivot['ABREVIATURA'].astype(str) + '_' + 
    df_pivot['NUMERO_IDENTIFICACION'].astype(str)
)

# PASO 2: USAR MERGE EN LUGAR DE DICCIONARIO + BUCLE
# Seleccionar SOLO las columnas necesarias para evitar duplicados
df_adres_merge = df_consolidad_adres[[
    'clave', 'AFL_ID', 'ENT_ID', 'DPR_ID', 'MNC_ID', 'TPS_EST_AFL_ID'
]].copy()

# Renombrar columnas EXPLÍCITAMENTE
df_adres_merge = df_adres_merge.rename(columns={
    'ENT_ID': 'Regimen_Adres',
    'DPR_ID': 'DPR_ADRES',
    'MNC_ID': 'MNC_ADRES',
    'TPS_EST_AFL_ID': 'Estado_ADRES'
})

# VALIDACIÓN: Verificar tipos de datos antes del merge
print("\n" + "="*80)
print("🔍 VALIDACIÓN DE TIPOS DE DATOS ANTES DE MERGE")
print("="*80)
print("df_adres_merge dtypes:")
print(df_adres_merge.dtypes)
print("\ndf_pivot clave dtype:")
print(f"  {df_pivot['clave'].dtype}")

# MERGE LEFT (busca coincidencias, mantiene estructura de df_pivot)
# indicator=True permite validar qué registros fueron encontrados
df_pivot = df_pivot.merge(
    df_adres_merge,
    on='clave',
    how='left',
    indicator=False  # No usar indicator para evitar columna extra
)

# PASO 3: IDENTIFICAR REGISTROS ADRES SIN SERVICIOS (SIN BUCLES)
# Usar isin() que es vectorizado
df_adres_sin_servicios = df_consolidad_adres[
    ~df_consolidad_adres['clave'].isin(df_pivot['clave'].dropna().unique())
].copy()

print("\n" + "="*80)
print("📊 IDENTIFICACIÓN DE USUARIOS EN MAESTRO ADRES")
print("="*80)

total_antes_pivot = len(df_pivot) - len(df_adres_sin_servicios)
total_adres = len(df_consolidad_adres)
identificados_adres = df_pivot['AFL_ID'].notna().sum()
no_identificados = total_antes_pivot - identificados_adres

print(f"✓ Total de registros en df_pivot (antes): {total_antes_pivot:,}")
print(f"✓ Total de registros en df_consolidad_adres: {total_adres:,}")
print(f"✓ Registros identificados (ADRES): {identificados_adres:,} ({(identificados_adres/total_antes_pivot)*100:.2f}%)")
print(f"✓ Registros NO identificados: {no_identificados:,} ({(no_identificados/total_antes_pivot)*100:.2f}%)")

# PASO 4: CREAR REGISTROS SIN SERVICIOS (SIN ITERROWS - VECTORIZADO)
print("\n" + "="*80)
print("📋 AUDITORÍA DE USUARIOS ADRES")
print("="*80)
print(f"✓ Registros de ADRES SIN servicios: {len(df_adres_sin_servicios):,}")

# Obtener lista de columnas de df_pivot
columnas_pivot = [col for col in df_pivot.columns if col != 'clave']

# CREAR DATAFRAME DE NUEVOS REGISTROS CON VALORES EXPLÍCITOS
# Sin loops, todo vectorizado
df_adres_sin_servicios_completo = pd.DataFrame()

# Crear solo con las columnas que tienen valores de ADRES
df_adres_sin_servicios_completo['ABREVIATURA'] = df_adres_sin_servicios['TPS_IDN_ID'].values
df_adres_sin_servicios_completo['NUMERO_IDENTIFICACION'] = df_adres_sin_servicios['HST_IDN_NUMERO_IDENTIFICACION'].values
df_adres_sin_servicios_completo['AFL_ID'] = df_adres_sin_servicios['AFL_ID'].values
df_adres_sin_servicios_completo['Regimen_Adres'] = df_adres_sin_servicios['ENT_ID'].values
df_adres_sin_servicios_completo['DPR_ADRES'] = df_adres_sin_servicios['DPR_ID'].values
df_adres_sin_servicios_completo['MNC_ADRES'] = df_adres_sin_servicios['MNC_ID'].values
df_adres_sin_servicios_completo['Estado_ADRES'] = df_adres_sin_servicios['TPS_EST_AFL_ID'].values

# Agregar columnas restantes con None explícitamente
for col in columnas_pivot:
    if col not in df_adres_sin_servicios_completo.columns:
        df_adres_sin_servicios_completo[col] = None

# Reordenar columnas para que coincidan exactamente con df_pivot
df_adres_sin_servicios_completo = df_adres_sin_servicios_completo[columnas_pivot]

# VALIDACIÓN: Verificar estructura
print("\n" + "="*80)
print("🔍 VALIDACIÓN DE ESTRUCTURA")
print("="*80)
print(f"✓ Columnas df_pivot: {len(df_pivot.columns)}")
print(f"✓ Columnas df_adres_sin_servicios_completo: {len(df_adres_sin_servicios_completo.columns)}")
print(f"✓ ¿Columnas coinciden?: {list(df_pivot.columns) == list(df_adres_sin_servicios_completo.columns)}")

# Verificar tipos de datos coinciden
print("\n🔍 COINCIDENCIA DE TIPOS DE DATOS")
tipos_coinciden = True
for col in df_pivot.columns:
    if col in df_adres_sin_servicios_completo.columns:
        tipo_pivot = df_pivot[col].dtype
        tipo_adres = df_adres_sin_servicios_completo[col].dtype
        if tipo_pivot != tipo_adres:
            # Para columnas None, esto es normal
            if df_adres_sin_servicios_completo[col].dtype == 'object' and tipo_pivot == 'object':
                continue
            print(f"  ⚠️  {col}: pivot={tipo_pivot}, adres={tipo_adres}")
            tipos_coinciden = False

if tipos_coinciden:
    print("  ✅ Todos los tipos de datos coinciden")

# PASO 5: AUDITORÍA SIN BUCLES (VECTORIZADO)
df_auditoria_adres = pd.DataFrame({
    'Tp_D': df_adres_sin_servicios['TPS_IDN_ID'].values,
    'No_D': df_adres_sin_servicios['HST_IDN_NUMERO_IDENTIFICACION'].values,
    'Descripción': ('Adres sin servicios ' + df_adres_sin_servicios['TPS_EST_AFL_ID'].astype(str)).values
})

# Validación de auditoría
print("\n🔍 VALIDACIÓN DE AUDITORÍA")
print(f"  ✓ Registros en auditoría: {len(df_auditoria_adres):,}")
print(f"  ✓ Registros sin Tp_D (nulos): {df_auditoria_adres['Tp_D'].isna().sum():,}")
print(f"  ✓ Registros sin No_D (nulos): {df_auditoria_adres['No_D'].isna().sum():,}")

# Concatenar con auditoría existente
registros_auditoria_antes = len(df_auditoria) if len(df_auditoria) > 0 else 0

if len(df_auditoria) > 0:
    df_auditoria = pd.concat([df_auditoria, df_auditoria_adres], ignore_index=True)
else:
    df_auditoria = df_auditoria_adres.copy()

registros_auditoria_despues = len(df_auditoria)

print(f"  ✓ Registros en auditoría antes: {registros_auditoria_antes:,}")
print(f"  ✓ Registros en auditoría después: {registros_auditoria_despues:,}")
print(f"  ✓ Registros AGREGADOS: {registros_auditoria_despues - registros_auditoria_antes:,}")

# PASO 6: AGREGAR A df_pivot
total_pivot_antes = len(df_pivot)

# Validar que no haya duplicados
duplicados_antes = df_pivot.duplicated(subset=['ABREVIATURA', 'NUMERO_IDENTIFICACION']).sum()
if duplicados_antes > 0:
    print(f"\n⚠️  ADVERTENCIA: Hay {duplicados_antes:,} duplicados potenciales en df_pivot")

df_pivot = pd.concat([df_pivot, df_adres_sin_servicios_completo], ignore_index=True)
total_pivot_despues = len(df_pivot)

# VALIDACIÓN POST-CONCAT
print("\n🔍 VALIDACIÓN POST-CONCATENACIÓN")
print(f"  ✓ Registros df_pivot antes: {total_pivot_antes:,}")
print(f"  ✓ Registros df_pivot después: {total_pivot_despues:,}")
print(f"  ✓ Registros AGREGADOS: {total_pivot_despues - total_pivot_antes:,}")
print(f"  ✓ ¿Suma correcta?: {(total_pivot_antes + len(df_adres_sin_servicios_completo)) == total_pivot_despues}")

# Verificar integridad de datos
print(f"  ✓ Total de filas (sin clave): {len(df_pivot):,}")
print(f"  ✓ Valores nulos en AFL_ID: {df_pivot['AFL_ID'].isna().sum():,}")
print(f"  ✓ Valores nulos en Estado_ADRES: {df_pivot['Estado_ADRES'].isna().sum():,}")

# LIMPIAR COLUMNAS TEMPORALES
df_pivot = df_pivot.drop('clave', axis=1)
df_consolidad_adres = df_consolidad_adres.drop('clave', axis=1)

print("\n" + "="*80)
print("✅ PROCESO COMPLETADO CON VALIDACIÓN DE INTEGRIDAD")
print("="*80)
print("✓ Columnas ADRES agregadas a df_pivot:")
print("  - AFL_ID")
print("  - Regimen_Adres")
print("  - DPR_ADRES")
print("  - MNC_ADRES")
print("  - Estado_ADRES")
print("✓ Registros ADRES sin servicios agregados")
print("✓ DataFrame de auditoría actualizado")
print("="*80)

# RESUMEN FINAL CON VALIDACIONES
print("\n" + "="*80)
print("📈 RESUMEN DEL MOVIMIENTO DE DATOS (CON VALIDACIONES)")
print("="*80)
print(f"✓ Registros totales en df_pivot: {len(df_pivot):,}")
print(f"✓ Registros totales en df_auditoria: {len(df_auditoria):,}")
print(f"✓ Registros con datos ADRES completos: {df_pivot['AFL_ID'].notna().sum():,}")
print(f"✓ Registros ADRES sin servicios: {len(df_adres_sin_servicios_completo):,}")
print(f"✓ Integridad: PN + ADRES sin servicios = {identificados_adres + len(df_adres_sin_servicios_completo):,}")

tiempo_final = time.time()
tiempo_transcurrido = tiempo_final - tiempo_inicio
print(f"\n⏱️ TIEMPO TRANSCURRIDO: {tiempo_transcurrido:.2f} segundos ({tiempo_transcurrido/60:.2f} minutos)")
print("="*80)

In [ ]:
# Obtener lista de todas las columnas
columnas = df_pivot.columns.tolist()

# Remover 'AFL_ID' de su posición actual
columnas.remove('AFL_ID')

# Insertar 'AFL_ID' al inicio
columnas.insert(0, 'AFL_ID')

# Reordenar el DataFrame
df_pivot = df_pivot[columnas]

# Verificar el nuevo orden
print("\n" + "="*80)
print("📊 ORDEN DE COLUMNAS ACTUALIZADO")
print("="*80)
print(f"✓ Primera columna: {df_pivot.columns[0]}")
print(f"✓ Total de columnas: {len(df_pivot.columns)}")
print(f"\n✓ Primeras 10 columnas:")
print(df_pivot.columns[:10].tolist())
print("="*80)

## Identicar Usuarios Maestro SIE

In [ ]:
print("\n" + "="*80)
print("🔗 INTEGRACIÓN DE CONSOLIDADO SIE CON df_pivot")
print("="*80)

import time
tiempo_inicio = time.time()

# ============================================================================
# PASO 1: CREAR CLAVES PARA MERGE (VECTORIZADO)
# ============================================================================
# Crear clave en df_consolidado_sie (convertir a string una sola vez)
df_consolidado_sie['clave'] = (
    df_consolidado_sie['tipo_documento'].astype(str) + '_' +
    df_consolidado_sie['numero_identificacion'].astype(str)
)

# Crear clave en df_pivot (igual estructura)
df_pivot['clave'] = (
    df_pivot['ABREVIATURA'].astype(str) + '_' +
    df_pivot['NUMERO_IDENTIFICACION'].astype(str)
)

print(f"\n✓ Claves creadas en ambos DataFrames")
print(f"  - df_consolidado_sie: {len(df_consolidado_sie):,} registros")
print(f"  - df_pivot: {len(df_pivot):,} registros")

# ============================================================================
# PASO 2: IDENTIFICAR REGISTROS SIE SIN SERVICIOS (VECTORIZADO)
# ============================================================================
# Sin usar loops, todo vectorizado con isin()
df_sie_sin_servicios = df_consolidado_sie[
    ~df_consolidado_sie['clave'].isin(df_pivot['clave'].dropna().unique())
].copy()

print(f"\n✓ Registros SIE SIN servicios identificados: {len(df_sie_sin_servicios):,}")
print(f"  ({(len(df_sie_sin_servicios) / len(df_consolidado_sie) * 100):.2f}% del total)")

# ============================================================================
# PASO 3: PREPARAR DATAFRAME PARA MERGE (SELECCIONAR)
# ============================================================================
# Seleccionar columnas que queremos agregar a df_pivot
columnas_sie_para_merge = ['clave', 'municipio_SIE', 'estado_SIE', 
                           'regimen_SIE', 'estado_traslado', 'PE']

df_sie_merge = df_consolidado_sie[columnas_sie_para_merge].copy()

print(f"\n✓ Columnas SIE seleccionadas para merge: {columnas_sie_para_merge}")

# ============================================================================
# PASO 4: MERGE LEFT (df_pivot es la tabla principal)
# ============================================================================
total_pivot_antes = len(df_pivot)

df_pivot = df_pivot.merge(
    df_sie_merge,
    on='clave',
    how='left',
    indicator=False
)

total_pivot_despues = len(df_pivot)

print(f"\n✓ MERGE completado:")
print(f"  - Registros df_pivot antes: {total_pivot_antes:,}")
print(f"  - Registros df_pivot después: {total_pivot_despues:,}")
print(f"  - Integridad: ¿Se mantienen registros?: {total_pivot_antes == total_pivot_despues}")

# ============================================================================
# PASO 5: CREAR AUDITORÍA CON REGISTROS SIE SIN SERVICIOS (VECTORIZADO)
# ============================================================================
# Crear descripción VECTORIZADA (sin bucles)
descripcion_sie = (
    'SIE sin servicios ' + df_sie_sin_servicios['estado_SIE'].astype(str)
)

# Crear DataFrame de auditoría directamente (sin concatenaciones)
df_auditoria_sie = pd.DataFrame({
    'Tp_D': df_sie_sin_servicios['tipo_documento'].values,
    'No_D': df_sie_sin_servicios['numero_identificacion'].values,
    'Descripción': descripcion_sie.values
})

print(f"\n✓ DataFrame de auditoría SIE creado:")
print(f"  - Registros: {len(df_auditoria_sie):,}")
print(f"  - Columnas: {df_auditoria_sie.columns.tolist()}")

# VALIDACIÓN: Verificar que no hay nulos en descripción
nulos_en_descripcion = df_auditoria_sie['Descripción'].isna().sum()
print(f"  - Valores nulos en Descripción: {nulos_en_descripcion:,}")

# ============================================================================
# PASO 6: AGREGAR REGISTROS SIE A AUDITORÍA (EFICIENTE)
# ============================================================================
registros_auditoria_antes = len(df_auditoria) if 'df_auditoria' in dir() else 0

if 'df_auditoria' in dir() and len(df_auditoria) > 0:
    df_auditoria = pd.concat([df_auditoria, df_auditoria_sie], ignore_index=True)
    registros_auditoria_despues = len(df_auditoria)
    
    print(f"\n✓ Auditoría actualizada:")
    print(f"  - Registros antes: {registros_auditoria_antes:,}")
    print(f"  - Registros agregados: {len(df_auditoria_sie):,}")
    print(f"  - Registros después: {registros_auditoria_despues:,}")
else:
    df_auditoria = df_auditoria_sie.copy()
    print(f"\n✓ DataFrame df_auditoria creado con registros SIE: {len(df_auditoria):,}")

# ============================================================================
# PASO 7: AGREGAR REGISTROS SIE SIN SERVICIOS A df_pivot (VECTORIZADO CORRECTO)
# ============================================================================
# Obtener lista de columnas de df_pivot
columnas_pivot = [col for col in df_pivot.columns if col != 'clave']

# ✅ MÉTODO CORRECTO: Crear DataFrame con los datos de df_sie_sin_servicios
# El truco es usar .reindex() para agregar columnas faltantes con None
df_sie_sin_servicios_completo = df_sie_sin_servicios[['tipo_documento', 'numero_identificacion', 
                                                       'municipio_SIE', 'estado_SIE', 
                                                       'regimen_SIE', 'estado_traslado', 'PE', 'clave']].copy()

# Renombrar columnas para que coincidan con df_pivot
df_sie_sin_servicios_completo = df_sie_sin_servicios_completo.rename(columns={
    'tipo_documento': 'ABREVIATURA',
    'numero_identificacion': 'NUMERO_IDENTIFICACION'
})

# ✅ REINDEXAR: Agregar todas las columnas que falten con NaN/None
df_sie_sin_servicios_completo = df_sie_sin_servicios_completo.reindex(columns=columnas_pivot)

print(f"\n✓ Registros SIE sin servicios preparados para agregar: {len(df_sie_sin_servicios_completo):,}")

# Validar estructura
columnas_coinciden = list(df_pivot.columns) == list(df_sie_sin_servicios_completo.columns)
print(f"  - ¿Columnas coinciden?: {columnas_coinciden}")

if not columnas_coinciden:
    print(f"  ⚠️  ADVERTENCIA: Las columnas no coinciden")
    print(f"  Columnas df_pivot: {list(df_pivot.columns)}")
    print(f"  Columnas df_sie: {list(df_sie_sin_servicios_completo.columns)}")

# Concatenar
total_pivot_antes_concat = len(df_pivot)
df_pivot = pd.concat([df_pivot, df_sie_sin_servicios_completo], ignore_index=True)
total_pivot_despues_concat = len(df_pivot)

print(f"\n✓ Registros SIE sin servicios agregados a df_pivot:")
print(f"  - Registros df_pivot antes: {total_pivot_antes_concat:,}")
print(f"  - Registros agregados: {len(df_sie_sin_servicios_completo):,}")
print(f"  - Registros df_pivot después: {total_pivot_despues_concat:,}")
print(f"  - Integridad suma: {(total_pivot_antes_concat + len(df_sie_sin_servicios_completo)) == total_pivot_despues_concat}")

# ============================================================================
# PASO 8: ELIMINAR COLUMNA CLAVE (LIMPIEZA)
# ============================================================================
df_pivot = df_pivot.drop('clave', axis=1)
df_consolidado_sie = df_consolidado_sie.drop('clave', axis=1)

print(f"\n✓ Columna 'clave' eliminada de ambos DataFrames")

# ============================================================================
# PASO 9: VALIDACIÓN FINAL
# ============================================================================
print("\n" + "="*80)
print("🔍 VALIDACIÓN FINAL DE INTEGRIDAD")
print("="*80)

# Conteo de datos SIE
tiene_datos_sie = df_pivot[['municipio_SIE', 'estado_SIE', 'regimen_SIE']].notna().any(axis=1).sum()
sin_datos_sie = len(df_pivot) - tiene_datos_sie

print(f"\n✓ Distribución de datos SIE en df_pivot:")
print(f"  - Registros CON datos SIE: {tiene_datos_sie:,} ({(tiene_datos_sie/len(df_pivot)*100):.2f}%)")
print(f"  - Registros SIN datos SIE: {sin_datos_sie:,} ({(sin_datos_sie/len(df_pivot)*100):.2f}%)")

# Análisis de nulidad por columna SIE
print(f"\n✓ Análisis de nulidad en columnas SIE:")
print("-" * 80)
columnas_sie = ['municipio_SIE', 'estado_SIE', 'regimen_SIE', 'estado_traslado', 'PE']
for col in columnas_sie:
    nulos = df_pivot[col].isna().sum()
    porcentaje = (nulos / len(df_pivot)) * 100
    print(f"  {col:25s}: {nulos:10,d} nulos ({porcentaje:6.2f}%)")

# Validar integridad de auditoría
print(f"\n✓ Validación de auditoría:")
print(f"  - Total de registros en df_auditoria: {len(df_auditoria):,}")
print(f"  - Registros 'SIE sin servicios': {df_auditoria['Descripción'].str.contains('SIE sin servicios').sum():,}")
print(f"  - Valores nulos en Descripción: {df_auditoria['Descripción'].isna().sum():,}")

# Muestra de descripciones SIE
print(f"\n✓ Ejemplos de descripciones SIE en auditoría:")
print("-" * 80)
descripciones_sie = df_auditoria[df_auditoria['Descripción'].str.contains('SIE sin servicios')]
if len(descripciones_sie) > 0:
    for descripcion in descripciones_sie['Descripción'].unique()[:10]:
        cantidad = (descripciones_sie['Descripción'] == descripcion).sum()
        print(f"  '{descripcion}': {cantidad:,} registros")

# ============================================================================
# RESUMEN FINAL
# ============================================================================
print("\n" + "="*80)
print("📈 RESUMEN DE INTEGRACIÓN SIE")
print("="*80)
print(f"✓ Total de registros en df_consolidado_sie: {len(df_consolidado_sie) + len(df_sie_sin_servicios):,}")
print(f"  - Con servicios (en df_pivot): {tiene_datos_sie:,}")
print(f"  - Sin servicios (en auditoría): {len(df_auditoria_sie):,}")
print(f"\n✓ Total de registros en df_pivot: {len(df_pivot):,}")
print(f"✓ Total de registros en df_auditoria: {len(df_auditoria):,}")
print(f"\n✓ Columnas SIE agregadas a df_pivot:")
for col in columnas_sie:
    print(f"  - {col}")

tiempo_final = time.time()
tiempo_transcurrido = tiempo_final - tiempo_inicio

print(f"\n⏱️  TIEMPO TRANSCURRIDO: {tiempo_transcurrido:.4f} segundos")
print("="*80)

# ============================================================================
# PASO 10: ELIMINACIÓN DE CONSOLIDADO SIE (LIBERACIÓN DE MEMORIA)
# ============================================================================
del df_consolidado_sie

print("\n✓ DataFrame df_consolidado_sie eliminado (memoria liberada)")

## Identificación Población Indigena

In [ ]:
import time

print("\n" + "="*80)
print("🌿 INTEGRACIÓN DE POBLACIÓN INDÍGENA EN df_pivot")
print("="*80)
tiempo_inicio = time.time()

total_registros = len(df_pivot)
total_indigenas = len(df_ms_indigenas)
print(f"✓ Registros actuales en df_pivot: {total_registros:,}")
print(f"✓ Registros en df_ms_indigenas: {total_indigenas:,}")

# -----------------------------------------------------------------------------
# Paso 1: Vaciar columna PE y crear/limpiar MUNICIPIO_PE
# -----------------------------------------------------------------------------
df_pivot['PE'] = None
if 'MUNICIPIO_PE' not in df_pivot.columns:
    posicion_pe = df_pivot.columns.get_loc('PE') + 1
    df_pivot.insert(posicion_pe, 'MUNICIPIO_PE', None)
else:
    df_pivot['MUNICIPIO_PE'] = None

print("\n✓ Columna 'PE' vaciada y 'MUNICIPIO_PE' preparada")

# -----------------------------------------------------------------------------
# Paso 2: Crear claves de emparejamiento
# -----------------------------------------------------------------------------
df_ms_indigenas['clave_pe'] = (
    df_ms_indigenas['T_DOC'].astype(str).str.strip() + '_' +
    df_ms_indigenas['N_DOC'].astype(str).str.strip()
)

df_pivot['clave_pe'] = (
    df_pivot['ABREVIATURA'].astype(str).str.strip() + '_' +
    df_pivot['NUMERO_IDENTIFICACION'].astype(str).str.strip()
)

poblacion_map = dict(zip(df_ms_indigenas['clave_pe'], df_ms_indigenas['POBLACIÓN']))
municipio_map = dict(zip(df_ms_indigenas['clave_pe'], df_ms_indigenas['MUNICIPIO_IPS']))

df_pivot['PE'] = df_pivot['clave_pe'].map(poblacion_map)
df_pivot['MUNICIPIO_PE'] = df_pivot['clave_pe'].map(municipio_map)

actualizados = df_pivot['PE'].notna().sum()

# -----------------------------------------------------------------------------
# Paso 3: Auditoría de usuarios sin coincidencia (si ocurre)
# -----------------------------------------------------------------------------
ms_sin_pivot = df_ms_indigenas[~df_ms_indigenas['clave_pe'].isin(df_pivot['clave_pe'].unique())]
cantidad_sin_pivot = len(ms_sin_pivot)

if not ms_sin_pivot.empty:
    # Auditoría
    df_auditoria_pe = pd.DataFrame({
        'Tp_D': ms_sin_pivot['T_DOC'].values,
        'No_D': ms_sin_pivot['N_DOC'].values,
        'Descripción': 'PE no identificada en la red de servicio'
    })
    df_auditoria = pd.concat([df_auditoria, df_auditoria_pe], ignore_index=True)

    # Filas placeholder en df_pivot
    df_pe_extra = pd.DataFrame({
        'ABREVIATURA': ms_sin_pivot['T_DOC'].values,
        'NUMERO_IDENTIFICACION': ms_sin_pivot['N_DOC'].astype(str).values,
        'PE': ms_sin_pivot['POBLACIÓN'].values,
        'MUNICIPIO_PE': ms_sin_pivot['MUNICIPIO_IPS'].values
    })

    for col in df_pivot.columns:
        if col not in df_pe_extra.columns:
            df_pe_extra[col] = None

    df_pe_extra = df_pe_extra[df_pivot.columns]
    df_pivot = pd.concat([df_pivot, df_pe_extra], ignore_index=True)

# -----------------------------------------------------------------------------
# Paso 4: Limpieza de columnas temporales
# -----------------------------------------------------------------------------
if 'clave_pe' in df_pivot.columns:
    df_pivot = df_pivot.drop(columns='clave_pe')
if 'clave_pe' in df_ms_indigenas.columns:
    df_ms_indigenas = df_ms_indigenas.drop(columns='clave_pe')

# -----------------------------------------------------------------------------
# Resumen final con validación de integridad
# -----------------------------------------------------------------------------
print("\n" + "="*80)
print("📊 RESUMEN DEL PROCESO PE")
print("="*80)

encontrados = actualizados
no_encontrados = cantidad_sin_pivot
total_procesados = encontrados + no_encontrados

print(f"\n  📥 Total población indígena (entrada):  {total_indigenas:,}")
print(f"  ✅ Encontrados en df_pivot:              {encontrados:,} ({encontrados/total_indigenas*100:.2f}%)")
print(f"  ❌ NO encontrados en df_pivot:           {no_encontrados:,} ({no_encontrados/total_indigenas*100:.2f}%)")
print(f"  📊 Total procesados:                     {total_procesados:,}")

# Validación de integridad
print(f"\n  {'─'*60}")
if total_procesados == total_indigenas:
    print(f"  ✅ INTEGRIDAD OK: Todos los {total_indigenas:,} registros fueron procesados")
else:
    diferencia = total_indigenas - total_procesados
    print(f"  ⚠️  PROBLEMA DE INTEGRIDAD: Faltan {diferencia:,} registros por procesar")

if no_encontrados == 0:
    print(f"  ✅ SIN PROBLEMAS: Toda la población indígena fue integrada exitosamente")
else:
    print(f"  ⚠️  {no_encontrados:,} registros enviados a auditoría y agregados como placeholders")

print(f"\n  📋 Registros con PE en df_pivot:         {df_pivot['PE'].notna().sum():,}")
print(f"  📋 Registros con MUNICIPIO_PE:           {df_pivot['MUNICIPIO_PE'].notna().sum():,}")
print(f"  📋 Total registros en df_pivot:           {len(df_pivot):,}")
print(f"  📋 Total registros en df_auditoria:       {len(df_auditoria):,}")

tiempo_final = time.time()
print(f"\n⏱️  Tiempo transcurrido: {tiempo_final - tiempo_inicio:.2f} segundos")
print("="*80)

del df_ms_indigenas

## Identificación Población Alto Costo

### Depurara no altocosto

In [ ]:
import time
import re

print("\n" + "="*80)
print("🏥 NORMALIZACIÓN Y FILTRADO DE ALTO COSTO EN df_ms_altocosto")
print("="*80)
tiempo_inicio = time.time()

total_inicial = len(df_ms_altocosto)
print(f"✓ Registros iniciales en df_ms_altocosto: {total_inicial:,}")

# -----------------------------------------------------------------------------
# Paso 1: Identificar registros que contienen "alto costo" o "altocosto"
# -----------------------------------------------------------------------------
# Crear columna auxiliar: quitar espacios extras, pasar a minúsculas
descripcion_limpia = (
    df_ms_altocosto['descripcion']
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r'\s+', ' ', regex=True)  # múltiples espacios → uno solo
)

# Patrón: detecta "alto costo" o "altocosto" (con o sin espacio)
patron = r'alto\s*costo'

mascara = descripcion_limpia.str.contains(patron, flags=re.IGNORECASE, na=False)

# -----------------------------------------------------------------------------
# Paso 2: Auditoría de categorías encontradas vs eliminadas
# -----------------------------------------------------------------------------
categorias_encontradas = (
    df_ms_altocosto.loc[mascara, 'descripcion']
    .astype(str)
    .str.strip()
    .value_counts()
)

categorias_eliminadas = (
    df_ms_altocosto.loc[~mascara, 'descripcion']
    .astype(str)
    .str.strip()
    .value_counts()
)

# -----------------------------------------------------------------------------
# Paso 3: Filtrar solo registros que coincidan y estandarizar
# -----------------------------------------------------------------------------
registros_conservados = mascara.sum()
registros_eliminados = total_inicial - registros_conservados

df_ms_altocosto = df_ms_altocosto[mascara].copy()
df_ms_altocosto['descripcion'] = 'ALTO COSTO'

# -----------------------------------------------------------------------------
# Resumen final
# -----------------------------------------------------------------------------
print("\n" + "="*80)
print("📊 RESUMEN DEL PROCESO")
print("="*80)

print(f"\n  📥 Total registros iniciales:            {total_inicial:,}")
print(f"  ✅ Registros conservados (Alto Costo):    {registros_conservados:,} ({registros_conservados/total_inicial*100:.2f}%)")
print(f"  ❌ Registros eliminados (otros):          {registros_eliminados:,} ({registros_eliminados/total_inicial*100:.2f}%)")
print(f"  🔄 Registros estandarizados a 'ALTO COSTO': {registros_conservados:,}")

if not categorias_encontradas.empty:
    print(f"\n  {'─'*60}")
    print(f"  📋 CATEGORÍAS NORMALIZADAS → 'ALTO COSTO':")
    for categoria, cantidad in categorias_encontradas.items():
        print(f"     • \"{categoria}\" → 'ALTO COSTO'  ({cantidad:,} registros)")

if not categorias_eliminadas.empty:
    print(f"\n  {'─'*60}")
    print(f"  🗑️  CATEGORÍAS ELIMINADAS (no contienen 'Alto Costo'):")
    for categoria, cantidad in categorias_eliminadas.items():
        print(f"     • \"{categoria}\"  ({cantidad:,} registros)")

print(f"\n  {'─'*60}")
print(f"  📋 Total final en df_ms_altocosto:       {len(df_ms_altocosto):,}")

if registros_eliminados == 0:
    print(f"  ✅ SIN PROBLEMAS: Todos los registros contenían 'Alto Costo'")
else:
    print(f"  ⚠️  Se eliminaron {registros_eliminados:,} registros que no contenían 'Alto Costo'")

tiempo_final = time.time()
print(f"\n⏱️  Tiempo transcurrido: {tiempo_final - tiempo_inicio:.2f} segundos")
print("="*80)

### Altocosto en df_pivot

In [ ]:
import time

print("\n" + "="*80)
print("🏥 INTEGRACIÓN DE ALTO COSTO EN df_pivot")
print("="*80)
tiempo_inicio = time.time()

# 1) Crear claves de emparejamiento
df_ms_altocosto['clave_altocosto'] = (
    df_ms_altocosto['tipo_documento'].astype(str).str.strip() + '_' +
    df_ms_altocosto['numero_documento'].astype(str).str.strip()
)

df_pivot['clave_altocosto'] = (
    df_pivot['ABREVIATURA'].astype(str).str.strip() + '_' +
    df_pivot['NUMERO_IDENTIFICACION'].astype(str).str.strip()
)

# 2) Traer columna 'descripcion' (marcador ALTO COSTO)
mapa_altocosto = dict(zip(df_ms_altocosto['clave_altocosto'], df_ms_altocosto['descripcion']))
df_pivot['descripcion_altocosto'] = df_pivot['clave_altocosto'].map(mapa_altocosto)

# 3) Métricas de identificación
total_altocosto = len(df_ms_altocosto)
registros_identificados = df_pivot['descripcion_altocosto'].notna().sum()
personas_identificadas = df_pivot.loc[df_pivot['descripcion_altocosto'].notna(), 'clave_altocosto'].nunique()
porcentaje_identificados = (personas_identificadas / total_altocosto * 100) if total_altocosto else 0
no_identificados = total_altocosto - personas_identificadas

print(f"\n✓ Total registros Alto Costo (df_ms_altocosto): {total_altocosto:,}")
print(f"✓ Registros marcados en df_pivot (filas):        {registros_identificados:,}")
print(f"✓ Personas únicas identificadas:                {personas_identificadas:,}")
print(f"✓ Cobertura sobre Alto Costo:                   {porcentaje_identificados:6.2f}%")
print(f"✓ Personas no encontradas:                      {no_identificados:,}")


# 6) Limpieza de columnas temporales
df_pivot.drop(columns='clave_altocosto', inplace=True)
df_ms_altocosto.drop(columns='clave_altocosto', inplace=True)

tiempo_final = time.time()
print("\n" + "="*80)
print(f"⏱️  Tiempo transcurrido: {tiempo_final - tiempo_inicio:.4f} segundos")
print("="*80)

del df_ms_altocosto

## Validar duplicados df_pivot

### Asignar codigo BDUA

In [ ]:
import pandas as pd
import gc
from typing import Dict

# ============================================================================
# ACTUALIZACIÓN DE AFL_ID (Serial BDUA) - ESTRATEGIA MULTI-FUENTE Y AUDITORÍA
# ============================================================================

print("=" * 70)
print("ACTUALIZACIÓN DE AFL_ID (Serial BDUA)")
print("=" * 70)

def normalizar_clave(df: pd.DataFrame, col_tipo: str, col_num: str) -> pd.Series:
    """Estandariza mayúsculas, elimina espacios y suprime decimales fantasmas (.0)."""
    tipo = df[col_tipo].astype(str).str.strip().str.upper()
    numero = df[col_num].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
    return tipo + '_' + numero

# 1. Creación de Claves de Cruce (Vectorizado y Normalizado)
df_pivot['_clave_cruce'] = normalizar_clave(df_pivot, 'ABREVIATURA', 'NUMERO_IDENTIFICACION')
df_historico_bdua['_clave_cruce'] = normalizar_clave(df_historico_bdua, 'TPS_IDN_ID', 'HST_IDN_NUMERO_IDENTIFICACION')
df_consolidad_adres['_clave_cruce'] = normalizar_clave(df_consolidad_adres, 'TPS_IDN_ID', 'HST_IDN_NUMERO_IDENTIFICACION')

# 2. Construcción de Diccionarios de Mapeo (Excluyendo nulos)
mapeo_historico: Dict[str, str] = (
    df_historico_bdua.dropna(subset=['AFL_ID'])
    .drop_duplicates(subset='_clave_cruce', keep='last')
    .set_index('_clave_cruce')['AFL_ID']
    .to_dict()
)

mapeo_adres: Dict[str, str] = (
    df_consolidad_adres.dropna(subset=['AFL_ID'])
    .drop_duplicates(subset='_clave_cruce', keep='last')
    .set_index('_clave_cruce')['AFL_ID']
    .to_dict()
)

# 3. Mapeo y Auditoría
total_pivot = len(df_pivot)

# 3.1 Evaluar cruce con Histórico BDUA
serie_historico = df_pivot['_clave_cruce'].map(mapeo_historico)
match_historico = serie_historico.notna().sum()

# 3.2 Evaluar cruce con ADRES y consolidar
serie_adres = df_pivot['_clave_cruce'].map(mapeo_adres)
serie_consolidada = serie_historico.fillna(serie_adres)

# 3.3 Calcular métricas de auditoría
total_encontrados = serie_consolidada.notna().sum()
match_adres = total_encontrados - match_historico
no_encontrados = total_pivot - total_encontrados

# 3.4 Asignación final al DataFrame
df_pivot['AFL_ID'] = serie_consolidada.fillna('')

# 4. Impresión de Resultados (Auditoría en Consola)
print("\n--- RESUMEN DE CRUCE (AUDITORÍA) ---")
print(f"Total de registros procesados : {total_pivot:,}")
print(f" [+] Asignados por BDUA        : {match_historico:,} ({match_historico/total_pivot:.1%})")
print(f" [+] Asignados por ADRES       : {match_adres:,} ({match_adres/total_pivot:.1%})")
print(f" [-] No encontrados (vacíos)   : {no_encontrados:,} ({no_encontrados/total_pivot:.1%})")
print("-" * 36)

# 5. Limpieza de Memoria
df_pivot.drop(columns=['_clave_cruce'], inplace=True)
df_historico_bdua.drop(columns=['_clave_cruce'], inplace=True, errors='ignore')
df_consolidad_adres.drop(columns=['_clave_cruce'], inplace=True, errors='ignore')

del mapeo_historico, mapeo_adres, serie_historico, serie_adres, serie_consolidada
gc.collect()

### ESTADÍSTICAS DE IDENTIFICACIÓN

In [ ]:
# ============================================================================
# ESTADÍSTICAS DE IDENTIFICACIÓN
# ============================================================================
total_registros = len(df_pivot)
identificados = (df_pivot['AFL_ID'].astype(str).str.strip() != '').sum()
no_identificados = total_registros - identificados
pct_identificados = (identificados / total_registros) * 100

print(f"\nTotal registros en df_pivot:        {total_registros:>10,}")
print(f"Registros identificados (AFL_ID):   {identificados:>10,} ({pct_identificados:.2f}%)")
print(f"Registros NO identificados:         {no_identificados:>10,} ({(100-pct_identificados):.2f}%)")

## ANÁLISIS DE DUPLICADOS POR AFL_ID Y CATEGORIZACIÓN

In [ ]:
# ============================================================================
# ANÁLISIS DE DUPLICADOS POR AFL_ID Y CATEGORIZACIÓN
# ============================================================================
print("\n" + "=" * 70)
print("ANÁLISIS DE DUPLICADOS POR AFL_ID (Serial BDUA)")
print("=" * 70)

df_con_id = df_pivot[df_pivot['AFL_ID'].astype(str).str.strip() != ''].copy()
duplicados_afl = df_con_id[df_con_id.duplicated(subset=['AFL_ID'], keep=False)].copy()

total_con_id = len(df_con_id)
total_duplicados = len(duplicados_afl)
seriales_duplicados = duplicados_afl['AFL_ID'].nunique()

print(f"\nRegistros con AFL_ID asignado:      {total_con_id:>10,}")
print(f"Registros en grupos duplicados:     {total_duplicados:>10,} ({(total_duplicados/total_con_id*100 if total_con_id > 0 else 0):.2f}%)")
print(f"Seriales BDUA que se repiten:       {seriales_duplicados:>10,}")

if total_duplicados > 0:
    doc_hierarchy = {
        'MS': 1, 'CN': 2, 'RC': 3, 'TI': 4, 'PE': 5, 
        'SC': 6, 'PT': 7, 'CC': 8, 'CE': 9
    }

    def sort_evolution(docs):
        unique_docs = sorted(list(set(docs)), key=lambda x: doc_hierarchy.get(x, 99))
        return ' → '.join(unique_docs)

    grupos_dup = duplicados_afl.groupby('AFL_ID')['ABREVIATURA'].apply(sort_evolution).reset_index()
    grupos_dup.columns = ['AFL_ID', 'CATEGORIA_TIPOS']

    duplicados_afl = duplicados_afl.merge(grupos_dup, on='AFL_ID', how='left')

    evoluciones_conocidas = {
        'CN → RC': 'Evolución CN a RC (Certificado Nacido Vivo → Registro Civil)',
        'RC → TI': 'Evolución RC a TI (Registro Civil → Tarjeta de Identidad)',
        'TI → CC': 'Evolución TI a CC (Tarjeta de Identidad → Cédula de Ciudadanía)',
        'CN → RC → TI': 'Evolución CN a RC a TI',
        'CN → TI': 'Evolución CN a TI',
        'PE → PT': 'Evolución PE a PT',
        'SC → PT': 'Evolución SC a PT',
        'RC → CC': 'Evolución RC a CC',
        'CN → CC': 'Evolución CN a CC',
        'MS → TI': 'Evolución MS a TI',
        'CN → RC → CC': 'Evolución CN a RC a CC',
        'CN → RC → TI → CC': 'Evolución completa CN a RC a TI a CC',
        'RC → TI → CC': 'Evolución RC a TI a CC',
        'TI → CC → CE': 'Evolución TI a CC a CE',
    }

    resumen_categorias = duplicados_afl.groupby('CATEGORIA_TIPOS').agg(
        REGISTROS=('AFL_ID', 'count'),
        SERIALES_UNICOS=('AFL_ID', 'nunique')
    ).sort_values('REGISTROS', ascending=False).reset_index()

    resumen_categorias['PCT_REGISTROS'] = (resumen_categorias['REGISTROS'] / total_duplicados * 100).round(2)
    resumen_categorias['DESCRIPCION'] = resumen_categorias['CATEGORIA_TIPOS'].map(evoluciones_conocidas).fillna('⚠️ NO CONTEMPLADA')

    print(f"\n{'CATEGORÍA':<30} {'REGISTROS':>10} {'%':>8} {'SERIALES':>10} {'DESCRIPCIÓN'}")
    print("-" * 110)

    for _, row in resumen_categorias.iterrows():
        print(f"{row['CATEGORIA_TIPOS']:<30} {row['REGISTROS']:>10,} {row['PCT_REGISTROS']:>7.2f}% {row['SERIALES_UNICOS']:>10,} {row['DESCRIPCION']}")

    no_contempladas = resumen_categorias[resumen_categorias['DESCRIPCION'] == '⚠️ NO CONTEMPLADA']
    contempladas = resumen_categorias[resumen_categorias['DESCRIPCION'] != '⚠️ NO CONTEMPLADA']

    print(f"\n{'─' * 70}")
    print(f"Categorías de evolución conocidas:     {len(contempladas):>5} ({contempladas['REGISTROS'].sum():>10,} registros)")
    print(f"Categorías NO contempladas:            {len(no_contempladas):>5} ({no_contempladas['REGISTROS'].sum():>10,} registros)")

else:
    print("\n✅ No se encontraron duplicados por AFL_ID.")

# Limpieza final de variables del bloque
del df_con_id, duplicados_afl, resumen_categorias, grupos_dup
gc.collect()

print("\n" + "=" * 70)
print("FIN DEL ANÁLISIS")
print("=" * 70)

### Quitar duplicados por evoluciones

In [ ]:
import pandas as pd
import numpy as np
import gc

# ============================================================================
# FUSIÓN Y ELIMINACIÓN DE DUPLICADOS POR EVOLUCIÓN (AFL_ID)
# ============================================================================
print("\n" + "=" * 70)
print("CONSOLIDACIÓN DE DATOS Y ELIMINACIÓN DE DUPLICADOS")
print("=" * 70)

# 1. Definir columnas objetivo para consolidación
cols_consolidadas = [
    'Regimen_Adres', 'ID_MNC_ADRES', 'Estado_ADRES', 'municipio_SIE',
    'estado_SIE', 'regimen_SIE', 'estado_traslado', 'PE', 'MUNICIPIO_PE',
    'descripcion_altocosto', 'Certificados', 'Estado_Portabilidad', 'municipio_receptor'
]

# Validar que las columnas existan en el DataFrame para evitar KeyError
cols_presentes = [c for c in cols_consolidadas if c in df_pivot.columns]
if len(cols_presentes) < len(cols_consolidadas):
    faltantes = set(cols_consolidadas) - set(cols_presentes)
    print(f"⚠️ Advertencia: No se encontraron estas columnas y serán omitidas: {faltantes}")

# 2. Separar registros válidos de los huérfanos
mask_afl_valido = df_pivot['AFL_ID'].astype(str).str.strip() != ''
df_validos = df_pivot[mask_afl_valido].copy()
df_vacios = df_pivot[~mask_afl_valido].copy()

total_inicial = len(df_validos)

# 3. Normalizar vacíos a NaN para habilitar la propagación matemática
df_validos[cols_presentes] = df_validos[cols_presentes].replace(r'^\s*$', np.nan, regex=True)

# 4. Asignar Jerarquía y Ordenar
doc_hierarchy = {
    'MS': 1, 'CN': 2, 'RC': 3, 'TI': 4, 'PE': 5, 
    'SC': 6, 'PT': 7, 'CC': 8, 'CE': 9
}
# Mapeamos la jerarquía. Si hay un tipo no contemplado, recibe 0.
df_validos['_jerarquia'] = df_validos['ABREVIATURA'].str.strip().str.upper().map(doc_hierarchy).fillna(0)

# Orden ascendente: Los de mayor jerarquía quedan al final del grupo
df_validos.sort_values(by=['AFL_ID', '_jerarquia'], ascending=[True, True], inplace=True)

# 5. Consolidación de Datos (Data Fusion) Vectorizada
# Propaga valores no nulos hacia adelante y luego hacia atrás dentro de cada grupo
df_validos[cols_presentes] = df_validos.groupby('AFL_ID')[cols_presentes].ffill()
df_validos[cols_presentes] = df_validos.groupby('AFL_ID')[cols_presentes].bfill()

# 6. Deduplicación Estricta
# Conservamos el 'last' (último), que corresponde a la jerarquía más alta por nuestro sort
df_validos.drop_duplicates(subset=['AFL_ID'], keep='last', inplace=True)

total_final = len(df_validos)
registros_eliminados = total_inicial - total_final

# 7. Limpieza Final y Reensamblaje
df_validos.drop(columns=['_jerarquia'], inplace=True)
# Revertimos los NaN que quedaron (columnas que estaban vacías en todos los duplicados) a string vacío
df_validos[cols_presentes] = df_validos[cols_presentes].fillna('')

df_pivot = pd.concat([df_validos, df_vacios], ignore_index=True)

# 8. Auditoría
print(f"Registros procesados (con AFL_ID) : {total_inicial:>10,}")
print(f"Registros consolidados finales    : {total_final:>10,}")
print(f"Duplicados obsoletos eliminados   : {registros_eliminados:>10,}")
print(f"Total general en df_pivot actual  : {len(df_pivot):>10,}")

# Liberar RAM
del df_validos, df_vacios
gc.collect()

### Morver a auditoria sin ADRES y no compensados

In [ ]:
import pandas as pd
import gc
from typing import List

# 1. Validación de pre-requisitos: Columnas requeridas en df_pivot
required_columns: List[str] = ['Estado_ADRES', 'Certificados', 'ABREVIATURA', 'NUMERO_IDENTIFICACION']
missing_cols: List[str] = [col for col in required_columns if col not in df_pivot.columns]

if missing_cols:
    raise ValueError(f"Error Crítico: Faltan las siguientes columnas en df_pivot: {missing_cols}")

# 2. Captura de estado inicial para validación de integridad
initial_rows_pivot: int = len(df_pivot)

# 3. Identificación de registros a mover (Criterio: Sin ADRES Y No Compensado/Pagado)
# Se manejan nulos (NaN), strings vacíos "" y strings con solo espacios " "
mask_no_adres: pd.Series = (
    df_pivot['Estado_ADRES'].isna() | 
    (df_pivot['Estado_ADRES'].astype(str).str.strip() == "")
)

mask_no_cert: pd.Series = (
    df_pivot['Certificados'].isna() | 
    (df_pivot['Certificados'].astype(str).str.strip() == "")
)

# El criterio es la intersección (AND) de ambas condiciones de ausencia de información
mask_to_move: pd.Series = mask_no_adres & mask_no_cert

# 4. Extracción y Renombramiento de registros para auditoría
# Se seleccionan las columnas y se renombran inmediatamente para coincidir con df_auditoria
df_to_audit: pd.DataFrame = df_pivot.loc[
    mask_to_move, ['ABREVIATURA', 'NUMERO_IDENTIFICACION']
].copy().rename(columns={
    'ABREVIATURA': 'Tp_D',
    'NUMERO_IDENTIFICACION': 'No_D'
})

# Asignación de la descripción técnica del hallazgo
df_to_audit['Descripción'] = "No existe MS ADRES y no Certificado"

# 5. Concatenación a df_auditoria (Asegurando estructura [Tp_D, No_D, Descripción])
cols_auditoria: List[str] = ['Tp_D', 'No_D', 'Descripción']

if 'df_auditoria' not in locals() or df_auditoria is None:
    df_auditoria = pd.DataFrame(columns=cols_auditoria)
else:
    # Si df_auditoria ya existe, validamos que tenga las columnas correctas o lo reestructuramos
    # Esto previene errores si la celda se ejecuta después de versiones anteriores del código
    if not all(c in df_auditoria.columns for c in cols_auditoria):
        # Si tiene columnas viejas (ABREVIATURA/NUMERO_IDENTIFICACION), las renombramos para no perder datos previos
        rename_map = {'ABREVIATURA': 'Tp_D', 'NUMERO_IDENTIFICACION': 'No_D'}
        df_auditoria = df_auditoria.rename(columns=rename_map)
    
    # Aseguramos que solo queden las 3 columnas solicitadas
    df_auditoria = df_auditoria[cols_auditoria]

df_auditoria = pd.concat([df_auditoria, df_to_audit], ignore_index=True)

# 6. Depuración de df_pivot (Eliminación de registros movidos)
df_pivot = df_pivot.loc[~mask_to_move].reset_index(drop=True)

# 7. Validaciones de Integridad Finales
final_rows_pivot: int = len(df_pivot)
moved_count: int = len(df_to_audit)

if initial_rows_pivot != (final_rows_pivot + moved_count):
    raise AssertionError(
        f"Inconsistencia de datos: Inicial ({initial_rows_pivot}) != "
        f"Final ({final_rows_pivot}) + Movidos ({moved_count}). "
        "Se pudo haber perdido información en el proceso."
    )

print(f"Proceso completado exitosamente.")
print(f"Registros movidos a auditoría: {moved_count}")
print(f"Registros restantes en df_pivot: {final_rows_pivot}")

# 8. Liberación de memoria (Optimización de recursos)
del required_columns, missing_cols, mask_no_adres, mask_no_cert, mask_to_move, df_to_audit, cols_auditoria
gc.collect()

## Depurar AF, SD y RE

In [ ]:
import pandas as pd
import gc
from typing import List

# 1. Validación de Pre-requisitos: Columnas requeridas en df_pivot
required_cols: List[str] = ['Estado_ADRES', 'estado_SIE', 'ABREVIATURA', 'NUMERO_IDENTIFICACION']
missing_cols: List[str] = [col for col in required_cols if col not in df_pivot.columns]

if missing_cols:
    raise ValueError(f"Error Crítico: Faltan las siguientes columnas en df_pivot: {missing_cols}")

# 2. Captura de estado inicial para validación de integridad
initial_rows_pivot: int = len(df_pivot)

# 3. Definición de criterios de depuración (Estados Prohibidos)
status_adres_exclude: List[str] = ['AF', 'RE', 'SD', 'SM']
status_sie_exclude: List[str] = ['AF', 'SD', 'SM']

# Generación de máscaras booleanas (Normalizadas con strip)
mask_adres: pd.Series = df_pivot['Estado_ADRES'].astype(str).str.strip().isin(status_adres_exclude)
mask_sie: pd.Series = df_pivot['estado_SIE'].astype(str).str.strip().isin(status_sie_exclude)

# Análisis de Intersección y Diferencia (Para visibilidad en el reporte)
mask_both: pd.Series = mask_adres & mask_sie
mask_only_adres: pd.Series = mask_adres & ~mask_sie
mask_only_sie: pd.Series = mask_sie & ~mask_adres
mask_total_remove: pd.Series = mask_adres | mask_sie

# 4. Creación de registros para Auditoría - Caso ADRES
df_audit_adres: pd.DataFrame = df_pivot.loc[mask_adres, ['ABREVIATURA', 'NUMERO_IDENTIFICACION']].copy()
df_audit_adres = df_audit_adres.rename(columns={'ABREVIATURA': 'Tp_D', 'NUMERO_IDENTIFICACION': 'No_D'})
df_audit_adres['Descripción'] = "Afiliado " + df_pivot.loc[mask_adres, 'Estado_ADRES'].astype(str) + " en ADRES"

# 5. Creación de registros para Auditoría - Caso SIE
df_audit_sie: pd.DataFrame = df_pivot.loc[mask_sie, ['ABREVIATURA', 'NUMERO_IDENTIFICACION']].copy()
df_audit_sie = df_audit_sie.rename(columns={'ABREVIATURA': 'Tp_D', 'NUMERO_IDENTIFICACION': 'No_D'})
df_audit_sie['Descripción'] = "Afiliado " + df_pivot.loc[mask_sie, 'estado_SIE'].astype(str) + " en SIE"

# 6. Consolidación en df_auditoria
target_audit_cols: List[str] = ['Tp_D', 'No_D', 'Descripción']
if 'df_auditoria' not in locals() or df_auditoria is None:
    df_auditoria = pd.DataFrame(columns=target_audit_cols)

df_new_audit: pd.DataFrame = pd.concat([df_audit_adres, df_audit_sie], ignore_index=True)
df_auditoria = pd.concat([df_auditoria, df_new_audit[target_audit_cols]], ignore_index=True)

# 7. Depuración de df_pivot (Operación Crítica)
df_pivot = df_pivot.loc[~mask_total_remove].reset_index(drop=True)

# 8. Reporte Detallado de Identificación (Visibilidad de Solapamientos)
final_rows_pivot: int = len(df_pivot)

print("-" * 65)
print(f"{'RESUMEN DETALLADO DE DEPURACIÓN (ESTADOS CRÍTICOS)':^65}")
print("-" * 65)
print(f"1. Registros que fallan SOLO en ADRES:      {mask_only_adres.sum():>10,}")
print(f"2. Registros que fallan SOLO en SIE:        {mask_only_sie.sum():>10,}")
print(f"3. Registros que fallan en AMBOS (Cruce):   {mask_both.sum():>10,}")
print("-" * 65)
print(f"Total registros únicos eliminados (1+2+3):  {mask_total_remove.sum():>10,}")
print(f"Total registros en Auditoría (Con dups):    {len(df_new_audit):>10,}")
print(f"Registros restantes en df_pivot:            {final_rows_pivot:>10,}")
print("-" * 65)

# Validación de seguridad: No se deben perder registros fuera de la máscara
if initial_rows_pivot - mask_total_remove.sum() != final_rows_pivot:
    raise AssertionError("Error de Integridad: El conteo de filas final no coincide con la depuración.")

# 9. Liberación de memoria
del required_cols, status_adres_exclude, status_sie_exclude
del mask_adres, mask_sie, mask_both, mask_only_adres, mask_only_sie, mask_total_remove
del df_audit_adres, df_audit_sie, df_new_audit, target_audit_cols
gc.collect()

## Depurar df_auditoria

In [ ]:
conteo_desc = df_auditoria['Descripción'].value_counts(dropna=False)
porcentaje_desc = (conteo_desc / len(df_auditoria) * 100).round(2)

df_resumen_desc = (
    pd.DataFrame({
        'Descripción': conteo_desc.index.astype(str),
        'Registros': conteo_desc.values,
        'Porcentaje': porcentaje_desc.values
    })
    .sort_values('Registros', ascending=False)
    .reset_index(drop=True)
)

print("\n📊 Distribución de df_auditoria['Descripción']")
print(df_resumen_desc.to_string(index=False))

In [ ]:
# Eliminar categorías específicas de df_auditoria['Descripción'] y generar resumen
categorias_excluir = [
    "Adres sin servicios AF",
    "SIE sin servicios Retirado",
    "Adres sin servicios RE",
    "SIE sin servicios Fallecido",
    "Adres sin servicios SD",
    "SIE sin servicios Suspendido",
    "PN sin servicios",
    "Adres sin servicios SM"
]

total_antes = len(df_auditoria)
eliminados = df_auditoria['Descripción'].isin(categorias_excluir).sum()

df_auditoria = df_auditoria[~df_auditoria['Descripción'].isin(categorias_excluir)].reset_index(drop=True)
total_despues = len(df_auditoria)

print("\n" + "="*80)
print("🧹 LIMPIEZA DE CATEGORÍAS EN df_auditoria['Descripción']")
print("="*80)
print(f"✓ Registros antes del proceso: {total_antes:,}")
print(f"✓ Registros eliminados:        {eliminados:,}")
print(f"✓ Registros después:           {total_despues:,}")

if total_despues > 0:
    conteo_restante = df_auditoria['Descripción'].value_counts(dropna=False)
    porcentaje_restante = (conteo_restante / total_despues * 100).round(2)

    df_resumen_restante = (
        pd.DataFrame({
            'Descripción': conteo_restante.index.astype(str),
            'Registros': conteo_restante.values,
            'Porcentaje': porcentaje_restante.values
        })
        .sort_values('Registros', ascending=False)
        .reset_index(drop=True)
    )

    print("\n📊 Distribución de categorías restantes:")
    print(df_resumen_restante.to_string(index=False))
else:
    print("\n⚠️ No quedaron registros en df_auditoria después del filtrado.")

# Organizar Columnas

In [ ]:
print("\n" + "="*80)
print("🔄 REORGANIZANDO ORDEN DE COLUMNAS")
print("="*80)

import time
tiempo_inicio = time.time()

# ============================================================================
# PASO 1: DEFINIR NUEVO ORDEN DE COLUMNAS (EXPLÍCITO)
# ============================================================================
nuevo_orden_columnas = [
    # Identificadores principales
    'AFL_ID',
    'ABREVIATURA',
    'NUMERO_IDENTIFICACION',
    
    # Datos ADRES
    'Regimen_Adres',
    'DPR_ADRES',
    'MNC_ADRES',
    'Estado_ADRES',
    
    # Datos SIE
    'municipio_SIE',
    'estado_SIE',
    'regimen_SIE',
    'estado_traslado',
    'PE',
    'MUNICIPIO_PE',
    'descripcion_altocosto',
    
    # Estado de certificación y portabilidad
    'Certificados',
    'Estado_Portabilidad',
    'municipio_receptor',
    
    # Ubicación del afiliado
    'MUNICIPIO',
    
    # SERVICIOS (Orden original)
    'MEDICINA GENERAL',
    'LABORATORIO CLINICO',
    'ODONTOLOGIA GENERAL',
    'MEDICAMENTOS',
    'OPTOMETRIA',
    'PROMOCION Y PREVENCION GENERAL',
    'OFTALMOLOGIA',
    'SERVICIO DE DEMANDA INDUCIDA A LOS PROGRAMAS DE DETECCION TEMPRANA Y PROTECCION ESPECIFICA',
    'PROCEDIMIENTOS MENORES',
    'RUTA MATERNO PERINATAL',
    'RUTA PROMOCION Y MANTENIMIENTO',
    'HOSPITALIZACION GENERAL',
    'SERVICIO DE URGENCIAS',
    'TRANSPORTE ASISTENCIAL BASICO',
    'RAYOS X',
    'FONOAUDIOLOGIA Y/O TERAPIA DEL LENGUAJE',
    'FISIOTERAPIA',
    'TERAPIA OCUPACIONAL',
    'TRANSPORTE ASISTENCIAL MEDICALIZADO',
    'TERAPIA RESPIRATORIA',
    'IMAGENES DIAGNOSTICAS - IONIZANTES',
    'ENFERMERIA',
    'ANESTESIA',
    'INFECTOLOGIA',
    'MEDICINA FAMILIAR',
    'CIRUGIA ORTOPEDICA',
    'PROTECCION ESPECIFICA - VACUNACION',
    'SERVICIO FARMACEUTICO',
    'NUTRICION Y DIETETICA',
    'TOMA DE MUESTRAS DE LABORATORIO CLINICO',
    'MEDICAMENTOS ESPECIALIZADOS',
]

# ============================================================================
# PASO 2: VALIDAR QUE TODAS LAS COLUMNAS EXISTAN
# ============================================================================
print("\n✓ Validando columnas...")
print("-" * 80)

columnas_actuales = set(df_pivot.columns)
columnas_esperadas = set(nuevo_orden_columnas)

# Columnas que faltan en el nuevo orden
columnas_faltantes = columnas_actuales - columnas_esperadas

if columnas_faltantes:
    print(f"\n⚠️  ADVERTENCIA: Columnas en df_pivot que NO están en nuevo_orden_columnas:")
    print(f"   Columnas: {sorted(columnas_faltantes)}")
    print(f"\n   Estas columnas se agregará al final del nuevo orden")
    nuevo_orden_columnas.extend(sorted(columnas_faltantes))

# Columnas que faltan en el DataFrame
columnas_inexistentes = columnas_esperadas - columnas_actuales

if columnas_inexistentes:
    print(f"\n⚠️  ADVERTENCIA: Columnas en nuevo_orden_columnas que NO existen en df_pivot:")
    print(f"   Columnas: {sorted(columnas_inexistentes)}")
    print(f"\n   Estas columnas se ELIMINARÁN del nuevo orden")
    nuevo_orden_columnas = [col for col in nuevo_orden_columnas if col not in columnas_inexistentes]

# ============================================================================
# PASO 3: REORDENAR DATAFRAME
# ============================================================================
print("\n✓ Aplicando nuevo orden de columnas...")

df_pivot = df_pivot[nuevo_orden_columnas]

print(f"\n✓ DataFrame reorganizado exitosamente")
print(f"  - Total de columnas: {len(df_pivot.columns)}")
print(f"  - Total de filas: {len(df_pivot):,}")

# ============================================================================
# PASO 4: VALIDACIÓN DEL NUEVO ORDEN
# ============================================================================
print("\n" + "="*80)
print("✅ VALIDACIÓN DEL NUEVO ORDEN")
print("="*80)

print(f"\n✓ SECCIONES DEL NUEVO ORDEN:")
print("-" * 80)

# Identificadores
print(f"\n1️⃣  IDENTIFICADORES PRINCIPALES:")
for i, col in enumerate(['AFL_ID', 'ABREVIATURA', 'NUMERO_IDENTIFICACION'], 1):
    if col in df_pivot.columns:
        print(f"    {i}. {col}")

# ADRES
print(f"\n2️⃣  DATOS ADRES:")
for i, col in enumerate(['Regimen_Adres', 'DPR_ADRES', 'MNC_ADRES', 'Estado_ADRES'], 1):
    if col in df_pivot.columns:
        print(f"    {i}. {col}")

# SIE
print(f"\n3️⃣  DATOS SIE:")
for i, col in enumerate(['municipio_SIE', 'estado_SIE', 'regimen_SIE', 'estado_traslado', 'PE'], 1):
    if col in df_pivot.columns:
        print(f"    {i}. {col}")

# Estados y Portabilidad
print(f"\n4️⃣  CERTIFICACIÓN Y PORTABILIDAD:")
for i, col in enumerate(['Certificados', 'Estado_Portabilidad', 'municipio_receptor'], 1):
    if col in df_pivot.columns:
        print(f"    {i}. {col}")

# Locación
print(f"\n5️⃣  UBICACIÓN:")
if 'MUNICIPIO' in df_pivot.columns:
    print(f"    1. MUNICIPIO")

# Servicios
print(f"\n6️⃣  SERVICIOS ASIGNADOS ({len([col for col in df_pivot.columns if col.isupper() and col not in ['AFL_ID', 'ABREVIATURA', 'NUMERO_IDENTIFICACION', 'MUNICIPIO', 'Regimen_Adres', 'DPR_ADRES', 'MNC_ADRES', 'Estado_ADRES', 'municipio_SIE', 'estado_SIE', 'regimen_SIE', 'estado_traslado', 'PE', 'Certificados', 'Estado_Portabilidad', 'municipio_receptor']])} servicios):")
servicios = [col for col in df_pivot.columns if col.isupper() and col not in ['AFL_ID', 'ABREVIATURA', 'NUMERO_IDENTIFICACION', 'MUNICIPIO', 'Regimen_Adres', 'DPR_ADRES', 'MNC_ADRES', 'Estado_ADRES', 'municipio_SIE', 'estado_SIE', 'regimen_SIE', 'estado_traslado', 'PE', 'Certificados', 'Estado_Portabilidad', 'municipio_receptor']]
for i, servicio in enumerate(servicios, 1):
    print(f"    {i:2d}. {servicio}")

# ============================================================================
# PASO 5: MUESTRA DE PRIMERAS FILAS
# ============================================================================
print("\n" + "="*80)
print("📋 MUESTRA DE PRIMERAS FILAS CON NUEVO ORDEN")
print("="*80)

# Mostrar solo primeras 5 columnas + algunos servicios
columnas_muestra = list(df_pivot.columns[:15]) + ['MEDICINA GENERAL', 'SERVICIOS', 'MEDICAMENTOS'][:3]
columnas_muestra_existentes = [col for col in columnas_muestra if col in df_pivot.columns]

print(f"\nPrimeras 5 registros (mostrando primeras 15 columnas):")
print("-" * 80)
print(df_pivot[columnas_muestra_existentes].head().to_string())

# ============================================================================
# PASO 6: INFORMACIÓN DE TIPOS DE DATOS
# ============================================================================
print("\n" + "="*80)
print("🔍 INFORMACIÓN DE TIPOS DE DATOS (PRIMERAS 20 COLUMNAS)")
print("="*80)

print("\nTipos de datos:")
print("-" * 80)
for col in df_pivot.columns[:20]:
    dtype = df_pivot[col].dtype
    nulos = df_pivot[col].isna().sum()
    porcentaje_nulos = (nulos / len(df_pivot)) * 100
    print(f"  {col:60s} | {str(dtype):15s} | Nulos: {nulos:10,d} ({porcentaje_nulos:5.1f}%)")

# ============================================================================
# PASO 7: RESUMEN FINAL
# ============================================================================
print("\n" + "="*80)
print("📊 RESUMEN DE REORGANIZACIÓN")
print("="*80)

# Contar secciones
identificadores = 3
adres = 4
sie = 5
portabilidad = 3
locacion = 1
servicios_count = len(servicios)

print(f"\n✓ DESGLOSE POR SECCIÓN:")
print(f"  - Identificadores: {identificadores} columnas")
print(f"  - Datos ADRES: {adres} columnas")
print(f"  - Datos SIE: {sie} columnas")
print(f"  - Certificación/Portabilidad: {portabilidad} columnas")
print(f"  - Ubicación: {locacion} columna")
print(f"  - Servicios: {servicios_count} columnas")
print(f"  {'─' * 40}")
print(f"  - TOTAL: {len(df_pivot.columns)} columnas")

print(f"\n✓ ESTRUCTURA FINAL:")
print(f"  - Registros: {len(df_pivot):,}")
print(f"  - Columnas: {len(df_pivot.columns)}")
print(f"  - Tamaño en memoria: {df_pivot.memory_usage(deep=True).sum() / (1024**2):.2f} MB")

print(f"\n✓ PRIMERAS COLUMNAS DEL NUEVO ORDEN:")
for i, col in enumerate(df_pivot.columns[:10], 1):
    print(f"  {i:2d}. {col}")

tiempo_final = time.time()
tiempo_transcurrido = tiempo_final - tiempo_inicio

print("\n" + "="*80)
print(f"⏱️  TIEMPO TRANSCURRIDO: {tiempo_transcurrido:.4f} segundos")
print("="*80)

# ============================================================================
# PASO 8: MOSTRAR ORDEN COMPLETO (OPCIONAL)
# ============================================================================
print("\n✓ ORDEN COMPLETO DE COLUMNAS:")
print("-" * 80)
for i, col in enumerate(df_pivot.columns, 1):
    print(f"  {i:2d}. {col}")

print("\n" + "="*80)
print("✅ REORGANIZACIÓN COMPLETADA EXITOSAMENTE")
print("="*80)

## Municipio ADRES

In [ ]:
import pandas as pd
import time

print("\n" + "="*80)
print("🔗 UNIFICANDO COLUMNAS DPR_ADRES Y MNC_ADRES")
print("="*80)

tiempo_inicio = time.time()

# ============================================================================
# PASO 1: VALIDAR QUE LAS COLUMNAS EXISTAN
# ============================================================================
print("\n✓ Validando columnas existentes...")
print("-" * 80)

columnas_requeridas = ['DPR_ADRES', 'MNC_ADRES']
columnas_presentes = [col for col in columnas_requeridas if col in df_pivot.columns]
columnas_faltantes = [col for col in columnas_requeridas if col not in df_pivot.columns]

print(f"✓ Columnas a unificar: {columnas_presentes}")

if columnas_faltantes:
    print(f"\n❌ ERROR: Las siguientes columnas NO existen en df_pivot:")
    for col in columnas_faltantes:
        print(f"   - {col}")
    print(f"\n⚠️  Operación cancelada. Verifica los nombres de las columnas.")
    
else:
    # ========================================================================
    # PASO 2: ANALIZAR DATOS ACTUALES
    # ========================================================================
    print(f"\n✓ Analizando datos actuales...")
    print("-" * 80)
    
    print(f"\n📊 COLUMNA 'DPR_ADRES' (Código Departamento - 2 dígitos):")
    print(f"   - Tipo de dato: {df_pivot['DPR_ADRES'].dtype}")
    print(f"   - Total de registros: {len(df_pivot):,}")
    print(f"   - Valores no nulos: {df_pivot['DPR_ADRES'].notna().sum():,}")
    print(f"   - Valores nulos/vacíos: {df_pivot['DPR_ADRES'].isna().sum():,}")
    print(f"   - Valores únicos: {df_pivot['DPR_ADRES'].nunique()}")
    print(f"   - Ejemplos:")
    ejemplos_dpr = df_pivot['DPR_ADRES'].dropna().unique()[:5]
    for ej in ejemplos_dpr:
        print(f"     • {ej} (tipo: {type(ej).__name__}, longitud: {len(str(ej))})")
    
    print(f"\n📊 COLUMNA 'MNC_ADRES' (Código Municipio - 3 dígitos):")
    print(f"   - Tipo de dato: {df_pivot['MNC_ADRES'].dtype}")
    print(f"   - Total de registros: {len(df_pivot):,}")
    print(f"   - Valores no nulos: {df_pivot['MNC_ADRES'].notna().sum():,}")
    print(f"   - Valores nulos/vacíos: {df_pivot['MNC_ADRES'].isna().sum():,}")
    print(f"   - Valores únicos: {df_pivot['MNC_ADRES'].nunique()}")
    print(f"   - Ejemplos:")
    ejemplos_mnc = df_pivot['MNC_ADRES'].dropna().unique()[:5]
    for ej in ejemplos_mnc:
        print(f"     • {ej} (tipo: {type(ej).__name__}, longitud: {len(str(ej))})")
    
    # ========================================================================
    # PASO 3: CREAR FUNCIÓN DE UNIFICACIÓN
    # ========================================================================
    print(f"\n✓ Creando función de unificación...")
    
    def unificar_id_municipal(fila):
        """
        Unifica DPR_ADRES (2 dígitos) + MNC_ADRES (3 dígitos)
        Resultado: XXXXX (5 dígitos) ó vacío si ambos son nulos/vacíos
        
        Ejemplos:
        - 85 + 315 = 85315
        - 09 + 001 = 09001
        - NaN + NaN = NaN (vacío)
        - 85 + NaN = NaN (ambos deben existir)
        """
        dpr = fila['DPR_ADRES']
        mnc = fila['MNC_ADRES']
        
        # Si ambos son nulos o vacíos
        if pd.isna(dpr) or pd.isna(mnc):
            return None
        
        # Convertir a string y limpiar espacios
        dpr_str = str(dpr).strip()
        mnc_str = str(mnc).strip()
        
        # Si después de limpiar quedan vacíos
        if not dpr_str or not mnc_str:
            return None
        
        # Rellenar con ceros si es necesario
        dpr_str = dpr_str.zfill(2)  # Asegurar 2 dígitos
        mnc_str = mnc_str.zfill(3)  # Asegurar 3 dígitos
        
        # Concatenar
        return dpr_str + mnc_str
    
    # ========================================================================
    # PASO 4: CREAR NUEVA COLUMNA
    # ========================================================================
    print(f"\n✓ Creando nueva columna 'ID_MNC_ADRES'...")
    
    # Encontrar índice de MNC_ADRES para insertar la nueva columna en su lugar
    indice_mnc = df_pivot.columns.get_loc('MNC_ADRES')
    
    # Crear la nueva columna
    df_pivot['ID_MNC_ADRES'] = df_pivot.apply(unificar_id_municipal, axis=1)
    
    print(f"✓ Nueva columna creada exitosamente")
    
    # ========================================================================
    # PASO 5: VALIDAR RESULTADOS ANTES DE ELIMINAR
    # ========================================================================
    print(f"\n✓ Validando resultados de unificación...")
    print("-" * 80)
    
    # Estadísticas de la nueva columna
    print(f"\n📊 COLUMNA 'ID_MNC_ADRES' (Nueva):")
    print(f"   - Tipo de dato: {df_pivot['ID_MNC_ADRES'].dtype}")
    print(f"   - Valores no nulos: {df_pivot['ID_MNC_ADRES'].notna().sum():,}")
    print(f"   - Valores nulos/vacíos: {df_pivot['ID_MNC_ADRES'].isna().sum():,}")
    print(f"   - Valores únicos: {df_pivot['ID_MNC_ADRES'].nunique()}")
    print(f"   - Longitud mínima: {df_pivot['ID_MNC_ADRES'].str.len().min() if df_pivot['ID_MNC_ADRES'].notna().any() else 'N/A'}")
    print(f"   - Longitud máxima: {df_pivot['ID_MNC_ADRES'].str.len().max() if df_pivot['ID_MNC_ADRES'].notna().any() else 'N/A'}")
    
    # Ejemplos de unificación
    print(f"\n📋 EJEMPLOS DE UNIFICACIÓN:")
    print(f"   DPR_ADRES → MNC_ADRES → ID_MNC_ADRES")
    print(f"   ─" * 30)
    
    # Seleccionar ejemplos variados
    df_ejemplos = df_pivot[['DPR_ADRES', 'MNC_ADRES', 'ID_MNC_ADRES']].drop_duplicates()
    df_ejemplos_muestra = pd.concat([
        df_ejemplos[df_ejemplos['ID_MNC_ADRES'].notna()].head(5),
        df_ejemplos[df_ejemplos['ID_MNC_ADRES'].isna()].head(3)
    ]).drop_duplicates()
    
    for idx, row in df_ejemplos_muestra.iterrows():
        dpr_val = str(row['DPR_ADRES']) if pd.notna(row['DPR_ADRES']) else 'NaN'
        mnc_val = str(row['MNC_ADRES']) if pd.notna(row['MNC_ADRES']) else 'NaN'
        id_val = str(row['ID_MNC_ADRES']) if pd.notna(row['ID_MNC_ADRES']) else 'NaN'
        
        print(f"   {dpr_val:6s} → {mnc_val:6s} → {id_val}")
    
    # Validar que no hay valores duplicados incorrectamente
    print(f"\n✓ Validaciones adicionales:")
    
    # Contar coincidencias correctas
    df_valida = df_pivot[df_pivot['ID_MNC_ADRES'].notna()].copy()
    
    if len(df_valida) > 0:
        # Verificar que los primeros 2 caracteres coincidan con DPR_ADRES
        dpr_check = df_valida['ID_MNC_ADRES'].str[:2] == df_valida['DPR_ADRES'].astype(str).str.zfill(2)
        print(f"   • Coincidencia DPR_ADRES con ID_MNC_ADRES: {dpr_check.sum():,} / {len(dpr_check):,} registros")
        
        # Verificar que los últimos 3 caracteres coincidan con MNC_ADRES
        mnc_check = df_valida['ID_MNC_ADRES'].str[-3:] == df_valida['MNC_ADRES'].astype(str).str.zfill(3)
        print(f"   • Coincidencia MNC_ADRES con ID_MNC_ADRES: {mnc_check.sum():,} / {len(mnc_check):,} registros")
    
    # ========================================================================
    # PASO 6: REORDENAR Y ELIMINAR COLUMNAS
    # ========================================================================
    print(f"\n✓ Reorganizando dataframe...")
    print("-" * 80)
    
    # Obtener la lista de columnas actual
    columnas_actuales = list(df_pivot.columns)
    
    # Obtener índices de las columnas a manipular
    idx_dpr = columnas_actuales.index('DPR_ADRES')
    idx_mnc = columnas_actuales.index('MNC_ADRES')
    idx_id_mnc = columnas_actuales.index('ID_MNC_ADRES')
    
    print(f"\n   Posiciones actuales:")
    print(f"   • DPR_ADRES en posición: {idx_dpr}")
    print(f"   • MNC_ADRES en posición: {idx_mnc}")
    print(f"   • ID_MNC_ADRES en posición: {idx_id_mnc}")
    
    # Crear nueva lista de columnas sin DPR_ADRES, MNC_ADRES e ID_MNC_ADRES
    nuevas_columnas = [col for col in columnas_actuales 
                       if col not in ['DPR_ADRES', 'MNC_ADRES', 'ID_MNC_ADRES']]
    
    # Insertar ID_MNC_ADRES en la posición de MNC_ADRES (que es la menor de las dos)
    posicion_insercion = min(idx_dpr, idx_mnc)
    nuevas_columnas.insert(posicion_insercion, 'ID_MNC_ADRES')
    
    # Aplicar nuevo orden
    df_pivot = df_pivot[nuevas_columnas]
    
    print(f"\n   Nuevo orden aplicado:")
    print(f"   • ID_MNC_ADRES en posición: {nuevas_columnas.index('ID_MNC_ADRES')}")
    print(f"   • DPR_ADRES: ELIMINADA ✓")
    print(f"   • MNC_ADRES: ELIMINADA ✓")
    
    # ========================================================================
    # PASO 7: RESUMEN FINAL
    # ========================================================================
    print(f"\n" + "="*80)
    print("✅ UNIFICACIÓN COMPLETADA EXITOSAMENTE")
    print("="*80)
    
    print(f"\n📊 RESUMEN DE CAMBIOS:")
    print(f"-" * 80)
    print(f"✓ Columnas eliminadas: 2")
    print(f"  • DPR_ADRES (Código Departamento)")
    print(f"  • MNC_ADRES (Código Municipio)")
    
    print(f"\n✓ Columnas añadidas: 1")
    print(f"  • ID_MNC_ADRES (Código Departamento + Municipio unificado)")
    
    print(f"\n✓ Estadísticas de la nueva columna:")
    print(f"  • Registros totales: {len(df_pivot):,}")
    print(f"  • Valores rellenados (no nulos): {df_pivot['ID_MNC_ADRES'].notna().sum():,}")
    print(f"  • Valores vacíos (nulos): {df_pivot['ID_MNC_ADRES'].isna().sum():,}")
    print(f"  • Porcentaje rellenado: {(df_pivot['ID_MNC_ADRES'].notna().sum() / len(df_pivot) * 100):.2f}%")
    print(f"  • Valores únicos: {df_pivot['ID_MNC_ADRES'].nunique()}")
    
    print(f"\n✓ Cambio en estructura:")
    print(f"  • Columnas anteriores: {len(columnas_actuales)}")
    print(f"  • Columnas actuales: {len(df_pivot.columns)}")
    print(f"  • Diferencia: {len(columnas_actuales) - len(df_pivot.columns)} columna eliminada ✓")
    
    print(f"\n✓ Columnas cercanas a ID_MNC_ADRES:")
    idx_id_mnc_nueva = df_pivot.columns.get_loc('ID_MNC_ADRES')
    print(f"  • Posición: {idx_id_mnc_nueva}")
    
    start_idx = max(0, idx_id_mnc_nueva - 2)
    end_idx = min(len(df_pivot.columns), idx_id_mnc_nueva + 3)
    
    for i, col in enumerate(df_pivot.columns[start_idx:end_idx], start_idx + 1):
        marcador = " ← NEW" if col == 'ID_MNC_ADRES' else ""
        print(f"  • [{i:2d}] {col}{marcador}")
    
    # ========================================================================
    # PASO 8: MUESTRA DE DATOS
    # ========================================================================
    print(f"\n" + "="*80)
    print("📋 MUESTRA DE DATOS REORGANIZADOS")
    print("="*80)
    
    # Seleccionar columnas para mostrar
    columnas_muestra = [
        'AFL_ID',
        'NUMERO_IDENTIFICACION',
        'Regimen_Adres',
        'ID_MNC_ADRES',
        'Estado_ADRES',
        'municipio_SIE'
    ]
    
    columnas_muestra_existentes = [col for col in columnas_muestra if col in df_pivot.columns]
    
    print(f"\nPrimeros 10 registros:")
    print("-" * 80)
    print(df_pivot[columnas_muestra_existentes].head(10).to_string())
    
    # ========================================================================
    # PASO 9: INFORMACIÓN Final
    # ========================================================================
    tiempo_final = time.time()
    tiempo_transcurrido = tiempo_final - tiempo_inicio
    
    print(f"\n" + "="*80)
    print(f"⏱️  TIEMPO TRANSCURRIDO: {tiempo_transcurrido:.4f} segundos")
    print("="*80)
    
    print(f"\n✅ El dataframe está listo para la siguiente operación")
    print(f"\n💡 PRÓXIMOS PASOS RECOMENDADOS:")
    print(f"  1. Guardar el dataframe modificado")
    print(f"  2. Verificar integridad de datos si es necesario")
    print(f"  3. Exportar a Excel con el nuevo orden")
    
    print("\n" + "="*80)

## Estado SIE

In [ ]:
print("\n" + "="*80)
print("📊 DISTRIBUCIÓN DE CATEGORÍAS EN COLUMNA 'estado_SIE'")
print("="*80)

conteo_estado_sie = df_pivot['estado_SIE'].value_counts(dropna=False)

for categoria, cantidad in conteo_estado_sie.items():
    porcentaje = (cantidad / len(df_pivot)) * 100
    label = categoria if pd.notna(categoria) else "Sin dato (NaN)"
    print(f"  {label:30s}: {cantidad:10,d} ({porcentaje:6.2f}%)")

print("\n" + "="*80)
print(f"✓ Total de registros: {len(df_pivot):,}")
print(f"✓ Categorías únicas: {df_pivot['estado_SIE'].nunique()}")
print("="*80)

In [ ]:
import pandas as pd
import time

print("\n" + "="*80)
print("🔄 NORMALIZANDO COLUMNA 'estado_SIE'")
print("="*80)

tiempo_inicio = time.time()

# ============================================================================
# PASO 1: DEFINIR DICCIONARIO DE MAPEO
# ============================================================================
diccionario_estado_sie = {
    'Activo': 'AC',
    'Retirado': 'RE',
    'Fallecido': 'AF',
    'Suspendido': 'SM'
}

print("\n✓ Diccionario de mapeo definido:")
print("-" * 80)
for clave, valor in diccionario_estado_sie.items():
    print(f"  {clave:20s} → {valor}")

# ============================================================================
# PASO 2: ANALIZAR DATOS ACTUALES
# ============================================================================
print("\n✓ Analizando datos actuales de 'estado_SIE'...")
print("-" * 80)

print(f"\n📊 ESTADÍSTICAS ANTES DEL MAPEO:")
print(f"  - Total de registros: {len(df_pivot):,}")
print(f"  - Valores no nulos: {df_pivot['estado_SIE'].notna().sum():,}")
print(f"  - Valores nulos/vacíos: {df_pivot['estado_SIE'].isna().sum():,}")
print(f"  - Valores únicos: {df_pivot['estado_SIE'].nunique()}")

# Mostrar distribución actual
print(f"\n📋 DISTRIBUCIÓN ACTUAL DE 'estado_SIE':")
print("-" * 80)

conteo_actual = df_pivot['estado_SIE'].value_counts(dropna=False)

for categoria, cantidad in conteo_actual.items():
    porcentaje = (cantidad / len(df_pivot)) * 100
    label = categoria if pd.notna(categoria) else "NaN (vacío)"
    print(f"  {label:30s}: {cantidad:10,d} ({porcentaje:6.2f}%)")

# ============================================================================
# PASO 3: VALIDAR QUE TODAS LAS CATEGORÍAS EXISTAN EN EL DICCIONARIO
# ============================================================================
print("\n✓ Validando mapeo...")
print("-" * 80)

# Obtener categorías únicas (sin NaN)
categorias_actuales = set(df_pivot['estado_SIE'].dropna().unique())
categorias_diccionario = set(diccionario_estado_sie.keys())

# Categorías que no están en el diccionario
categorias_sin_mapeo = categorias_actuales - categorias_diccionario

if categorias_sin_mapeo:
    print(f"\n⚠️  ADVERTENCIA: Algunas categorías NO están en el diccionario:")
    for categoria in sorted(categorias_sin_mapeo):
        cantidad = (df_pivot['estado_SIE'] == categoria).sum()
        porcentaje = (cantidad / len(df_pivot)) * 100
        print(f"  - '{categoria}': {cantidad:,} registros ({porcentaje:.2f}%)")
    print(f"\n  Estas categorías se MANTENDRÁN sin cambios (si no están en diccionario)")
else:
    print(f"\n✅ Todas las categorías están en el diccionario")

# Categorías en el diccionario que no existen en los datos
categorias_no_usadas = categorias_diccionario - categorias_actuales

if categorias_no_usadas:
    print(f"\n💡 INFORMACIÓN: Algunas categorías del diccionario NO aparecen en los datos:")
    for categoria in sorted(categorias_no_usadas):
        print(f"  - '{categoria}' → '{diccionario_estado_sie[categoria]}'")

# ============================================================================
# PASO 4: CREAR FUNCIÓN DE MAPEO (MANTIENE NULOS Y SIN MAPEO)
# ============================================================================
print("\n✓ Creando función de mapeo...")

def mapear_estado_sie(valor):
    """
    Mapea valores de estado_SIE según el diccionario.
    - Si el valor está en el diccionario, retorna el valor mapeado
    - Si es NaN/None, retorna NaN/None (mantiene vacíos)
    - Si no está en el diccionario, retorna el valor original
    
    Ejemplos:
    - 'Activo' → 'AC'
    - 'Fallecido' → 'AF'
    - NaN → NaN (vacío)
    - 'OtroValor' → 'OtroValor' (sin cambios si no está en diccionario)
    """
    # Si es nulo, retornar nulo
    if pd.isna(valor):
        return None
    
    # Si está en el diccionario, retornar el valor mapeado
    if valor in diccionario_estado_sie:
        return diccionario_estado_sie[valor]
    
    # Si no está en el diccionario, retornar el valor original
    return valor

# ============================================================================
# PASO 5: APLICAR MAPEO (VECTORIZADO CUANDO ES POSIBLE)
# ============================================================================
print("\n✓ Aplicando mapeo a la columna...")

# MÉTODO VECTORIZADO: map() + fillna() es más eficiente que apply()
# Para Series, .map() es mejor que apply() porque puede usar diccionarios directamente

# Crear máscara de valores no nulos
mascara_no_nulos = df_pivot['estado_SIE'].notna()

# Aplicar mapeo SOLO a valores no nulos
df_pivot['estado_SIE'] = df_pivot['estado_SIE'].map(diccionario_estado_sie)

print(f"✓ Mapeo aplicado exitosamente")

# ============================================================================
# PASO 6: VALIDAR RESULTADOS
# ============================================================================
print("\n✓ Validando resultados...")
print("-" * 80)

print(f"\n📊 ESTADÍSTICAS DESPUÉS DEL MAPEO:")
print(f"  - Total de registros: {len(df_pivot):,}")
print(f"  - Valores no nulos: {df_pivot['estado_SIE'].notna().sum():,}")
print(f"  - Valores nulos/vacíos: {df_pivot['estado_SIE'].isna().sum():,}")
print(f"  - Valores únicos: {df_pivot['estado_SIE'].nunique()}")

# ============================================================================
# PASO 7: IMPRIMIR CATEGORÍAS NORMALIZADAS
# ============================================================================
print("\n" + "="*80)
print("📊 DISTRIBUCIÓN DE CATEGORÍAS NORMALIZADAS EN 'estado_SIE'")
print("="*80)

# Contar categorías
conteo_normalizado = df_pivot['estado_SIE'].value_counts(dropna=False).sort_values(ascending=False)

print(f"\n✓ CATEGORÍAS ENCONTRADAS ({len(conteo_normalizado)}):")
print("-" * 80)

for categoria, cantidad in conteo_normalizado.items():
    porcentaje = (cantidad / len(df_pivot)) * 100
    label = categoria if pd.notna(categoria) else "NaN (vacío/sin dato)"
    
    # Buscar la categoría original en el diccionario
    categoria_original = None
    for original, nueva in diccionario_estado_sie.items():
        if nueva == categoria:
            categoria_original = original
            break
    
    if categoria_original:
        etiqueta_completa = f"{label} ({categoria_original})"
    else:
        etiqueta_completa = label
    
    print(f"  {etiqueta_completa:50s}: {cantidad:10,d} ({porcentaje:6.2f}%)")

# Crear tabla resumen
print(f"\n" + "="*80)
print("📋 TABLA RESUMEN DE NORMALIZACIÓN")
print("="*80)

df_resumen_normalizacion = pd.DataFrame({
    'Categoría_Normalizada': conteo_normalizado.index,
    'Cantidad': conteo_normalizado.values,
    'Porcentaje': (conteo_normalizado.values / len(df_pivot) * 100).round(2)
})

# Agregar categoría original
def obtener_categoria_original(codigo):
    if pd.isna(codigo):
        return "Sin dato"
    for original, nueva in diccionario_estado_sie.items():
        if nueva == codigo:
            return original
    return "N/A"

df_resumen_normalizacion['Categoría_Original'] = df_resumen_normalizacion['Categoría_Normalizada'].apply(obtener_categoria_original)

# Reordenar columnas
df_resumen_normalizacion = df_resumen_normalizacion[['Categoría_Original', 'Categoría_Normalizada', 'Cantidad', 'Porcentaje']]

print("\n" + df_resumen_normalizacion.to_string(index=False))

# ============================================================================
# PASO 8: VALIDACIÓN DE INTEGRIDAD
# ============================================================================
print("\n" + "="*80)
print("🔍 VALIDACIÓN DE INTEGRIDAD")
print("="*80)

# Verificar que el total de registros se mantiene
print(f"\n✓ Integridad de registros:")
print(f"  - Total registros: {len(df_pivot):,}")
print(f"  - Suma de distribución: {conteo_normalizado.sum():,}")
print(f"  - ¿Coinciden?: {len(df_pivot) == conteo_normalizado.sum()}")

# Verificar códigos válidos
codigos_validos = set(diccionario_estado_sie.values())
codigos_en_datos = set(df_pivot['estado_SIE'].dropna().unique())
codigos_inesperados = codigos_en_datos - codigos_validos

if codigos_inesperados:
    print(f"\n⚠️  ADVERTENCIA: Códigos encontrados que NO están en el diccionario:")
    for codigo in sorted(codigos_inesperados):
        print(f"  - {codigo}")
else:
    print(f"\n✅ Todos los códigos son válidos")

# ============================================================================
# PASO 9: MUESTRA DE DATOS
# ============================================================================
print("\n" + "="*80)
print("📋 MUESTRA DE DATOS NORMALIZADOS")
print("="*80)

# Seleccionar columnas para mostrar
columnas_muestra = [
    'AFL_ID',
    'NUMERO_IDENTIFICACION',
    'Estado_ADRES',
    'estado_SIE',
    'regimen_SIE'
]

columnas_muestra_existentes = [col for col in columnas_muestra if col in df_pivot.columns]

print(f"\nPrimeros 15 registros:")
print("-" * 80)
print(df_pivot[columnas_muestra_existentes].head(15).to_string(index=False))

# ============================================================================
# PASO 10: RESUMEN FINAL
# ============================================================================
tiempo_final = time.time()
tiempo_transcurrido = tiempo_final - tiempo_inicio

print("\n" + "="*80)
print("✅ NORMALIZACIÓN COMPLETADA EXITOSAMENTE")
print("="*80)

print(f"\n📊 RESUMEN:")
print(f"  - Total de registros: {len(df_pivot):,}")
print(f"  - Columna normalizada: 'estado_SIE'")
print(f"  - Categorías después: {df_pivot['estado_SIE'].nunique()}")
print(f"  - Valores nulos preservados: {df_pivot['estado_SIE'].isna().sum():,}")

print(f"\n🔄 MAPEO APLICADO:")
for original, nuevo in diccionario_estado_sie.items():
    cantidad = (df_resumen_normalizacion['Categoría_Original'] == original).sum()
    if cantidad > 0:
        cantidad_real = df_resumen_normalizacion[df_resumen_normalizacion['Categoría_Original'] == original]['Cantidad'].values[0]
        print(f"  '{original}' → '{nuevo}': {cantidad_real:,} registros")

print(f"\n⏱️  TIEMPO TRANSCURRIDO: {tiempo_transcurrido:.4f} segundos")
print("="*80)

## Regimen SIE

In [ ]:
import pandas as pd
import time

print("\n" + "="*80)
print("🔄 NORMALIZANDO COLUMNA 'regimen_SIE'")
print("="*80)

tiempo_inicio = time.time()

# ============================================================================
# PASO 1: DEFINIR DICCIONARIO DE MAPEO
# ============================================================================
diccionario_regimen_sie = {
    'Subsidiado': 'EPS025',
    'Contributivo': 'EPSC25'
}

print("\n✓ Diccionario de mapeo definido:")
print("-" * 80)
for clave, valor in diccionario_regimen_sie.items():
    print(f"  {clave:20s} → {valor}")

# ============================================================================
# PASO 2: ANALIZAR DATOS ACTUALES
# ============================================================================
print("\n✓ Analizando datos actuales de 'regimen_SIE'...")
print("-" * 80)

print(f"\n📊 ESTADÍSTICAS ANTES DEL MAPEO:")
print(f"  - Total de registros: {len(df_pivot):,}")
print(f"  - Valores no nulos: {df_pivot['regimen_SIE'].notna().sum():,}")
print(f"  - Valores nulos/vacíos: {df_pivot['regimen_SIE'].isna().sum():,}")
print(f"  - Valores únicos: {df_pivot['regimen_SIE'].nunique()}")

# Mostrar distribución actual
print(f"\n📋 DISTRIBUCIÓN ACTUAL DE 'regimen_SIE':")
print("-" * 80)

conteo_actual = df_pivot['regimen_SIE'].value_counts(dropna=False)

for categoria, cantidad in conteo_actual.items():
    porcentaje = (cantidad / len(df_pivot)) * 100
    label = categoria if pd.notna(categoria) else "NaN (vacío)"
    print(f"  {label:30s}: {cantidad:10,d} ({porcentaje:6.2f}%)")

# ============================================================================
# PASO 3: VALIDAR QUE TODAS LAS CATEGORÍAS EXISTAN EN EL DICCIONARIO
# ============================================================================
print("\n✓ Validando mapeo...")
print("-" * 80)

# Obtener categorías únicas (sin NaN)
categorias_actuales = set(df_pivot['regimen_SIE'].dropna().unique())
categorias_diccionario = set(diccionario_regimen_sie.keys())

# Categorías que no están en el diccionario
categorias_sin_mapeo = categorias_actuales - categorias_diccionario

if categorias_sin_mapeo:
    print(f"\n⚠️  ADVERTENCIA: Algunas categorías NO están en el diccionario:")
    for categoria in sorted(categorias_sin_mapeo):
        cantidad = (df_pivot['regimen_SIE'] == categoria).sum()
        porcentaje = (cantidad / len(df_pivot)) * 100
        print(f"  - '{categoria}': {cantidad:,} registros ({porcentaje:.2f}%)")
    print(f"\n  Estas categorías se MANTENDRÁN sin cambios (si no están en diccionario)")
else:
    print(f"\n✅ Todas las categorías están en el diccionario")

# Categorías en el diccionario que no existen en los datos
categorias_no_usadas = categorias_diccionario - categorias_actuales

if categorias_no_usadas:
    print(f"\n💡 INFORMACIÓN: Algunas categorías del diccionario NO aparecen en los datos:")
    for categoria in sorted(categorias_no_usadas):
        print(f"  - '{categoria}' → '{diccionario_regimen_sie[categoria]}'")

# ============================================================================
# PASO 4: APLICAR MAPEO (VECTORIZADO)
# ============================================================================
print("\n✓ Aplicando mapeo a la columna...")

# MÉTODO VECTORIZADO: map() es más eficiente que apply()
df_pivot['regimen_SIE'] = df_pivot['regimen_SIE'].map(diccionario_regimen_sie)

print(f"✓ Mapeo aplicado exitosamente")

# ============================================================================
# PASO 5: VALIDAR RESULTADOS
# ============================================================================
print("\n✓ Validando resultados...")
print("-" * 80)

print(f"\n📊 ESTADÍSTICAS DESPUÉS DEL MAPEO:")
print(f"  - Total de registros: {len(df_pivot):,}")
print(f"  - Valores no nulos: {df_pivot['regimen_SIE'].notna().sum():,}")
print(f"  - Valores nulos/vacíos: {df_pivot['regimen_SIE'].isna().sum():,}")
print(f"  - Valores únicos: {df_pivot['regimen_SIE'].nunique()}")

# ============================================================================
# PASO 6: IMPRIMIR CATEGORÍAS NORMALIZADAS
# ============================================================================
print("\n" + "="*80)
print("📊 DISTRIBUCIÓN DE CATEGORÍAS NORMALIZADAS EN 'regimen_SIE'")
print("="*80)

# Contar categorías
conteo_normalizado = df_pivot['regimen_SIE'].value_counts(dropna=False).sort_values(ascending=False)

print(f"\n✓ CATEGORÍAS ENCONTRADAS ({len(conteo_normalizado)}):")
print("-" * 80)

for categoria, cantidad in conteo_normalizado.items():
    porcentaje = (cantidad / len(df_pivot)) * 100
    label = categoria if pd.notna(categoria) else "NaN (vacío/sin dato)"
    
    # Buscar la categoría original en el diccionario
    categoria_original = None
    for original, nueva in diccionario_regimen_sie.items():
        if nueva == categoria:
            categoria_original = original
            break
    
    if categoria_original:
        etiqueta_completa = f"{label} ({categoria_original})"
    else:
        etiqueta_completa = label
    
    print(f"  {etiqueta_completa:50s}: {cantidad:10,d} ({porcentaje:6.2f}%)")

# Crear tabla resumen
print(f"\n" + "="*80)
print("📋 TABLA RESUMEN DE NORMALIZACIÓN")
print("="*80)

df_resumen_normalizacion = pd.DataFrame({
    'Categoría_Normalizada': conteo_normalizado.index,
    'Cantidad': conteo_normalizado.values,
    'Porcentaje': (conteo_normalizado.values / len(df_pivot) * 100).round(2)
})

# Agregar categoría original
def obtener_categoria_original(codigo):
    if pd.isna(codigo):
        return "Sin dato"
    for original, nueva in diccionario_regimen_sie.items():
        if nueva == codigo:
            return original
    return "N/A"

df_resumen_normalizacion['Categoría_Original'] = df_resumen_normalizacion['Categoría_Normalizada'].apply(obtener_categoria_original)

# Reordenar columnas
df_resumen_normalizacion = df_resumen_normalizacion[['Categoría_Original', 'Categoría_Normalizada', 'Cantidad', 'Porcentaje']]

print("\n" + df_resumen_normalizacion.to_string(index=False))

# ============================================================================
# PASO 7: VALIDACIÓN DE INTEGRIDAD
# ============================================================================
print("\n" + "="*80)
print("🔍 VALIDACIÓN DE INTEGRIDAD")
print("="*80)

# Verificar que el total de registros se mantiene
print(f"\n✓ Integridad de registros:")
print(f"  - Total registros: {len(df_pivot):,}")
print(f"  - Suma de distribución: {conteo_normalizado.sum():,}")
print(f"  - ¿Coinciden?: {len(df_pivot) == conteo_normalizado.sum()}")

# Verificar códigos válidos
codigos_validos = set(diccionario_regimen_sie.values())
codigos_en_datos = set(df_pivot['regimen_SIE'].dropna().unique())
codigos_inesperados = codigos_en_datos - codigos_validos

if codigos_inesperados:
    print(f"\n⚠️  ADVERTENCIA: Códigos encontrados que NO están en el diccionario:")
    for codigo in sorted(codigos_inesperados):
        print(f"  - {codigo}")
else:
    print(f"\n✅ Todos los códigos son válidos")

# ============================================================================
# PASO 8: MUESTRA DE DATOS
# ============================================================================
print("\n" + "="*80)
print("📋 MUESTRA DE DATOS NORMALIZADOS")
print("="*80)

# Seleccionar columnas para mostrar
columnas_muestra = [
    'AFL_ID',
    'NUMERO_IDENTIFICACION',
    'Regimen_Adres',
    'regimen_SIE',
    'estado_SIE'
]

columnas_muestra_existentes = [col for col in columnas_muestra if col in df_pivot.columns]

print(f"\nPrimeros 15 registros:")
print("-" * 80)
print(df_pivot[columnas_muestra_existentes].head(15).to_string(index=False))

# ============================================================================
# PASO 9: COMPARACIÓN ENTRE COLUMNAS
# ============================================================================
print("\n" + "="*80)
print("📊 COMPARACIÓN: regimen_SIE vs Regimen_Adres")
print("="*80)

# Crear tabla de comparación
df_comparacion = pd.DataFrame({
    'regimen_SIE': df_pivot['regimen_SIE'].value_counts(dropna=False),
    'Regimen_Adres': df_pivot['Regimen_Adres'].value_counts(dropna=False)
}).fillna(0).astype(int)

print("\nDistribución comparativa:")
print("-" * 80)
print(df_comparacion.to_string())

# ============================================================================
# PASO 10: RESUMEN FINAL
# ============================================================================
tiempo_final = time.time()
tiempo_transcurrido = tiempo_final - tiempo_inicio

print("\n" + "="*80)
print("✅ NORMALIZACIÓN COMPLETADA EXITOSAMENTE")
print("="*80)

print(f"\n📊 RESUMEN:")
print(f"  - Total de registros: {len(df_pivot):,}")
print(f"  - Columna normalizada: 'regimen_SIE'")
print(f"  - Categorías después: {df_pivot['regimen_SIE'].nunique()}")
print(f"  - Valores nulos preservados: {df_pivot['regimen_SIE'].isna().sum():,}")

print(f"\n🔄 MAPEO APLICADO:")
for original, nuevo in diccionario_regimen_sie.items():
    cantidad = (df_resumen_normalizacion['Categoría_Original'] == original).sum()
    if cantidad > 0:
        cantidad_real = df_resumen_normalizacion[df_resumen_normalizacion['Categoría_Original'] == original]['Cantidad'].values[0]
        print(f"  '{original}' → '{nuevo}': {cantidad_real:,} registros")

print(f"\n💡 NOTAS:")
print(f"  - Solo hay 2 categorías (Subsidiado y Contributivo)")
print(f"  - Se normalizó a códigos de EPS: EPS025 (Subsidiado) y EPSC25 (Contributivo)")
print(f"  - Los valores nulos se mantienen como vacíos")

print(f"\n⏱️  TIEMPO TRANSCURRIDO: {tiempo_transcurrido:.4f} segundos")
print("="*80)

## Municipio SIE

In [ ]:
import pandas as pd
import time

print("\n" + "="*80)
print("🗺️  NORMALIZANDO COLUMNA 'municipio_SIE' A CÓDIGO DANE")
print("="*80)

tiempo_inicio = time.time()

# ============================================================================
# PASO 1: ANÁLISIS PREVIO DE DATOS
# ============================================================================
print("\n✓ Analizando datos antes de la normalización...")
print("-" * 80)

print(f"\n📊 ESTADÍSTICAS DE 'municipio_SIE' ANTES:")
print(f"  - Total de registros: {len(df_pivot):,}")
print(f"  - Valores no nulos: {df_pivot['municipio_SIE'].notna().sum():,}")
print(f"  - Valores nulos/vacíos: {df_pivot['municipio_SIE'].isna().sum():,}")
print(f"  - Valores únicos: {df_pivot['municipio_SIE'].nunique()}")

valores_vacios = (df_pivot['municipio_SIE'].isna()) | (df_pivot['municipio_SIE'].str.strip() == '')
print(f"  - Valores vacíos/NaN total: {valores_vacios.sum():,}")

print(f"\n📊 INFORMACIÓN DE 'df_municipios_sie':")
print(f"  - Total de municipios en catálogo: {len(df_municipios_sie):,}")
print(f"  - Columna 'municipio' (código): {df_municipios_sie['municipio'].nunique()} códigos únicos")
print(f"  - Columna 'descripcion' (nombre): {df_municipios_sie['descripcion'].nunique()} descripciones únicas")

# ============================================================================
# PASO 2: REVISAR DUPLICADOS EN EL CATÁLOGO DE MUNICIPIOS
# ============================================================================
print("\n✓ Verificando integridad del catálogo de municipios...")
print("-" * 80)

descripciones_duplicadas = df_municipios_sie[df_municipios_sie.duplicated(subset=['descripcion'], keep=False)]

if len(descripciones_duplicadas) > 0:
    print(f"\n⚠️  ADVERTENCIA: Se encontraron descripciones duplicadas:")
    print(f"  Cantidad de registros con descripciones duplicadas: {len(descripciones_duplicadas)}")
    print(f"\n  Primeras duplicadas encontradas:")
    print(descripciones_duplicadas.groupby('descripcion').size().head(10).to_string())
else:
    print(f"\n✅ No hay descripciones duplicadas en el catálogo")

# ============================================================================
# PASO 3: CREAR DICCIONARIO DE BÚSQUEDA Y GUARDAR NOMBRES ANTES DE ELIMINAR
# ============================================================================
print("\n✓ Creando diccionario de búsqueda y guardando información...")
print("-" * 80)

# ✅ GUARDAR DICCIONARIO DE NOMBRES ANTES DE ELIMINAR df_municipios_sie
diccionario_municipios = dict(zip(
    df_municipios_sie['descripcion'].str.strip().str.upper(),
    df_municipios_sie['municipio'].astype(str)
))

# ✅ GUARDAR DICCIONARIO INVERSO (código → nombre) PARA PASO 14
diccionario_nombres_municipios = dict(zip(
    df_municipios_sie['municipio'].astype(str),
    df_municipios_sie['descripcion'].str.strip()
))

print(f"  - Diccionario creado con {len(diccionario_municipios)} pares descripción-código")
print(f"  - Diccionario inverso creado con {len(diccionario_nombres_municipios)} pares código-nombre")

# ============================================================================
# PASO 4: PREPARAR COLUMNA PARA BÚSQUEDA
# ============================================================================
print("\n✓ Preparando datos para búsqueda...")
print("-" * 80)

municipio_sIE_original = df_pivot['municipio_SIE'].copy()
municipio_sIE_normalizado = df_pivot['municipio_SIE'].str.strip().str.upper()

print(f"  - Datos preparados y listos para búsqueda")

# ============================================================================
# PASO 5: REALIZAR BÚSQUEDA Y MAPEO
# ============================================================================
print("\n✓ Realizando búsqueda y mapeo...")
print("-" * 80)

df_pivot['municipio_SIE_codigo'] = municipio_sIE_normalizado.map(diccionario_municipios)

print(f"  - Búsqueda completada")

# ============================================================================
# PASO 6: IDENTIFICAR REGISTROS NO NORMALIZADOS
# ============================================================================
print("\n✓ Identificando registros no normalizados...")
print("-" * 80)

registros_vacios = valores_vacios.sum()

registros_sin_mapeo = (
    (municipio_sIE_original.notna()) & 
    (municipio_sIE_original.str.strip() != '') & 
    (df_pivot['municipio_SIE_codigo'].isna())
).sum()

registros_normalizados = (df_pivot['municipio_SIE_codigo'].notna()).sum() - registros_vacios

print(f"\n📊 RESULTADO DEL MAPEO:")
print(f"  - Registros normalizados: {registros_normalizados:,}")
print(f"  - Registros sin mapeo (error): {registros_sin_mapeo:,}")
print(f"  - Registros vacíos (sin valor): {registros_vacios:,}")
print(f"  - Total procesado: {registros_normalizados + registros_sin_mapeo + registros_vacios:,}")

# ============================================================================
# PASO 7: MOSTRAR MUNICIPIOS NO ENCONTRADOS
# ============================================================================
if registros_sin_mapeo > 0:
    print("\n" + "="*80)
    print("⚠️  MUNICIPIOS NO ENCONTRADOS EN CATÁLOGO DANE")
    print("="*80)
    
    municipios_no_encontrados = df_pivot[
        (municipio_sIE_original.notna()) & 
        (municipio_sIE_original.str.strip() != '') & 
        (df_pivot['municipio_SIE_codigo'].isna())
    ]['municipio_SIE'].unique()
    
    print(f"\n✗ {len(municipios_no_encontrados)} municipios únicos no encontrados:")
    print("-" * 80)
    
    for idx, municipio in enumerate(sorted(municipios_no_encontrados), 1):
        cantidad = (municipio_sIE_original == municipio).sum()
        print(f"  {idx:3d}. '{municipio}' → {cantidad:,} registros")

# ============================================================================
# PASO 8: COPIAR REGISTROS NO NORMALIZADOS A AUDITORÍA
# ============================================================================
print("\n✓ Copiando registros no normalizados a auditoría...")
print("-" * 80)

registros_auditoria = df_pivot[
    (municipio_sIE_original.notna()) & 
    (municipio_sIE_original.str.strip() != '') & 
    (df_pivot['municipio_SIE_codigo'].isna())
].copy()

if registros_sin_mapeo > 0:
    registros_auditoria['Descripción'] = 'No se identifica codigo DANE SIE'
    registros_auditoria['municipio_SIE_original'] = registros_auditoria['municipio_SIE']
    
    df_auditoria = pd.concat([df_auditoria, registros_auditoria], ignore_index=True)
    
    print(f"  - {registros_sin_mapeo:,} registros copiados a auditoría")
    print(f"  - Descripción asignada: 'No se identifica codigo DANE SIE'")
    print(f"  - Total registros en auditoría: {len(df_auditoria):,}")
else:
    print(f"  - No hay registros para copiar a auditoría")

# ============================================================================
# PASO 9: VALIDAR CÓDIGOS DANE (5 DÍGITOS)
# ============================================================================
print("\n✓ Validando formato de códigos DANE...")
print("-" * 80)

print(f"\n  Analizando códigos DANE obtenidos...")

codigos_validos = 0
codigos_invalidos = 0
codigos_invalidos_list = []

for codigo in df_pivot['municipio_SIE_codigo'].dropna().unique():
    try:
        codigo_str = str(codigo)
        if len(codigo_str) == 5 and codigo_str.isdigit():
            codigos_validos += 1
        else:
            codigos_invalidos += 1
            if codigo not in codigos_invalidos_list:
                codigos_invalidos_list.append(codigo)
    except:
        codigos_invalidos += 1

print(f"  - Códigos DANE válidos (5 dígitos): {codigos_validos:,}")
if codigos_invalidos > 0:
    print(f"  ⚠️  Códigos DANE inválidos: {codigos_invalidos:,}")
    print(f"     Códigos encontrados: {codigos_invalidos_list[:10]}")

# ============================================================================
# PASO 10: MOSTRAR EJEMPLOS DE NORMALIZACIÓN EXITOSA
# ============================================================================
print("\n" + "="*80)
print("✅ EJEMPLOS DE NORMALIZACIÓN EXITOSA")
print("="*80)

df_normalizados_ej = df_pivot[
    (municipio_sIE_original.notna()) & 
    (df_pivot['municipio_SIE_codigo'].notna())
][['municipio_SIE', 'municipio_SIE_codigo']].drop_duplicates()

print(f"\nPrimeros 20 municipios normalizados:")
print("-" * 80)

for idx, (nombre, codigo) in enumerate(df_normalizados_ej.head(20).values, 1):
    print(f"  {idx:2d}. '{nombre}' → '{codigo}'")

if len(df_normalizados_ej) > 20:
    print(f"  ... y {len(df_normalizados_ej) - 20} municipios más")

# ============================================================================
# PASO 11: REEMPLAZAR COLUMNA ORIGINAL
# ============================================================================
print("\n✓ Reemplazando columna original con códigos DANE...")
print("-" * 80)

df_pivot['municipio_SIE'] = df_pivot['municipio_SIE_codigo']

df_pivot.drop(columns=['municipio_SIE_codigo'], inplace=True)

print(f"  - Columna 'municipio_SIE' actualizada con códigos DANE")
print(f"  - Columna temporal eliminada")

# ============================================================================
# PASO 12: ANÁLISIS FINAL DE DATOS
# ============================================================================
print("\n" + "="*80)
print("📊 ESTADÍSTICAS DE 'municipio_SIE' DESPUÉS DE NORMALIZACIÓN")
print("="*80)

print(f"\n✓ ESTADÍSTICAS FINALES:")
print(f"  - Total de registros: {len(df_pivot):,}")
print(f"  - Valores no nulos (códigos DANE): {df_pivot['municipio_SIE'].notna().sum():,}")
print(f"  - Valores nulos/vacíos: {df_pivot['municipio_SIE'].isna().sum():,}")
print(f"  - Valores únicos: {df_pivot['municipio_SIE'].nunique()}")

# ============================================================================
# PASO 13: TABLA RESUMEN DE NORMALIZACIÓN
# ============================================================================
print("\n" + "="*80)
print("📋 RESUMEN DE LA NORMALIZACIÓN")
print("="*80)

resumen_normalizacion = pd.DataFrame({
    'Concepto': [
        'Registros normalizados exitosamente',
        'Registros sin mapeo (enviados a auditoría)',
        'Registros vacíos/sin valor original',
        'Total de registros procesados'
    ],
    'Cantidad': [
        registros_normalizados,
        registros_sin_mapeo,
        registros_vacios,
        len(df_pivot)
    ],
    'Porcentaje': [
        f"{(registros_normalizados/len(df_pivot)*100):.2f}%",
        f"{(registros_sin_mapeo/len(df_pivot)*100):.2f}%",
        f"{(registros_vacios/len(df_pivot)*100):.2f}%",
        "100.00%"
    ]
})

print("\n" + resumen_normalizacion.to_string(index=False))

# ============================================================================
# PASO 14: DISTRIBUCIÓN DE MUNICIPIOS NORMALIZADOS (USA DICCIONARIO GUARDADO)
# ============================================================================
print("\n" + "="*80)
print("🗺️  DISTRIBUCIÓN DE MUNICIPIOS NORMALIZADOS")
print("="*80)

distribucion_municipios = df_pivot['municipio_SIE'].value_counts(dropna=True).head(20)

print(f"\nTop 20 municipios normalizados:")
print("-" * 80)

for idx, (codigo, cantidad) in enumerate(distribucion_municipios.items(), 1):
    # ✅ CONVERTIR CÓDIGO A STRING Y LIMPIAR
    codigo_str = str(int(float(codigo))) if pd.notna(codigo) else "N/A"
    
    # ✅ USAR DICCIONARIO GUARDADO (no df_municipios_sie)
    nombre = diccionario_nombres_municipios.get(codigo_str, "N/A")
    porcentaje = (cantidad / df_pivot['municipio_SIE'].notna().sum()) * 100
    
    # ✅ ASEGURAR QUE nombre ES STRING
    nombre_str = str(nombre) if nombre != "N/A" else "N/A"
    
    print(f"  {idx:2d}. Código {codigo_str} ({nombre_str:40s}): {cantidad:10,d} ({porcentaje:6.2f}%)")

# ============================================================================
# PASO 15: LIMPIAR MEMORIA - ELIMINAR df_municipios_sie
# ============================================================================
print("\n✓ Limpiando memoria...")
print("-" * 80)

tamaño_antes = len(df_municipios_sie)

# del df_municipios_sie

print(f"  - DataFrame 'df_municipios_sie' eliminado ({tamaño_antes:,} registros liberados)")
print(f"  - Memoria liberada exitosamente")

# ============================================================================
# PASO 16: MUESTRA DE DATOS FINALES
# ============================================================================
print("\n" + "="*80)
print("📋 MUESTRA DE DATOS FINALES")
print("="*80)

columnas_muestra = [
    'AFL_ID',
    'NUMERO_IDENTIFICACION',
    'municipio_SIE',
    'MUNICIPIO',
    'estado_SIE'
]

columnas_muestra_existentes = [col for col in columnas_muestra if col in df_pivot.columns]

print(f"\nPrimeros 15 registros:")
print("-" * 80)
print(df_pivot[columnas_muestra_existentes].head(15).to_string(index=False))

# ============================================================================
# PASO 17: ESTADÍSTICAS DE AUDITORÍA
# ============================================================================
print("\n" + "="*80)
print("🔍 ESTADÍSTICAS DE AUDITORÍA")
print("="*80)

print(f"\n✓ INFORMACIÓN DE df_auditoria:")
print(f"  - Total de registros: {len(df_auditoria):,}")
print(f"  - Registros por municipio_SIE no encontrado: {registros_sin_mapeo:,}")

if 'Descripción' in df_auditoria.columns:
    conteo_descripciones = df_auditoria['Descripción'].value_counts()
    print(f"\n  Descripción de errores registrados:")
    for descripcion, cantidad in conteo_descripciones.items():
        print(f"    - '{descripcion}': {cantidad:,}")

# ============================================================================
# PASO 18: VERIFICACIÓN DE INTEGRIDAD
# ============================================================================
print("\n" + "="*80)
print("✅ VERIFICACIÓN DE INTEGRIDAD")
print("="*80)

total_verificacion = registros_normalizados + registros_sin_mapeo + registros_vacios

print(f"\n✓ Verificación de conteos:")
print(f"  - Suma de categorías: {total_verificacion:,}")
print(f"  - Total de registros: {len(df_pivot):,}")
print(f"  - ¿Coinciden?: {total_verificacion == len(df_pivot)}")

if total_verificacion == len(df_pivot):
    print(f"\n✅ Integridad verificada correctamente")
else:
    print(f"\n⚠️  ADVERTENCIA: Hay discrepancia en los conteos")

# ============================================================================
# PASO 19: RESUMEN FINAL
# ============================================================================
tiempo_final = time.time()
tiempo_transcurrido = tiempo_final - tiempo_inicio

print("\n" + "="*80)
print("✅ NORMALIZACIÓN COMPLETADA EXITOSAMENTE")
print("="*80)

print(f"\n📊 RESUMEN FINAL:")
print(f"  ✓ Columna normalizada: 'municipio_SIE'")
print(f"  ✓ Tipo de normalización: Descripción → Código DANE (5 dígitos)")
print(f"  ✓ Registros procesados exitosamente: {registros_normalizados:,}")
print(f"  ✗ Registros no encontrados y auditados: {registros_sin_mapeo:,}")
print(f"  ∅ Registros vacíos preservados: {registros_vacios:,}")
print(f"  ✓ Municipios únicos normalizados: {df_pivot['municipio_SIE'].nunique()}")

print(f"\n🔧 OPERACIONES REALIZADAS:")
print(f"  ✓ Búsqueda de nombres en catálogo DANE")
print(f"  ✓ Mapeo a códigos de 5 dígitos")
print(f"  ✓ Validación de códigos DANE")
print(f"  ✓ Registro de errores en auditoría")
print(f"  ✓ Limpieza de memoria (df_municipios_sie eliminado)")

print(f"\n💾 ESTADO DE DataFrames:")
print(f"  - df_pivot: {len(df_pivot):,} registros")
print(f"  - df_auditoria: {len(df_auditoria):,} registros")

print(f"\n⏱️  TIEMPO TRANSCURRIDO: {tiempo_transcurrido:.4f} segundos")
print("="*80 + "\n")

# Comparar red de servicios SIE vs Contratada

In [ ]:
import pandas as pd

print("\n" + "="*80)
print("🔍 VALIDANDO COLUMNAS DISPONIBLES")
print("="*80)

# Verificar qué columnas existen en df_pivot
print("\n✓ Primeras columnas de df_pivot:")
print(df_pivot.columns[:20].tolist())

# Identificar la columna correcta para EPS
columna_eps = None
if 'ABREVIATURA' in df_pivot.columns:
    columna_eps = 'ABREVIATURA'
elif 'abreviatura' in df_pivot.columns:
    columna_eps = 'abreviatura'
else:
    print("\n⚠️  ADVERTENCIA: No se encontró columna de EPS/abreviatura")
    print(f"Columnas disponibles: {df_pivot.columns.tolist()}")

print(f"\n✓ Columna EPS identificada: {columna_eps}")

# ============================================================================
# 1. PREPARAR LOS DATOS
# ============================================================================
mapeo_servicios = {
    'MEDICINA GENERAL': 'MEDICINA GENERAL',
    'LABORATORIO CLINICO': 'LABORATORIO',
    'ODONTOLOGIA GENERAL': 'ODONTOLOGIA GENERAL',
    'MEDICAMENTOS': 'MEDICAMENTOS',
    'OPTOMETRIA': 'OPTOMETRIA',
    'IMAGENES DIAGNOSTICAS - IONIZANTES': 'IMAGENES DIAGNOSTICAS - IONIZANTES',
}

# ============================================================================
# 2. NORMALIZAR NOMBRES DE SERVICIOS EN df_pivot
# ============================================================================
servicios_pivot = ['MEDICINA GENERAL', 'LABORATORIO CLINICO', 'ODONTOLOGIA GENERAL', 
                    'MEDICAMENTOS', 'OPTOMETRIA', 'IMAGENES DIAGNOSTICAS - IONIZANTES']

# Filtrar solo servicios que existan en df_pivot
servicios_validos = [s for s in servicios_pivot if s in df_pivot.columns]

print(f"\n✓ Servicios disponibles para validar: {len(servicios_validos)}")
print(f"  Servicios: {servicios_validos}")

# ============================================================================
# 3. CREAR UN LOOKUP DE RED CONTRATADA POR (MUNICIPIO, SERVICIO)
# ============================================================================
red_lookup = {}
for idx, row in df_red_contratada.iterrows():
    municipio = str(row['MUNICIPIO']).strip()
    servicio = str(row['NOMBRE SERVICIO']).strip()
    nit = str(row['NIT']).strip()
    
    key = (municipio, servicio)
    if key not in red_lookup:
        red_lookup[key] = {
            'NIT': nit,
            'CODIGO_SERVICIO': row['CODIGO SERVICIO'],
            'ID': row['ID'],
            'NOMBRE_IPS': row['NOMBRE IPS']
        }

print(f"\n✓ Red contratada cargada: {len(red_lookup)} pares municipio-servicio")

# ============================================================================
# 4. VALIDAR Y COMPARAR
# ============================================================================
def validar_red_asignada(row, municipio_col, eps_col, servicios_a_validar):
    """
    Valida si la red asignada en SIE coincide con la contratada.
    Retorna: 'OK', 'VACIO', 'INCORRECTO', o 'SIN_CONTRATO'
    """
    municipio = str(row[municipio_col]).strip()
    
    validaciones = {}
    
    for servicio in servicios_a_validar:
        nit_asignado = str(row[servicio]).strip() if pd.notna(row[servicio]) else None
        
        key = (municipio, servicio)
        
        if key in red_lookup:
            nit_contratado = red_lookup[key]['NIT']
            
            if nit_asignado is None or nit_asignado == '':
                validaciones[servicio] = {
                    'estado': 'VACIO',
                    'asignado': None,
                    'contratado': nit_contratado,
                    'deberia_ser': nit_contratado
                }
            elif nit_asignado == nit_contratado:
                validaciones[servicio] = {
                    'estado': 'OK',
                    'asignado': nit_asignado,
                    'contratado': nit_contratado
                }
            else:
                validaciones[servicio] = {
                    'estado': 'INCORRECTO',
                    'asignado': nit_asignado,
                    'contratado': nit_contratado,
                    'deberia_ser': nit_contratado
                }
        else:
            validaciones[servicio] = {
                'estado': 'SIN_CONTRATO',
                'asignado': nit_asignado,
                'contratado': None
            }
    
    return validaciones

# ============================================================================
# 5. APLICAR VALIDACIÓN
# ============================================================================
print(f"\n✓ Aplicando validación a {len(df_pivot):,} registros...")

df_pivot['validacion_red'] = df_pivot.apply(
    lambda row: validar_red_asignada(row, 'municipio_SIE', columna_eps, servicios_validos), 
    axis=1
)

print(f"✓ Validación completada")

# ============================================================================
# 6. EXPANDIR RESULTADOS PARA ANÁLISIS
# ============================================================================
def expandir_validacion(row, eps_col):
    """Convierte el dict de validación en filas expandidas"""
    municipio = row['municipio_SIE']
    eps = row[eps_col]
    numero_id = row['NUMERO_IDENTIFICACION']
    
    validaciones = row['validacion_red']
    
    resultados = []
    for servicio, val in validaciones.items():
        resultados.append({
            'municipio': municipio,
            'eps': eps,
            'numero_id': numero_id,
            'servicio': servicio,
            'estado': val['estado'],
            'nit_asignado': val.get('asignado'),
            'nit_contratado': val.get('contratado'),
            'deberia_ser': val.get('deberia_ser')
        })
    
    return resultados

# ============================================================================
# 7. CREAR DATAFRAME EXPANDIDO
# ============================================================================
print(f"\n✓ Expandiendo resultados...")

resultados_expandidos = []
for idx, row in df_pivot.iterrows():
    resultados_expandidos.extend(expandir_validacion(row, columna_eps))

df_validacion = pd.DataFrame(resultados_expandidos)

print(f"✓ DataFrame de validación creado: {len(df_validacion):,} registros")

# ============================================================================
# 8. REPORTE
# ============================================================================
print("\n" + "="*80)
print("RESUMEN DE VALIDACIÓN DE RED")
print("="*80)
print(df_validacion['estado'].value_counts())
print("\n")

print("SERVICIOS CON PROBLEMAS POR MUNICIPIO:")
problemas = df_validacion[df_validacion['estado'].isin(['VACIO', 'INCORRECTO'])]
resumen_problemas = problemas.groupby(['municipio', 'servicio', 'estado']).size().reset_index(name='cantidad')
print(resumen_problemas.to_string(index=False))
print("\n")

print("EJEMPLO DE REGISTROS CON RED INCORRECTA:")
incorrectos = df_validacion[df_validacion['estado'] == 'INCORRECTO'].head(10)
print(incorrectos[['municipio', 'servicio', 'nit_asignado', 'nit_contratado']].to_string(index=False))

print("\n" + "="*80)
print("✅ VALIDACIÓN COMPLETADA")
print("="*80)

In [ ]:
print(df_pivot.columns)

In [ ]:
print(df_red_contratada['NOMBRE SERVICIO'].value_counts(dropna=False))

In [ ]:
df_red_contratada.columns

# Guardar resultados

In [ ]:
import os
from datetime import datetime
import pandas as pd
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils.dataframe import dataframe_to_rows

print("\n" + "="*80)
print("💾 GUARDANDO MÚLTIPLES DATAFRAMES EN EXCEL")
print("="*80)

# ============================================================================
# PASO 1: PREPARAR NOMBRE DE ARCHIVO Y RUTA
# ============================================================================
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
nombre_archivo = f"Red_Asignada_Consolidada_{timestamp}.xlsx"
ruta_completa = os.path.join(R_Salida, nombre_archivo)

print(f"\n✓ Archivo a crear: {nombre_archivo}")
print(f"✓ Ruta completa: {ruta_completa}")

# ============================================================================
# PASO 2: CREAR ARCHIVO EXCEL CON MÚLTIPLES HOJAS
# ============================================================================
try:
    # Crear ExcelWriter (gestor de contexto para múltiples hojas)
    with pd.ExcelWriter(ruta_completa, engine='openpyxl') as writer:
        
        # ====================================================================
        # HOJA 1: Red consolidada (df_pivot)
        # ====================================================================
        print("\n✓ Escribiendo hoja 1: 'Red_Consolidada'")
        df_pivot.to_excel(
            writer, 
            sheet_name='Red_Consolidada',
            index=False,
            startrow=0,
            startcol=0
        )
        
        registros_pivot = len(df_pivot)
        columnas_pivot = len(df_pivot.columns)
        print(f"  - Registros: {registros_pivot:,}")
        print(f"  - Columnas: {columnas_pivot}")
        
        # ====================================================================
        # HOJA 2: Auditoría (df_auditoria)
        # ====================================================================
        print("\n✓ Escribiendo hoja 2: 'Auditoria'")
        df_auditoria.to_excel(
            writer,
            sheet_name='Auditoria',
            index=False,
            startrow=0,
            startcol=0
        )
        
        registros_auditoria = len(df_auditoria)
        columnas_auditoria = len(df_auditoria.columns)
        print(f"  - Registros: {registros_auditoria:,}")
        print(f"  - Columnas: {columnas_auditoria}")
        
        # ====================================================================
        # HOJA 3: Resumen de auditoría SIE (opcional pero recomendado)
        # ====================================================================
        # Crear resumen de tipos de auditoría
        df_resumen_auditoria = pd.DataFrame({
            'Tipo_Auditoria': df_auditoria['Descripción'].value_counts().index,
            'Cantidad_Registros': df_auditoria['Descripción'].value_counts().values,
            'Porcentaje': (df_auditoria['Descripción'].value_counts().values / len(df_auditoria) * 100).round(2)
        }).reset_index(drop=True)
        
        print("\n✓ Escribiendo hoja 3: 'Resumen_Auditoria'")
        df_resumen_auditoria.to_excel(
            writer,
            sheet_name='Resumen_Auditoria',
            index=False,
            startrow=0,
            startcol=0
        )
        
        print(f"  - Tipos de auditoría identificados: {len(df_resumen_auditoria)}")
        
        # ====================================================================
        # HOJA 4: Estadísticas Generales (opcional pero muy útil)
        # ====================================================================
        # Crear hoja de estadísticas
        df_estadisticas = pd.DataFrame({
            'Concepto': [
                'Total de registros en Red Consolidada',
                'Total de registros en Auditoría',
                'Afiliados con servicios asignados',
                'Afiliados sin servicios',
                'Porcentaje con servicios',
                'Porcentaje sin servicios',
                'Total de columnas de servicios',
                'Total de columnas en Red Consolidada',
                'Fecha de generación',
                'Nombre del archivo'
            ],
            'Valor': [
                registros_pivot,
                registros_auditoria,
                registros_pivot - registros_auditoria,
                registros_auditoria,
                f"{((registros_pivot - registros_auditoria) / registros_pivot * 100):.2f}%",
                f"{(registros_auditoria / registros_pivot * 100):.2f}%",
                columnas_pivot - 3,  # Menos las columnas de identificación
                columnas_pivot,
                datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                nombre_archivo
            ]
        })
        
        print("\n✓ Escribiendo hoja 4: 'Estadisticas'")
        df_estadisticas.to_excel(
            writer,
            sheet_name='Estadisticas',
            index=False,
            startrow=0,
            startcol=0
        )
        
        # ====================================================================
        # OBTENER WORKBOOK PARA FORMATO (OPCIONAL)
        # ====================================================================
        workbook = writer.book
        
        # Formato para hoja de Estadísticas
        ws_stats = writer.sheets['Estadisticas']
        fill_header = PatternFill(start_color="4472C4", end_color="4472C4", fill_type="solid")
        font_header = Font(bold=True, color="FFFFFF", size=11)
        
        for cell in ws_stats[1]:
            if cell.value:
                cell.fill = fill_header
                cell.font = font_header
                cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        
        # Ajustar ancho de columnas en Estadísticas
        ws_stats.column_dimensions['A'].width = 45
        ws_stats.column_dimensions['B'].width = 35
        
        # Formato para hoja de Resumen de Auditoría
        ws_resumen = writer.sheets['Resumen_Auditoria']
        for cell in ws_resumen[1]:
            if cell.value:
                cell.fill = fill_header
                cell.font = font_header
                cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        
        ws_resumen.column_dimensions['A'].width = 40
        ws_resumen.column_dimensions['B'].width = 25
        ws_resumen.column_dimensions['C'].width = 20
        
        # Formato para hoja de Auditoría
        ws_auditoria = writer.sheets['Auditoria']
        for cell in ws_auditoria[1]:
            if cell.value:
                cell.fill = fill_header
                cell.font = font_header
                cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        
        ws_auditoria.column_dimensions['A'].width = 15
        ws_auditoria.column_dimensions['B'].width = 25
        ws_auditoria.column_dimensions['C'].width = 60
        
        # Formato para hoja de Red Consolidada
        ws_pivot = writer.sheets['Red_Consolidada']
        for cell in ws_pivot[1]:
            if cell.value:
                cell.fill = fill_header
                cell.font = font_header
                cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        
        # Ajustar ancho de primeras columnas de identificación
        ws_pivot.column_dimensions['A'].width = 12
        ws_pivot.column_dimensions['B'].width = 25
        ws_pivot.column_dimensions['C'].width = 30
        
    print("\n" + "="*80)
    print("✅ ARCHIVO GUARDADO EXITOSAMENTE")
    print("="*80)
    print(f"📁 Ruta: {ruta_completa}")
    print(f"💾 Tamaño del archivo: {os.path.getsize(ruta_completa) / (1024*1024):.2f} MB")
    
    print("\n📋 HOJAS CREADAS:")
    print("  1️⃣  'Red_Consolidada'    → Datos principales con servicios asignados")
    print("  2️⃣  'Auditoria'          → Registros sin servicios y sus razones")
    print("  3️⃣  'Resumen_Auditoria'  → Resumen estadístico de auditoría")
    print("  4️⃣  'Estadisticas'       → Estadísticas generales del proceso")
    
    print("\n" + "="*80)
    print("📊 RESUMEN DE CONTENIDO:")
    print("="*80)
    print(f"✓ Red Consolidada:")
    print(f"  - Registros: {registros_pivot:,}")
    print(f"  - Columnas: {columnas_pivot}")
    print(f"\n✓ Auditoría:")
    print(f"  - Registros totales: {registros_auditoria:,}")
    print(f"  - Tipos de auditoría: {len(df_resumen_auditoria)}")
    
    print("\n📈 Desglose de auditoría:")
    for _, row in df_resumen_auditoria.iterrows():
        print(f"  - {row['Tipo_Auditoria']:40s}: {row['Cantidad_Registros']:7,d} ({row['Porcentaje']:5.2f}%)")
    
    print("\n" + "="*80)
    
except FileNotFoundError:
    print(f"\n❌ ERROR: No se puede acceder a la ruta")
    print(f"Ruta: {R_Salida}")
    print(f"Por favor, verifica que la carpeta exista")
    
except PermissionError:
    print(f"\n❌ ERROR: Permiso denegado")
    print(f"El archivo podría estar abierto en Excel")
    print(f"Por favor, cierra el archivo e intenta nuevamente")
    
except Exception as e:
    print(f"\n❌ ERROR AL GUARDAR EL ARCHIVO")
    print(f"Tipo de error: {type(e).__name__}")
    print(f"Mensaje: {str(e)}")
    import traceback
    print("\nDetalles completos:")
    traceback.print_exc()

print("="*80)